In [1]:
# ============================================================
# Publishable inverse-PDE pipeline
# CABiSSM vs DeepONet vs FNOEnc vs VC-PINN-style
# Includes: equation selection, baselines, ablations, robustness sweeps,
# runtime analysis, normalized metrics, variance-ratio metrics, PDF + ZIP export.
#
# Recommended Kaggle use:
#   1) Paste this full script into one Kaggle notebook cell.
#   2) Edit ONLY the USER SETTINGS block.
#   3) Run one problem / one experiment mode at a time.
# ============================================================

import csv
import json
import math
import random
import time
import zipfile
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# USER SETTINGS: change this block only
# ============================================================
@dataclass
class CFG:
    # ------------------------------
    # Main experiment selection
    # ------------------------------
    # Available problems:
    #   "wave"         : variable wave speed c(x)
    #   "adr"          : variable diffusion D(x)
    #   "vkdv"         : random time-coefficient KdV-style g(t)
    #   "vkdv_paper"   : VC-PINN-paper-inspired vKdV coefficient families
    #   "vsg"          : variable-coefficient Sine-Gordon h(t)
    problem_name: str = "wave"

    # Run modes:
    #   "main"          : main model + baselines
    #   "ablation"      : CABiSSM ablation study
    #   "sensor_sweep"  : sparse sensor robustness sweep
    #   "noise_sweep"   : observation-noise robustness sweep
    #   "all_modes"     : run main + ablation + sensor_sweep + noise_sweep sequentially for one equation
    run_mode: str = "main"

    # For vkdv_paper and vsg. Options are problem-dependent.
    # vkdv_paper coefficient_family: "linear", "cubic", "cos", "exp_decay", "mixed_random"
    # vsg coefficient_family       : "linear", "quadratic", "cos", "mixed_random"
    coefficient_family: str = "mixed_random"

    # Main methods.
    # For main runs, keep all enabled.
    run_methods: Tuple[str, ...] = ("cabissm", "deeponet", "fnoenc", "vc_pinn")

    # ------------------------------
    # Data sizes
    # ------------------------------
    train_size: int = 16384
    val_size: int = 2048
    n_sensors: int = 128
    sensor_noise_std: float = 0.01
    sim_batch_size: int = 128

    # Grids
    n_x: int = 129
    n_t: int = 301
    n_q: int = 129

    # ------------------------------
    # Amortized model training
    # ------------------------------
    batch_size: int = 64
    epochs: int = 50
    baseline_epochs: int = 50
    lr: float = 3e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    lambda_grad: float = 0.5
    lambda_tv: float = 1e-4
    early_stop_patience: int = 12

    # Runtime
    seed: int = 42
    num_workers: int = 2
    use_amp: bool = True
    amp_dtype: str = "bfloat16"  # good on RTX PRO 6000 / H100; FFT sections force FP32 internally
    use_compile: bool = False
    out_root: str = "/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline"

    # ------------------------------
    # CABiSSM architecture
    # ------------------------------
    d_model: int = 128
    n_heads: int = 8
    n_layers: int = 4
    state_dim: int = 8
    mlp_ratio: int = 4
    dropout: float = 0.0
    use_fourier_features: bool = True
    use_cross_attention: bool = True
    use_ssm: bool = True

    # ------------------------------
    # DeepONet baseline
    # ------------------------------
    deeponet_width: int = 128
    deeponet_p: int = 128

    # ------------------------------
    # FNO-Encoder baseline
    # ------------------------------
    fno_width: int = 48
    fno_modes_t: int = 16
    fno_modes_x: int = 16
    fno_layers: int = 4
    fno_t_bins: int = 129
    fno_x_bins: int = 129

    # ------------------------------
    # VC-PINN-style baseline
    # ------------------------------
    # This is sample-wise, so full validation is expensive.
    vc_eval_samples: int = 16
    vc_hidden: int = 128
    vc_blocks_u: int = 4
    vc_blocks_c: int = 3
    vc_layers_per_block: int = 2
    vc_adam_steps: int = 2000
    vc_lbfgs_steps: int = 100
    vc_lr: float = 1e-3
    vc_colloc_f: int = 4096
    vc_colloc_ic: int = 512
    vc_colloc_bc: int = 512
    vc_w_data: float = 10.0
    vc_w_pde: float = 1.0
    vc_w_ic: float = 10.0
    vc_w_bc: float = 10.0
    vc_w_cbc: float = 10.0
    vc_w_c_smooth: float = 1e-4
    vc_log_every: int = 500

    # ------------------------------
    # Journal-strength extra experiments
    # ------------------------------
    sensor_sweep_values: Tuple[int, ...] = (16, 32, 64, 128)
    noise_sweep_values: Tuple[float, ...] = (0.0, 0.01, 0.03, 0.05)

    # For sweeps, use smaller sizes if you want a quick result.
    sweep_train_size: int = 8192
    sweep_val_size: int = 1024
    sweep_epochs: int = 35

    # Ablation variants.
    # These are trained only when run_mode == "ablation".
    ablation_names: Tuple[str, ...] = (
        "full_cabissm",
        "no_ssm_attention_only",
        "no_cross_ssm_only",
        "no_fourier_features",
        "no_gradient_loss",
    )

    # Artifacts
    make_pdf_summary: bool = True
    make_zip: bool = True
    save_checkpoints: bool = True


# ============================================================
# Utilities
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_amp_dtype(cfg: CFG):
    return torch.bfloat16 if cfg.amp_dtype.lower() == "bfloat16" else torch.float16


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def gradient_1d(field: torch.Tensor) -> torch.Tensor:
    return field[:, 1:, :] - field[:, :-1, :]


def total_variation_1d(field: torch.Tensor) -> torch.Tensor:
    return (field[:, 1:, :] - field[:, :-1, :]).abs().mean()


@torch.no_grad()
def coeff_metrics(c_hat: torch.Tensor, c_true: torch.Tensor) -> Dict[str, float]:
    c_hat = c_hat.float()
    c_true = c_true.float()

    mse = F.mse_loss(c_hat, c_true)
    mae = F.l1_loss(c_hat, c_true)
    rel_l2 = torch.norm(c_hat - c_true) / (torch.norm(c_true) + 1e-8)
    grad_mae = F.l1_loss(gradient_1d(c_hat), gradient_1d(c_true))

    pred_mean = c_hat.mean()
    pred_std = c_hat.std()
    true_mean = c_true.mean()
    true_std = c_true.std()

    norm_mae = mae / (true_std + 1e-8)
    norm_rmse = torch.sqrt(mse) / (true_std + 1e-8)
    var_ratio = pred_std / (true_std + 1e-8)
    mean_bias = pred_mean - true_mean

    return {
        "mse": float(mse.item()),
        "mae": float(mae.item()),
        "rel_l2": float(rel_l2.item()),
        "grad_mae": float(grad_mae.item()),
        "norm_mae": float(norm_mae.item()),
        "norm_rmse": float(norm_rmse.item()),
        "var_ratio": float(var_ratio.item()),
        "mean_bias": float(mean_bias.item()),
        "pred_mean": float(pred_mean.item()),
        "pred_std": float(pred_std.item()),
        "true_mean": float(true_mean.item()),
        "true_std": float(true_std.item()),
    }


def mean_metric(metrics: List[Dict[str, float]], key: str) -> float:
    vals = [m[key] for m in metrics if key in m]
    return sum(vals) / max(len(vals), 1)


def save_json(obj: Dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def move_batch(batch: Dict[str, torch.Tensor], device: torch.device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# Dataset
# ============================================================
class InversePDEDataset(Dataset):
    def __init__(self, sensors: torch.Tensor, q_grid: torch.Tensor, coeff_true: torch.Tensor):
        self.sensors = sensors.float()
        self.q_grid = q_grid.float()
        self.coeff_true = coeff_true.float()

    def __len__(self):
        return self.sensors.shape[0]

    def __getitem__(self, idx):
        return {
            "sensors": self.sensors[idx],
            "q_grid": self.q_grid[idx],
            "coeff_true": self.coeff_true[idx],
        }


def make_loader(ds: Dataset, cfg: CFG, shuffle: bool):
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(cfg.num_workers > 0),
    )


# ============================================================
# PDE problem classes
# ============================================================
class BaseProblem:
    name = "base"
    target_axis = "x"  # "x" or "t"
    coeff_min = 0.0
    coeff_max = 1.0
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0

    def make_grids(self, cfg: CFG, device: torch.device):
        x = torch.linspace(self.x_min, self.x_max, cfg.n_x, device=device)
        t = torch.linspace(self.t_min, self.t_max, cfg.n_t, device=device)
        if self.target_axis == "x":
            q = torch.linspace(self.x_min, self.x_max, cfg.n_q, device=device)
        else:
            q = torch.linspace(self.t_min, self.t_max, cfg.n_q, device=device)
        return x, t, q

    def generate_coeff(self, cfg: CFG, batch_size: int, q: torch.Tensor, device: torch.device):
        raise NotImplementedError

    def solve(self, cfg: CFG, coeff: torch.Tensor, x: torch.Tensor, t: torch.Tensor, q: torch.Tensor):
        raise NotImplementedError

    def sample_sensors(self, cfg: CFG, U: torch.Tensor, x: torch.Tensor, t: torch.Tensor):
        B = U.shape[0]
        Ns = cfg.n_sensors
        device = U.device
        ix = torch.randint(0, len(x), (B, Ns), device=device)
        it = torch.randint(0, len(t), (B, Ns), device=device)
        b = torch.arange(B, device=device).unsqueeze(1).expand(B, Ns)
        u = U[b, it, ix]
        if cfg.sensor_noise_std > 0:
            u = u + cfg.sensor_noise_std * torch.randn_like(u)
        return torch.stack([x[ix], t[it], u], dim=-1)

    @torch.no_grad()
    def build_dataset(self, cfg: CFG, size: int, device: torch.device, tag: str):
        x, t, q = self.make_grids(cfg, device)
        sensors_all, coeff_all, q_all = [], [], []
        num_batches = math.ceil(size / cfg.sim_batch_size)
        print(f"[{self.name}] Building {tag} dataset: {size} samples, {num_batches} simulation batches")
        start = time.time()
        for bi in range(num_batches):
            bs = min(cfg.sim_batch_size, size - bi * cfg.sim_batch_size)
            coeff = self.generate_coeff(cfg, bs, q, device)
            U = self.solve(cfg, coeff, x, t, q)
            sensors = self.sample_sensors(cfg, U, x, t)
            q_grid = q.view(1, -1, 1).repeat(bs, 1, 1)
            sensors_all.append(sensors.cpu())
            coeff_all.append(coeff.unsqueeze(-1).cpu())
            q_all.append(q_grid.cpu())
            if (bi + 1) % max(1, num_batches // 10) == 0 or (bi + 1) == num_batches:
                print(f"  [{self.name}/{tag}] batch {bi + 1}/{num_batches} | elapsed {time.time() - start:.1f}s")
        return InversePDEDataset(torch.cat(sensors_all), torch.cat(q_all), torch.cat(coeff_all))

    def coeff_input(self, x_col: torch.Tensor, t_col: torch.Tensor) -> torch.Tensor:
        return x_col if self.target_axis == "x" else t_col

    def pde_residual(self, model, x_col: torch.Tensor, t_col: torch.Tensor):
        raise NotImplementedError

    def aux_losses(self, model, sample: Dict[str, torch.Tensor], cfg: CFG, device: torch.device):
        return {}


class WaveProblem(BaseProblem):
    name = "wave"
    target_axis = "x"
    coeff_min = 0.75
    coeff_max = 1.45
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    beta_ic = 3

    def initial_condition(self, x):
        return torch.sin(math.pi * x) + 0.5 * torch.sin(self.beta_ic * math.pi * x)

    def smooth(self, field, kernel_size=9):
        pad = kernel_size // 2
        z = field.unsqueeze(1)
        z = F.pad(z, (pad, pad), mode="replicate")
        z = F.avg_pool1d(z, kernel_size=kernel_size, stride=1)
        return z.squeeze(1)

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            mode = random.choice(["sin", "gauss", "piecewise"])
            base = random.uniform(0.95, 1.15)
            xx = q_cpu.clone()
            if mode == "sin":
                c = base
                c = c + random.uniform(0.08, 0.20) * torch.sin(2 * math.pi * xx + random.uniform(0, 2 * math.pi))
                c = c + random.uniform(0.03, 0.10) * torch.sin(4 * math.pi * xx + random.uniform(0, 2 * math.pi))
            elif mode == "gauss":
                c = torch.full_like(xx, base)
                for _ in range(random.randint(1, 3)):
                    amp = random.uniform(-0.18, 0.18)
                    ctr = random.uniform(0.1, 0.9)
                    wid = random.uniform(0.03, 0.12)
                    c = c + amp * torch.exp(-0.5 * ((xx - ctr) / wid) ** 2)
            else:
                n_segments = random.randint(3, 6)
                edges = sorted(random.sample(range(8, cfg.n_q - 8), n_segments - 1))
                edges = [0] + edges + [cfg.n_q]
                c = torch.empty_like(xx)
                cur = base
                for s in range(len(edges) - 1):
                    cur = max(self.coeff_min, min(self.coeff_max, cur + random.uniform(-0.18, 0.18)))
                    c[edges[s]:edges[s + 1]] = cur
            fields.append(c)
        c = torch.stack(fields).to(device)
        c = self.smooth(c)
        return c.clamp(self.coeff_min, self.coeff_max)

    def div_operator(self, u, a_half, dx):
        out = torch.zeros_like(u)
        out[:, 1:-1] = (
            a_half[:, 1:] * (u[:, 2:] - u[:, 1:-1])
            - a_half[:, :-1] * (u[:, 1:-1] - u[:, :-2])
        ) / (dx * dx)
        return out

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        coeff_x = coeff if coeff.shape[1] == len(x) else F.interpolate(coeff.unsqueeze(1), size=len(x), mode="linear", align_corners=True).squeeze(1)
        B, Nx = coeff_x.shape
        Nt = len(t)
        dx = float((self.x_max - self.x_min) / (Nx - 1))
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        if dt > 0.95 * dx / self.coeff_max:
            print("Warning: wave CFL may be high. Increase n_t or reduce coeff_max.")
        a = coeff_x * coeff_x
        a_half = 0.5 * (a[:, :-1] + a[:, 1:])
        u0 = self.initial_condition(x).unsqueeze(0).repeat(B, 1)
        U = torch.zeros(B, Nt, Nx, device=x.device)
        U[:, 0] = u0
        Lu0 = self.div_operator(u0, a_half, dx)
        u1 = u0.clone()
        u1[:, 1:-1] = u0[:, 1:-1] + 0.5 * dt * dt * Lu0[:, 1:-1]
        u1[:, 0] = 0.0
        u1[:, -1] = 0.0
        U[:, 1] = u1
        up, uc = u0, u1
        for n in range(1, Nt - 1):
            Lu = self.div_operator(uc, a_half, dx)
            un = 2 * uc - up + dt * dt * Lu
            un[:, 0] = 0.0
            un[:, -1] = 0.0
            U[:, n + 1] = un
            up, uc = uc, un
        return U

    def pde_residual(self, model, x_col, t_col):
        u = model.u(x_col, t_col)
        c = model.coeff(x_col)
        a = c * c
        u_t = autograd_grad(u, t_col)
        u_tt = autograd_grad(u_t, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        a_x = autograd_grad(a, x_col)
        return u_tt - (a_x * u_x + a * u_xx)

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device).requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        u_ic = model.u(x_ic, t_ic)
        u_true = self.initial_condition(x_ic.squeeze(-1)).unsqueeze(-1)
        u_t_ic = autograd_grad(u_ic, t_ic)
        losses["ic"] = F.mse_loss(u_ic, u_true) + F.mse_loss(u_t_ic, torch.zeros_like(u_t_ic))
        t_bc = torch.rand(cfg.vc_colloc_bc, 1, device=device).requires_grad_(True)
        x0 = torch.zeros_like(t_bc).requires_grad_(True)
        x1 = torch.ones_like(t_bc).requires_grad_(True)
        losses["bc"] = F.mse_loss(model.u(x0, t_bc), torch.zeros_like(t_bc)) + F.mse_loss(model.u(x1, t_bc), torch.zeros_like(t_bc))
        return losses


class ADRProblem(BaseProblem):
    name = "adr"
    target_axis = "x"
    coeff_min = 0.0015
    coeff_max = 0.0060
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    v = 0.4
    lam = 1.0

    def initial_condition(self, x):
        return torch.sin(math.pi * x)

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            base = random.uniform(0.003, 0.0045)
            D = base + random.uniform(0.0004, 0.0012) * torch.sin(2 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            mode = random.choice(["smooth", "bumps", "piecewise"])
            if mode == "bumps":
                for _ in range(random.randint(1, 3)):
                    ctr = random.uniform(0.15, 0.85)
                    wid = random.uniform(0.035, 0.15)
                    amp = random.uniform(-0.0012, 0.0012)
                    D = D + amp * torch.exp(-0.5 * ((q_cpu - ctr) / wid) ** 2)
            elif mode == "piecewise":
                jump = torch.zeros_like(q_cpu)
                ctr = random.uniform(0.25, 0.75)
                jump[q_cpu > ctr] = random.uniform(-0.001, 0.001)
                D = D + jump
            fields.append(D)
        D = torch.stack(fields).to(device)
        D = F.avg_pool1d(F.pad(D.unsqueeze(1), (4, 4), mode="replicate"), kernel_size=9, stride=1).squeeze(1)
        return D.clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        D = coeff if coeff.shape[1] == len(x) else F.interpolate(coeff.unsqueeze(1), size=len(x), mode="linear", align_corners=True).squeeze(1)
        B, Nx = D.shape
        Nt = len(t)
        dx = float((self.x_max - self.x_min) / (Nx - 1))
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        D_half = 0.5 * (D[:, :-1] + D[:, 1:])
        U = torch.zeros(B, Nt, Nx, device=x.device)
        U[:, 0] = self.initial_condition(x).unsqueeze(0).repeat(B, 1)
        for n in range(Nt - 1):
            u = U[:, n]
            diff = torch.zeros_like(u)
            diff[:, 1:-1] = (
                D_half[:, 1:] * (u[:, 2:] - u[:, 1:-1])
                - D_half[:, :-1] * (u[:, 1:-1] - u[:, :-2])
            ) / (dx * dx)
            ux_up = torch.zeros_like(u)
            ux_up[:, 1:] = (u[:, 1:] - u[:, :-1]) / dx
            reaction = self.lam * u * (1.0 - u)
            un = u + dt * (diff - self.v * ux_up + reaction)
            un[:, 0] = 0.0
            un[:, -1] = 0.0
            U[:, n + 1] = un.clamp(-3.0, 3.0)
        return U

    def pde_residual(self, model, x_col, t_col):
        u = model.u(x_col, t_col)
        D = model.coeff(x_col)
        u_t = autograd_grad(u, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        D_x = autograd_grad(D, x_col)
        return u_t - (D_x * u_x + D * u_xx - self.v * u_x + self.lam * u * (1.0 - u))

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device).requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_condition(x_ic.squeeze(-1)).unsqueeze(-1))
        t_bc = torch.rand(cfg.vc_colloc_bc, 1, device=device).requires_grad_(True)
        x0 = torch.zeros_like(t_bc).requires_grad_(True)
        x1 = torch.ones_like(t_bc).requires_grad_(True)
        losses["bc"] = F.mse_loss(model.u(x0, t_bc), torch.zeros_like(t_bc)) + F.mse_loss(model.u(x1, t_bc), torch.zeros_like(t_bc))
        return losses


class VKdVProblem(BaseProblem):
    name = "vkdv"
    target_axis = "t"
    coeff_min = 0.50
    coeff_max = 1.50
    x_min = -1.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    kappa = 0.75
    x0 = -0.55

    def sech2(self, z):
        return 1.0 / torch.cosh(z).pow(2)

    def initial_profile(self, x):
        return 2.0 * self.kappa * self.kappa * self.sech2(self.kappa * (x - self.x0))

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            base = random.uniform(0.85, 1.15)
            g = base
            g = g + random.uniform(0.08, 0.22) * torch.sin(2 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            g = g + random.uniform(0.03, 0.12) * torch.sin(4 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            if random.random() < 0.5:
                g = g * torch.exp(-random.uniform(0.0, 0.4) * q_cpu)
            fields.append(g)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        g_t = coeff if coeff.shape[1] == len(t) else F.interpolate(coeff.unsqueeze(1), size=len(t), mode="linear", align_corners=True).squeeze(1)
        B, Nt = g_t.shape
        Nx = len(x)
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        tau = torch.zeros_like(g_t)
        tau[:, 1:] = torch.cumsum(0.5 * (g_t[:, 1:] + g_t[:, :-1]) * dt, dim=1)
        X = x.view(1, 1, Nx)
        Tau = tau.view(B, Nt, 1)
        center = self.x0 + 4.0 * self.kappa * self.kappa * Tau
        U = 2.0 * self.kappa * self.kappa * self.sech2(self.kappa * (X - center))
        return U

    def pde_residual(self, model, x_col, t_col):
        # u_t + 6 g(t) u u_x + g(t) u_xxx = 0
        u = model.u(x_col, t_col)
        g = model.coeff(t_col)
        u_t = autograd_grad(u, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        u_xxx = autograd_grad(u_xx, x_col)
        return u_t + 6.0 * g * u * u_x + g * u_xxx

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device) * (self.x_max - self.x_min) + self.x_min
        x_ic.requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_profile(x_ic).detach())
        return losses


class VKdVPaperProblem(VKdVProblem):
    name = "vkdv_paper"
    # Same PDE form as VKdVProblem in this code, but coefficients follow fixed families
    # inspired by VC-PINN vKdV experiments: linear, cubic, cosine, decaying oscillation.

    def generate_coeff(self, cfg, batch_size, q, device):
        t_cpu = q.detach().cpu()
        fields = []
        for _ in range(batch_size):
            fam = cfg.coefficient_family
            if fam == "mixed_random":
                fam = random.choice(["linear", "cubic", "cos", "exp_decay"])
            if fam == "linear":
                # scaled linear, with small random slope/offset
                a = random.uniform(0.25, 0.55)
                b = random.uniform(0.75, 1.05)
                g = b + a * t_cpu
            elif fam == "cubic":
                a = random.uniform(0.20, 0.50)
                b = random.uniform(0.80, 1.05)
                g = b + a * (t_cpu ** 3)
            elif fam == "cos":
                base = random.uniform(0.95, 1.10)
                amp = random.uniform(0.15, 0.35)
                phase = random.uniform(0, 2 * math.pi)
                g = base + amp * torch.cos(2 * math.pi * t_cpu + phase)
            elif fam == "exp_decay":
                base = random.uniform(0.90, 1.15)
                amp = random.uniform(0.15, 0.35)
                decay = random.uniform(0.8, 1.8)
                freq = random.uniform(1.0, 2.5)
                phase = random.uniform(0, 2 * math.pi)
                g = base + amp * torch.exp(-decay * t_cpu) * torch.cos(2 * math.pi * freq * t_cpu + phase)
            else:
                raise ValueError(f"Unknown vkdv_paper coefficient_family: {cfg.coefficient_family}")
            fields.append(g)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)


class VSGProblem(BaseProblem):
    name = "vsg"
    target_axis = "t"
    coeff_min = 0.50
    coeff_max = 1.50
    x_min = -1.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    k = 1.0

    def generate_coeff(self, cfg, batch_size, q, device):
        t_cpu = q.detach().cpu()
        fields = []
        for _ in range(batch_size):
            fam = cfg.coefficient_family
            if fam == "mixed_random":
                fam = random.choice(["linear", "quadratic", "cos"])
            if fam == "linear":
                h = random.uniform(0.7, 1.0) + random.uniform(0.2, 0.5) * t_cpu
            elif fam == "quadratic":
                h = random.uniform(0.7, 1.0) + random.uniform(0.2, 0.5) * (t_cpu ** 2)
            elif fam == "cos":
                h = random.uniform(0.95, 1.10) + random.uniform(0.15, 0.35) * torch.cos(2 * math.pi * t_cpu + random.uniform(0, 2 * math.pi))
            else:
                raise ValueError(f"Unknown vsg coefficient_family: {cfg.coefficient_family}")
            fields.append(h)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        h_t = coeff if coeff.shape[1] == len(t) else F.interpolate(coeff.unsqueeze(1), size=len(t), mode="linear", align_corners=True).squeeze(1)
        B, Nt = h_t.shape
        Nx = len(x)
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        omega = torch.zeros_like(h_t)
        omega[:, 1:] = torch.cumsum(0.5 * (h_t[:, 1:] + h_t[:, :-1]) * dt / self.k, dim=1)
        X = x.view(1, 1, Nx)
        Om = omega.view(B, Nt, 1)
        U = 4.0 * torch.atan(torch.exp(self.k * X - Om))
        return U

    def initial_profile(self, x):
        return 4.0 * torch.atan(torch.exp(self.k * x))

    def pde_residual(self, model, x_col, t_col):
        # u_xt + h(t) sin(u) = 0
        u = model.u(x_col, t_col)
        h = model.coeff(t_col)
        u_x = autograd_grad(u, x_col)
        u_xt = autograd_grad(u_x, t_col)
        return u_xt + h * torch.sin(u)

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device) * (self.x_max - self.x_min) + self.x_min
        x_ic.requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_profile(x_ic).detach())
        return losses


def get_problem(name: str) -> BaseProblem:
    if name == "wave":
        return WaveProblem()
    if name == "adr":
        return ADRProblem()
    if name == "vkdv":
        return VKdVProblem()
    if name == "vkdv_paper":
        return VKdVPaperProblem()
    if name == "vsg":
        return VSGProblem()
    raise ValueError(f"Unknown problem_name: {name}")


# ============================================================
# Model components
# ============================================================
class FourierFeatures(nn.Module):
    def __init__(self, in_dim: int, num_bands: int = 8, scale: float = 4.0, enabled: bool = True):
        super().__init__()
        self.in_dim = in_dim
        self.num_bands = num_bands
        self.enabled = enabled
        if enabled:
            freqs = torch.linspace(1.0, num_bands, num_bands) * scale
            self.register_buffer("freqs", freqs)
        else:
            self.register_buffer("freqs", torch.empty(0))

    @property
    def out_dim(self):
        if not self.enabled:
            return self.in_dim
        return self.in_dim + 2 * self.in_dim * self.num_bands

    def forward(self, x):
        if not self.enabled:
            return x
        outs = [x]
        for i in range(self.in_dim):
            xi = x[..., i:i + 1]
            w = self.freqs.view(*([1] * (x.ndim - 1)), -1)
            outs.append(torch.sin(2 * math.pi * xi * w))
            outs.append(torch.cos(2 * math.pi * xi * w))
        return torch.cat(outs, dim=-1)


class FeedForward(nn.Module):
    def __init__(self, d_model, mlp_ratio=4, dropout=0.0):
        super().__init__()
        hidden = d_model * mlp_ratio
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class SensorEncoder(nn.Module):
    def __init__(self, cfg: CFG, u_mean: float, u_std: float):
        super().__init__()
        ff_on = cfg.use_fourier_features
        self.coord_ff = FourierFeatures(2, num_bands=8, scale=4.0, enabled=ff_on)
        self.val_ff = FourierFeatures(1, num_bands=4, scale=4.0, enabled=ff_on)
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        in_dim = self.coord_ff.out_dim + self.val_ff.out_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, sensors):
        coords = sensors[..., :2]
        vals = (sensors[..., 2:3] - self.u_mean) / self.u_std
        z = torch.cat([self.coord_ff(coords), self.val_ff(vals)], dim=-1)
        return self.norm(self.net(z))


class QueryEmbedder(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.ff = FourierFeatures(1, num_bands=16, scale=4.0, enabled=cfg.use_fourier_features)
        self.net = nn.Sequential(
            nn.Linear(self.ff.out_dim, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, q):
        return self.norm(self.net(self.ff(q)))


class CrossAttentionBlock(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.norm_q = nn.LayerNorm(cfg.d_model)
        self.norm_kv = nn.LayerNorm(cfg.d_model)
        self.attn = nn.MultiheadAttention(cfg.d_model, cfg.n_heads, batch_first=True, dropout=cfg.dropout)
        self.norm_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)

    def forward(self, q, kv):
        qn = self.norm_q(q)
        kvn = self.norm_kv(kv)
        attn_out, _ = self.attn(qn, kvn, kvn, need_weights=False)
        x = q + attn_out
        x = x + self.ffn(self.norm_ffn(x))
        return x


class NoCrossConditioning(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )

    def forward(self, h, sensor_tokens):
        ctx = sensor_tokens.mean(dim=1, keepdim=True)
        return h + self.proj(ctx)


class DiagonalSelectiveSSM(nn.Module):
    """
    Lightweight selective diagonal SSM, not full Mamba.
    The update is input-dependent through dt and gate:
        h_k = alpha_k h_{k-1} + (1-alpha_k) B u_k
        y_k = C h_k + D u_k
    """
    def __init__(self, d_model, state_dim, dropout=0.0):
        super().__init__()
        self.d_model = d_model
        self.state_dim = state_dim
        self.in_proj = nn.Linear(d_model, d_model)
        self.gate_proj = nn.Linear(d_model, d_model)
        self.dt_proj = nn.Linear(d_model, d_model * state_dim)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.A_log = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.B = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.C = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.D = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        Bsz, L, Dm = x.shape
        u = self.in_proj(x)
        gate = torch.sigmoid(self.gate_proj(x))
        dt = F.softplus(self.dt_proj(x)).view(Bsz, L, Dm, self.state_dim) + 1e-4
        A = F.softplus(self.A_log).unsqueeze(0).unsqueeze(0)
        alpha = torch.exp(-dt * A)
        Bp = self.B.unsqueeze(0)
        Cp = self.C.unsqueeze(0)
        Dp = self.D.unsqueeze(0)
        h = torch.zeros(Bsz, Dm, self.state_dim, device=x.device, dtype=x.dtype)
        ys = []
        for k in range(L):
            uk = u[:, k, :].unsqueeze(-1)
            h = alpha[:, k] * h + (1.0 - alpha[:, k]) * (uk * Bp)
            yk = (h * Cp).sum(dim=-1) + Dp * u[:, k, :]
            ys.append(yk)
        y = torch.stack(ys, dim=1)
        y = gate * y
        return self.out_proj(self.dropout(y))


class BiSSMBlock(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.norm_f = nn.LayerNorm(cfg.d_model)
        self.norm_b = nn.LayerNorm(cfg.d_model)
        self.fwd = DiagonalSelectiveSSM(cfg.d_model, cfg.state_dim, cfg.dropout)
        self.bwd = DiagonalSelectiveSSM(cfg.d_model, cfg.state_dim, cfg.dropout)
        self.mix = nn.Linear(2 * cfg.d_model, cfg.d_model)
        self.norm_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)

    def forward(self, x):
        xf = self.fwd(self.norm_f(x))
        xb = torch.flip(self.bwd(torch.flip(self.norm_b(x), dims=[1])), dims=[1])
        x = x + self.mix(torch.cat([xf, xb], dim=-1))
        x = x + self.ffn(self.norm_ffn(x))
        return x


class CABiSSMInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.cfg = cfg
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.sensor_encoder = SensorEncoder(cfg, u_mean, u_std)
        self.query_embedder = QueryEmbedder(cfg)
        self.cross_blocks = nn.ModuleList()
        self.nocross_blocks = nn.ModuleList()
        self.ssm_blocks = nn.ModuleList()
        self.ffn_blocks = nn.ModuleList()
        for _ in range(cfg.n_layers):
            self.cross_blocks.append(CrossAttentionBlock(cfg))
            self.nocross_blocks.append(NoCrossConditioning(cfg))
            self.ssm_blocks.append(BiSSMBlock(cfg))
            self.ffn_blocks.append(nn.Sequential(nn.LayerNorm(cfg.d_model), FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)))
        self.norm = nn.LayerNorm(cfg.d_model)
        self.decoder = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(), nn.Linear(cfg.d_model, 1))

    def forward(self, sensors, q_grid):
        sensor_tokens = self.sensor_encoder(sensors)
        h = self.query_embedder(q_grid)
        for i in range(self.cfg.n_layers):
            if self.cfg.use_cross_attention:
                h = self.cross_blocks[i](h, sensor_tokens)
            else:
                h = self.nocross_blocks[i](h, sensor_tokens)
            if self.cfg.use_ssm:
                h = self.ssm_blocks[i](h)
            else:
                h = h + self.ffn_blocks[i](h)
        raw = self.decoder(self.norm(h))
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


class DeepONetInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        self.coord_ff = FourierFeatures(2, 8, 4.0, enabled=cfg.use_fourier_features)
        self.val_ff = FourierFeatures(1, 4, 4.0, enabled=cfg.use_fourier_features)
        sensor_in = self.coord_ff.out_dim + self.val_ff.out_dim
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_in, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_width),
        )
        self.branch = nn.Sequential(
            nn.Linear(2 * cfg.deeponet_width, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_p),
        )
        self.trunk_ff = FourierFeatures(1, 16, 4.0, enabled=cfg.use_fourier_features)
        self.trunk = nn.Sequential(
            nn.Linear(self.trunk_ff.out_dim, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_p),
        )
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, sensors, q_grid):
        vals = (sensors[..., 2:3] - self.u_mean) / self.u_std
        z = torch.cat([self.coord_ff(sensors[..., :2]), self.val_ff(vals)], dim=-1)
        z = self.sensor_mlp(z)
        z = torch.cat([z.mean(dim=1), z.max(dim=1).values], dim=-1)
        b = self.branch(z)
        tr = self.trunk(self.trunk_ff(q_grid))
        raw = (b.unsqueeze(1) * tr).sum(dim=-1, keepdim=True) / math.sqrt(b.shape[-1]) + self.bias
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_t, modes_x):
        super().__init__()
        scale = 1 / max(1, in_channels * out_channels)
        self.modes_t = modes_t
        self.modes_x = modes_x
        self.weights_pos = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes_t, modes_x, dtype=torch.cfloat))
        self.weights_neg = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes_t, modes_x, dtype=torch.cfloat))

    def compl_mul2d(self, x, w):
        return torch.einsum("bixy,ioxy->boxy", x, w)

    def forward(self, x):
        original_dtype = x.dtype
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            B, C, T, X = x.shape
            x_ft = torch.fft.rfft2(x, dim=(-2, -1))
            out_ft = torch.zeros(B, self.weights_pos.shape[1], T, X // 2 + 1, device=x.device, dtype=torch.cfloat)
            mt = min(self.modes_t, T)
            mx = min(self.modes_x, X // 2 + 1)
            out_ft[:, :, :mt, :mx] = self.compl_mul2d(x_ft[:, :, :mt, :mx], self.weights_pos[:, :, :mt, :mx])
            out_ft[:, :, -mt:, :mx] = self.compl_mul2d(x_ft[:, :, -mt:, :mx], self.weights_neg[:, :, :mt, :mx])
            y = torch.fft.irfft2(out_ft, s=(T, X), dim=(-2, -1))
        return y.to(original_dtype)


class FNOBlock(nn.Module):
    def __init__(self, width, modes_t, modes_x):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes_t, modes_x)
        self.point = nn.Conv2d(width, width, 1)
        self.norm = nn.GroupNorm(8, width)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.point(x)))


def sensors_to_grid(cfg: CFG, problem: BaseProblem, sensors, u_mean, u_std):
    device = sensors.device
    dtype = sensors.dtype
    B, Ns, _ = sensors.shape
    T, X = cfg.fno_t_bins, cfg.fno_x_bins
    x = sensors[..., 0]
    t = sensors[..., 1]
    u = (sensors[..., 2] - u_mean.to(device)) / u_std.to(device)
    ix = torch.round((x - problem.x_min) / (problem.x_max - problem.x_min) * (X - 1)).long().clamp(0, X - 1)
    it = torch.round((t - problem.t_min) / (problem.t_max - problem.t_min) * (T - 1)).long().clamp(0, T - 1)
    obs = torch.zeros(B, T, X, device=device, dtype=dtype)
    cnt = torch.zeros(B, T, X, device=device, dtype=dtype)
    b = torch.arange(B, device=device).view(B, 1).expand(B, Ns)
    flat = (b * T * X + it * X + ix).reshape(-1)
    obs.reshape(-1).scatter_add_(0, flat, u.reshape(-1).to(dtype))
    cnt.reshape(-1).scatter_add_(0, flat, torch.ones_like(u, dtype=dtype).reshape(-1))
    mask = (cnt > 0).to(dtype)
    obs = obs / cnt.clamp_min(1.0)
    return torch.stack([obs, mask], dim=1)


class FNOEncoderInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.cfg = cfg
        self.problem = problem
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        self.lift = nn.Conv2d(4, cfg.fno_width, 1)
        self.blocks = nn.ModuleList([FNOBlock(cfg.fno_width, cfg.fno_modes_t, cfg.fno_modes_x) for _ in range(cfg.fno_layers)])
        self.q_ff = FourierFeatures(1, 16, 4.0, enabled=cfg.use_fourier_features)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fno_width + self.q_ff.out_dim, cfg.fno_width), nn.GELU(),
            nn.Linear(cfg.fno_width, cfg.fno_width), nn.GELU(),
            nn.Linear(cfg.fno_width, 1),
        )

    def forward(self, sensors, q_grid):
        B = sensors.shape[0]
        T, X = self.cfg.fno_t_bins, self.cfg.fno_x_bins
        grid = sensors_to_grid(self.cfg, self.problem, sensors, self.u_mean, self.u_std)
        x_coord = torch.linspace(self.problem.x_min, self.problem.x_max, X, device=sensors.device, dtype=sensors.dtype).view(1, 1, 1, X).expand(B, 1, T, X)
        t_coord = torch.linspace(self.problem.t_min, self.problem.t_max, T, device=sensors.device, dtype=sensors.dtype).view(1, 1, T, 1).expand(B, 1, T, X)
        z = torch.cat([grid, x_coord, t_coord], dim=1)
        z = self.lift(z)
        for blk in self.blocks:
            z = blk(z)
        latent = z.mean(dim=(-2, -1))
        latent = latent.unsqueeze(1).expand(B, q_grid.shape[1], latent.shape[-1])
        raw = self.decoder(torch.cat([latent, self.q_ff(q_grid)], dim=-1))
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


# ============================================================
# Training helpers
# ============================================================
def compute_train_stats(ds: InversePDEDataset):
    u = ds.sensors[..., 2]
    c = ds.coeff_true
    g = gradient_1d(c)
    return {
        "u_mean": float(u.mean().item()),
        "u_std": float(u.std().item() + 1e-6),
        "c_mean": float(c.mean().item()),
        "c_std": float(c.std().item() + 1e-6),
        "g_std": float(g.std().item() + 1e-6),
    }


def supervised_loss(c_hat, c_true, stats, cfg):
    loss_field = F.mse_loss((c_hat - c_true) / stats["c_std"], torch.zeros_like(c_hat))
    if cfg.lambda_grad > 0:
        loss_grad = F.mse_loss((gradient_1d(c_hat) - gradient_1d(c_true)) / stats["g_std"], torch.zeros_like(gradient_1d(c_hat)))
    else:
        loss_grad = torch.tensor(0.0, device=c_hat.device)
    loss_tv = total_variation_1d(c_hat)
    return loss_field + cfg.lambda_grad * loss_grad + cfg.lambda_tv * loss_tv, {
        "loss_field": float(loss_field.detach().item()),
        "loss_grad": float(loss_grad.detach().item()),
        "loss_tv": float(loss_tv.detach().item()),
    }


def build_model(model_key: str, cfg: CFG, problem: BaseProblem, stats: Dict[str, float]) -> nn.Module:
    if model_key == "cabissm":
        return CABiSSMInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    if model_key == "deeponet":
        return DeepONetInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    if model_key == "fnoenc":
        return FNOEncoderInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    raise ValueError(model_key)


def nice_name(model_key: str) -> str:
    return {"cabissm": "CABiSSM", "deeponet": "DeepONet", "fnoenc": "FNOEnc"}.get(model_key, model_key)


def train_amortized_model(cfg, problem, model, model_name, train_ds, val_ds, stats, device, out_dir, epochs):
    train_loader = make_loader(train_ds, cfg, True)
    val_loader = make_loader(val_ds, cfg, False)
    model = model.to(device)
    if cfg.use_compile and hasattr(torch, "compile"):
        model = torch.compile(model)
    print(f"[{problem.name}] {model_name} parameters: {count_params(model):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    amp_dtype = get_amp_dtype(cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.use_amp and device.type == "cuda" and amp_dtype == torch.float16))
    history = {"train_loss": [], "val_loss": [], "val_mae": [], "val_rel_l2": [], "val_grad_mae": [], "val_norm_mae": [], "val_var_ratio": []}
    best_val = float("inf")
    best_metrics = None
    best_epoch = -1
    best_state = None
    no_improve = 0
    best_path = out_dir / f"{problem.name}_{model_name.lower()}_best.pt"
    start_train = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        train_total, train_count = 0.0, 0
        for batch in train_loader:
            batch = move_batch(batch, device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(cfg.use_amp and device.type == "cuda")):
                pred = model(batch["sensors"], batch["q_grid"])
                loss, _ = supervised_loss(pred, batch["coeff_true"], stats, cfg)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                opt.step()
            bs = batch["sensors"].shape[0]
            train_total += float(loss.detach()) * bs
            train_count += bs
        sched.step()
        model.eval()
        val_total, val_count = 0.0, 0
        metric_total: Dict[str, float] = {}
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch(batch, device)
                with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(cfg.use_amp and device.type == "cuda")):
                    pred = model(batch["sensors"], batch["q_grid"])
                    loss, _ = supervised_loss(pred, batch["coeff_true"], stats, cfg)
                bs = batch["sensors"].shape[0]
                val_total += float(loss.detach()) * bs
                val_count += bs
                m = coeff_metrics(pred.float(), batch["coeff_true"].float())
                for k, v in m.items():
                    metric_total[k] = metric_total.get(k, 0.0) + v * bs
        train_loss = train_total / max(train_count, 1)
        val_loss = val_total / max(val_count, 1)
        metrics = {k: v / max(val_count, 1) for k, v in metric_total.items()}
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(metrics["mae"])
        history["val_rel_l2"].append(metrics["rel_l2"])
        history["val_grad_mae"].append(metrics["grad_mae"])
        history["val_norm_mae"].append(metrics["norm_mae"])
        history["val_var_ratio"].append(metrics["var_ratio"])
        print(
            f"[{problem.name}] {model_name} ep {ep:03d} | train {train_loss:.3e} | val {val_loss:.3e} | "
            f"mae {metrics['mae']:.3e} | relL2 {metrics['rel_l2']:.3e} | grad {metrics['grad_mae']:.3e} | "
            f"normMAE {metrics['norm_mae']:.3e} | varRatio {metrics['var_ratio']:.3f}"
        )
        if val_loss < best_val:
            best_val = val_loss
            best_metrics = metrics
            best_epoch = ep
            no_improve = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if cfg.save_checkpoints:
                torch.save({"model": best_state, "metrics": metrics, "best_val": best_val, "epoch": ep, "history": history, "stats": stats}, best_path)
        else:
            no_improve += 1
        if model_name.startswith("CABiSSM") and no_improve >= cfg.early_stop_patience:
            print(f"[{problem.name}] Early stopping {model_name} at epoch {ep}; best epoch {best_epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    train_time = time.time() - start_train
    return model, history, best_metrics or {}, best_val, best_epoch, train_time


@torch.no_grad()
def evaluate_subset(model, ds, indices, device):
    model.eval()
    metrics, preds = [], []
    for idx in indices:
        s = ds[idx]
        sensors = s["sensors"].unsqueeze(0).to(device)
        q = s["q_grid"].unsqueeze(0).to(device)
        true = s["coeff_true"].unsqueeze(0).to(device)
        pred = model(sensors, q)
        metrics.append(coeff_metrics(pred.float(), true.float()))
        preds.append(pred.squeeze(0).cpu())
    return metrics, preds


@torch.no_grad()
def benchmark_inference(model, ds, cfg, device, n_batches=10):
    loader = make_loader(ds, cfg, False)
    model.eval()
    times = []
    count = 0
    # warmup
    for i, batch in enumerate(loader):
        batch = move_batch(batch, device)
        _ = model(batch["sensors"], batch["q_grid"])
        if i >= 2:
            break
    if device.type == "cuda":
        torch.cuda.synchronize()
    for i, batch in enumerate(loader):
        if i >= n_batches:
            break
        batch = move_batch(batch, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        _ = model(batch["sensors"], batch["q_grid"])
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - start
        times.append(elapsed)
        count += batch["sensors"].shape[0]
    total = sum(times)
    return {"inference_time_sec_total": total, "inference_samples": count, "inference_ms_per_sample": 1000.0 * total / max(count, 1)}


# ============================================================
# VC-PINN-style baseline
# ============================================================
def autograd_grad(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True, retain_graph=True)[0]


class PreActResidualBlock(nn.Module):
    def __init__(self, width, layers_per_block=2):
        super().__init__()
        layers = []
        for _ in range(layers_per_block):
            layers += [nn.Tanh(), nn.Linear(width, width)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.net(x)


class ResNetMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, blocks, layers_per_block):
        super().__init__()
        self.input = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([PreActResidualBlock(hidden, layers_per_block) for _ in range(blocks)])
        self.output = nn.Linear(hidden, out_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.input(x)
        for block in self.blocks:
            h = block(h)
        return self.output(h)


class VCPINNModel(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem):
        super().__init__()
        self.problem = problem
        self.u_net = ResNetMLP(2, cfg.vc_hidden, 1, cfg.vc_blocks_u, cfg.vc_layers_per_block)
        self.c_net = ResNetMLP(1, cfg.vc_hidden, 1, cfg.vc_blocks_c, cfg.vc_layers_per_block)

    def u(self, x, t):
        return self.u_net(torch.cat([x, t], dim=-1))

    def coeff(self, q):
        raw = self.c_net(q)
        return self.problem.coeff_min + (self.problem.coeff_max - self.problem.coeff_min) * torch.sigmoid(raw)


def vc_pinn_loss(problem, model, sample, cfg, device, colloc):
    sensors = sample["sensors"].to(device).float()
    q_grid = sample["q_grid"].to(device).float()
    coeff_true = sample["coeff_true"].to(device).float()
    x_s = sensors[:, 0:1]
    t_s = sensors[:, 1:2]
    u_s = sensors[:, 2:3]
    loss_data = F.mse_loss(model.u(x_s, t_s), u_s)
    x_f = colloc["x_f"].clone().detach().requires_grad_(True)
    t_f = colloc["t_f"].clone().detach().requires_grad_(True)
    loss_pde = torch.mean(problem.pde_residual(model, x_f, t_f) ** 2)
    aux = problem.aux_losses(model, sample, cfg, device)
    loss_ic = aux.get("ic", torch.tensor(0.0, device=device))
    loss_bc = aux.get("bc", torch.tensor(0.0, device=device))
    c0_pred = model.coeff(q_grid[0:1])
    c1_pred = model.coeff(q_grid[-1:])
    loss_cbc = F.mse_loss(c0_pred, coeff_true[0:1]) + F.mse_loss(c1_pred, coeff_true[-1:])
    c_grid = model.coeff(q_grid)
    loss_smooth = total_variation_1d(c_grid.unsqueeze(0))
    loss = (
        cfg.vc_w_data * loss_data + cfg.vc_w_pde * loss_pde + cfg.vc_w_ic * loss_ic +
        cfg.vc_w_bc * loss_bc + cfg.vc_w_cbc * loss_cbc + cfg.vc_w_c_smooth * loss_smooth
    )
    parts = {"data": float(loss_data.detach()), "pde": float(loss_pde.detach()), "ic": float(loss_ic.detach()), "bc": float(loss_bc.detach()), "cbc": float(loss_cbc.detach()), "smooth": float(loss_smooth.detach())}
    return loss, parts


@torch.no_grad()
def eval_vc_coeff(model, sample, device):
    q_grid = sample["q_grid"].to(device).float()
    coeff_true = sample["coeff_true"].to(device).float()
    pred = model.coeff(q_grid)
    return pred.cpu(), coeff_metrics(pred.unsqueeze(0), coeff_true.unsqueeze(0))


def run_vc_pinn_sample(problem, cfg, sample, device):
    model = VCPINNModel(cfg, problem).to(device).float()
    opt = torch.optim.Adam(model.parameters(), lr=cfg.vc_lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.vc_adam_steps)
    colloc = {
        "x_f": torch.rand(cfg.vc_colloc_f, 1, device=device) * (problem.x_max - problem.x_min) + problem.x_min,
        "t_f": torch.rand(cfg.vc_colloc_f, 1, device=device) * (problem.t_max - problem.t_min) + problem.t_min,
    }
    best_state, best_loss = None, float("inf")
    hist = []
    for step in range(1, cfg.vc_adam_steps + 1):
        opt.zero_grad(set_to_none=True)
        loss, parts = vc_pinn_loss(problem, model, sample, cfg, device, colloc)
        loss.backward()
        opt.step()
        sched.step()
        lv = float(loss.detach())
        hist.append(lv)
        if lv < best_loss:
            best_loss = lv
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if step == 1 or step % cfg.vc_log_every == 0 or step == cfg.vc_adam_steps:
            with torch.no_grad():
                _, m = eval_vc_coeff(model, sample, device)
            print(f"    [{problem.name}] VC-PINN {step:04d}/{cfg.vc_adam_steps} | loss {lv:.3e} | data {parts['data']:.2e} | pde {parts['pde']:.2e} | mae {m['mae']:.3e} | relL2 {m['rel_l2']:.3e}")
    if best_state is not None:
        model.load_state_dict(best_state)
    if cfg.vc_lbfgs_steps > 0:
        print(f"    [{problem.name}] VC-PINN L-BFGS refinement: {cfg.vc_lbfgs_steps} iters")
        lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=cfg.vc_lbfgs_steps, max_eval=2 * cfg.vc_lbfgs_steps, history_size=50, line_search_fn="strong_wolfe")
        def closure():
            lbfgs.zero_grad(set_to_none=True)
            loss, _ = vc_pinn_loss(problem, model, sample, cfg, device, colloc)
            loss.backward()
            return loss
        lbfgs.step(closure)
    pred, metrics = eval_vc_coeff(model, sample, device)
    return pred, metrics, hist


def run_vc_pinn_baseline(problem, cfg, val_ds, device, indices):
    metrics, preds, histories = [], [], []
    print(f"[{problem.name}] Running VC-PINN-style on {len(indices)} samples")
    total = time.time()
    for i, idx in enumerate(indices):
        print(f"  [{problem.name}] VC sample {i + 1}/{len(indices)} index={idx}")
        start = time.time()
        pred, m, hist = run_vc_pinn_sample(problem, cfg, val_ds[idx], device)
        m["wall_time_sec"] = time.time() - start
        metrics.append(m)
        preds.append(pred)
        histories.append(hist)
        print(f"  [{problem.name}] done idx={idx} | time {m['wall_time_sec']:.1f}s | MAE {m['mae']:.3e} | relL2 {m['rel_l2']:.3e}")
    print(f"[{problem.name}] VC-PINN total time: {time.time() - total:.1f}s")
    return metrics, preds, histories


# ============================================================
# Plotting and artifact saving
# ============================================================
def save_curves(history, out_dir, problem_name, method_name):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="train loss")
    plt.plot(epochs, history["val_loss"], label="val loss")
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"{problem_name}: {method_name} loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{problem_name}_{method_name}_loss.png", dpi=200)
    plt.close()
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_mae"], label="MAE")
    plt.plot(epochs, history["val_rel_l2"], label="relL2")
    plt.plot(epochs, history["val_grad_mae"], label="gradMAE")
    plt.plot(epochs, history["val_norm_mae"], label="normMAE")
    plt.xlabel("epoch")
    plt.ylabel("metric")
    plt.title(f"{problem_name}: {method_name} validation metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{problem_name}_{method_name}_metrics.png", dpi=200)
    plt.close()


def save_reconstruction_plot(problem, val_ds, indices, method_preds, out_dir, tag=""):
    if not method_preds:
        return
    n = len(indices)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))
    if n == 1:
        axes = axes[None, :]
    for row, idx in enumerate(indices):
        s = val_ds[idx]
        q = s["q_grid"].squeeze(-1)
        true = s["coeff_true"].squeeze(-1)
        sensors = s["sensors"]
        ax0 = axes[row, 0]
        sc = ax0.scatter(sensors[:, 0], sensors[:, 1], c=sensors[:, 2], s=16)
        ax0.set_xlabel("x")
        ax0.set_ylabel("t")
        ax0.set_title(f"{problem.name}: sparse sensors sample {idx}")
        plt.colorbar(sc, ax=ax0, fraction=0.046, pad=0.04)
        ax1 = axes[row, 1]
        ax1.plot(q, true, label="true coefficient", linewidth=2)
        for name, preds in method_preds.items():
            ax1.plot(q, preds[row].squeeze(-1), label=name, linewidth=2)
        ax1.set_xlabel(problem.target_axis)
        ax1.set_ylabel("coefficient")
        ax1.set_title(f"{problem.name}: coefficient reconstruction")
        ax1.legend(fontsize=8)
    plt.tight_layout()
    suffix = f"_{tag}" if tag else ""
    plt.savefig(out_dir / f"{problem.name}{suffix}_reconstructions.png", dpi=200)
    plt.close()


def save_barplot(problem_name, method_metrics, out_dir, tag=""):
    if not method_metrics:
        return
    keys = ["mae", "rel_l2", "grad_mae", "norm_mae", "var_ratio"]
    methods = list(method_metrics.keys())
    x = list(range(len(keys)))
    width = 0.8 / max(len(methods), 1)
    plt.figure(figsize=(12, 5))
    for i, method in enumerate(methods):
        vals = [mean_metric(method_metrics[method], k) for k in keys]
        offset = (i - (len(methods) - 1) / 2) * width
        plt.bar([j + offset for j in x], vals, width=width, label=method)
    plt.xticks(x, keys)
    plt.ylabel("metric value")
    plt.title(f"{problem_name}: comparison metrics")
    plt.legend()
    plt.tight_layout()
    suffix = f"_{tag}" if tag else ""
    plt.savefig(out_dir / f"{problem_name}{suffix}_barplot.png", dpi=200)
    plt.close()


def save_table_csv(rows: List[Dict[str, Any]], path: Path):
    if not rows:
        return
    keys = sorted({k for r in rows for k in r.keys()})
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def save_summary_tables(summary: Dict, out_dir: Path):
    full_rows = []
    for method, metrics in summary.get("full_val_metrics", {}).items():
        row = {"method": method}
        row.update(metrics)
        full_rows.append(row)
    save_table_csv(full_rows, out_dir / "full_val_metrics.csv")
    subset_rows = []
    for method, metrics in summary.get("subset_avg", {}).items():
        row = {"method": method}
        row.update(metrics)
        subset_rows.append(row)
    save_table_csv(subset_rows, out_dir / "subset_avg_metrics.csv")
    runtime_rows = []
    for method, metrics in summary.get("runtime", {}).items():
        row = {"method": method}
        row.update(metrics)
        runtime_rows.append(row)
    save_table_csv(runtime_rows, out_dir / "runtime_metrics.csv")


def add_text_page(pdf, title: str, lines: List[str], max_lines=42):
    for start in range(0, max(len(lines), 1), max_lines):
        chunk = lines[start:start + max_lines]
        fig = plt.figure(figsize=(11, 8.5))
        ax = fig.add_subplot(111)
        ax.axis("off")
        ax.text(0.02, 0.97, title if start == 0 else title + " (continued)", fontsize=16, fontweight="bold", va="top", family="monospace")
        y = 0.91
        for line in chunk:
            ax.text(0.02, y, str(line)[:150], fontsize=9, va="top", family="monospace")
            y -= 0.021
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)


def add_image_page(pdf, image_path: Path, title: str):
    if not image_path.exists():
        return
    try:
        img = plt.imread(str(image_path))
    except Exception:
        return
    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_subplot(111)
    ax.imshow(img)
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def save_pdf_report(summary: Dict, out_dir: Path):
    pdf_path = out_dir / "summary_report.pdf"
    with PdfPages(pdf_path) as pdf:
        lines = [
            f"Problem: {summary.get('problem')}",
            f"Run mode: {summary.get('run_mode')}",
            f"Coefficient family: {summary.get('coefficient_family')}",
            f"Target axis: {summary.get('target_axis')}",
            f"Coefficient range: {summary.get('coefficient_range')}",
            "",
            "Train stats:",
        ]
        for k, v in summary.get("train_stats", {}).items():
            lines.append(f"  {k}: {v}")
        add_text_page(pdf, "Experiment overview", lines)

        if "modes" in summary:
            mode_lines = []
            for mode_name, mode_summary in summary.get("modes", {}).items():
                mode_lines.append("=" * 80)
                mode_lines.append(f"MODE: {mode_name}")
                mode_lines.append("=" * 80)
                if "full_val_metrics" in mode_summary:
                    mode_lines.append("Full validation metrics:")
                    for method, metrics in mode_summary.get("full_val_metrics", {}).items():
                        mae = metrics.get("mae", None)
                        rel = metrics.get("rel_l2", None)
                        grad = metrics.get("grad_mae", None)
                        varr = metrics.get("var_ratio", None)
                        mode_lines.append(f"  {method}: MAE={mae}, relL2={rel}, gradMAE={grad}, varRatio={varr}")
                if "subset_avg" in mode_summary:
                    mode_lines.append("Subset averages:")
                    for method, metrics in mode_summary.get("subset_avg", {}).items():
                        mae = metrics.get("mae", None)
                        rel = metrics.get("rel_l2", None)
                        grad = metrics.get("grad_mae", None)
                        varr = metrics.get("var_ratio", None)
                        mode_lines.append(f"  {method}: MAE={mae}, relL2={rel}, gradMAE={grad}, varRatio={varr}")
                mode_lines.append("")
            add_text_page(pdf, "All-modes combined summary", mode_lines)

        full_lines = []
        for method, metrics in summary.get("full_val_metrics", {}).items():
            full_lines.append(method)
            for k, v in metrics.items():
                full_lines.append(f"  {k:<18}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            full_lines.append("")
        add_text_page(pdf, "Full validation metrics", full_lines)

        subset_lines = []
        for method, metrics in summary.get("subset_avg", {}).items():
            subset_lines.append(method)
            for k, v in metrics.items():
                subset_lines.append(f"  {k:<18}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            subset_lines.append("")
        add_text_page(pdf, "Subset metrics", subset_lines)

        runtime_lines = []
        for method, metrics in summary.get("runtime", {}).items():
            runtime_lines.append(method)
            for k, v in metrics.items():
                runtime_lines.append(f"  {k:<25}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            runtime_lines.append("")
        add_text_page(pdf, "Runtime metrics", runtime_lines)

        for png in sorted(out_dir.rglob("*.png")):
            add_image_page(pdf, png, str(png.relative_to(out_dir)))
    return pdf_path


def make_results_zip(out_dir: Path):
    zip_path = out_dir.parent / f"{out_dir.name}.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file in out_dir.rglob("*"):
            if file.is_file() and file != zip_path:
                zf.write(file, arcname=file.relative_to(out_dir.parent))
    return zip_path


# ============================================================
# Experiment runners
# ============================================================
def run_main_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path, tag: str = "main") -> Dict:
    ensure_dir(out_dir)
    train_ds = problem.build_dataset(cfg, cfg.train_size, device, tag=f"train_{tag}")
    val_ds = problem.build_dataset(cfg, cfg.val_size, device, tag=f"val_{tag}")
    stats = compute_train_stats(train_ds)
    print(f"[{problem.name}] train stats: {stats}")

    full_val_metrics: Dict[str, Dict] = {}
    subset_metrics: Dict[str, List[Dict]] = {}
    subset_preds: Dict[str, List[torch.Tensor]] = {}
    histories: Dict[str, Any] = {}
    runtime: Dict[str, Dict] = {}
    best_epochs: Dict[str, int] = {}

    subset_indices = list(range(min(cfg.vc_eval_samples, len(val_ds))))

    for model_key in cfg.run_methods:
        if model_key == "vc_pinn":
            continue
        model_name = nice_name(model_key)
        model = build_model(model_key, cfg, problem, stats)
        model, hist, metrics, best_val, best_epoch, train_time = train_amortized_model(cfg, problem, model, model_name, train_ds, val_ds, stats, device, out_dir, cfg.epochs if model_key == "cabissm" else cfg.baseline_epochs)
        histories[model_name] = hist
        full_val_metrics[model_name] = metrics
        best_epochs[model_name] = best_epoch
        runtime[model_name] = {"train_time_sec": train_time, "train_time_min": train_time / 60.0}
        runtime[model_name].update(benchmark_inference(model, val_ds, cfg, device, n_batches=10))
        save_curves(hist, out_dir, problem.name, model_name)
        m, p = evaluate_subset(model, val_ds, subset_indices, device)
        subset_metrics[model_name] = m
        subset_preds[model_name] = p

    if "vc_pinn" in cfg.run_methods:
        m, p, h = run_vc_pinn_baseline(problem, cfg, val_ds, device, subset_indices)
        subset_metrics["VC-PINN-style"] = m
        subset_preds["VC-PINN-style"] = p
        histories["VC-PINN-style"] = h
        runtime["VC-PINN-style"] = {
            "subset_total_time_sec": sum(x.get("wall_time_sec", 0.0) for x in m),
            "mean_time_per_sample_sec": mean_metric(m, "wall_time_sec"),
        }

    save_reconstruction_plot(problem, val_ds, subset_indices, subset_preds, out_dir, tag=tag)
    save_barplot(problem.name, subset_metrics, out_dir, tag=tag)

    summary = {
        "problem": problem.name,
        "run_mode": cfg.run_mode,
        "tag": tag,
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "train_stats": stats,
        "best_epochs": best_epochs,
        "full_val_metrics": full_val_metrics,
        "subset_avg": {method: {k: mean_metric(metrics, k) for k in metrics[0].keys()} for method, metrics in subset_metrics.items()},
        "runtime": runtime,
    }
    save_json(summary, out_dir / f"{tag}_summary.json")
    save_summary_tables(summary, out_dir)
    return summary


def ablation_cfg(base: CFG, name: str) -> CFG:
    cfg = replace(base)
    cfg.run_methods = ("cabissm",)
    cfg.epochs = base.epochs
    if name == "full_cabissm":
        pass
    elif name == "no_ssm_attention_only":
        cfg.use_ssm = False
    elif name == "no_cross_ssm_only":
        cfg.use_cross_attention = False
    elif name == "no_fourier_features":
        cfg.use_fourier_features = False
    elif name == "no_gradient_loss":
        cfg.lambda_grad = 0.0
    else:
        raise ValueError(name)
    return cfg


def run_ablation_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path) -> Dict:
    ensure_dir(out_dir)
    # Use the same dataset across ablations for fairness.
    train_ds = problem.build_dataset(cfg, cfg.train_size, device, tag="train_ablation_shared")
    val_ds = problem.build_dataset(cfg, cfg.val_size, device, tag="val_ablation_shared")
    base_stats = compute_train_stats(train_ds)
    subset_indices = list(range(min(cfg.vc_eval_samples, len(val_ds))))
    all_metrics, all_runtime, all_histories, subset_metrics, subset_preds = {}, {}, {}, {}, {}
    for abl in cfg.ablation_names:
        print("\n" + "=" * 100)
        print(f"Ablation: {abl}")
        print("=" * 100)
        acfg = ablation_cfg(cfg, abl)
        model_name = f"CABiSSM_{abl}"
        model = CABiSSMInverse(acfg, problem, base_stats["u_mean"], base_stats["u_std"])
        model, hist, metrics, best_val, best_epoch, train_time = train_amortized_model(acfg, problem, model, model_name, train_ds, val_ds, base_stats, device, out_dir, acfg.epochs)
        all_metrics[model_name] = metrics
        all_runtime[model_name] = {"train_time_sec": train_time, "train_time_min": train_time / 60.0}
        all_runtime[model_name].update(benchmark_inference(model, val_ds, acfg, device, n_batches=10))
        all_histories[model_name] = hist
        save_curves(hist, out_dir, problem.name, model_name)
        m, p = evaluate_subset(model, val_ds, subset_indices, device)
        subset_metrics[model_name] = m
        subset_preds[model_name] = p
    save_reconstruction_plot(problem, val_ds, subset_indices, subset_preds, out_dir, tag="ablation")
    save_barplot(problem.name, subset_metrics, out_dir, tag="ablation")
    summary = {
        "problem": problem.name,
        "run_mode": "ablation",
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "train_stats": base_stats,
        "full_val_metrics": all_metrics,
        "subset_avg": {method: {k: mean_metric(metrics, k) for k in metrics[0].keys()} for method, metrics in subset_metrics.items()},
        "runtime": all_runtime,
    }
    save_json(summary, out_dir / "ablation_summary.json")
    save_summary_tables(summary, out_dir)
    return summary


def run_sweep_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path, sweep_type: str) -> Dict:
    ensure_dir(out_dir)
    values = cfg.sensor_sweep_values if sweep_type == "sensor" else cfg.noise_sweep_values
    rows = []
    all_summaries = []
    for value in values:
        if sweep_type == "sensor":
            scfg = replace(cfg, n_sensors=int(value), train_size=cfg.sweep_train_size, val_size=cfg.sweep_val_size, epochs=cfg.sweep_epochs, baseline_epochs=cfg.sweep_epochs, run_methods=("cabissm", "deeponet", "fnoenc"))
            tag = f"sensor_{int(value)}"
        else:
            scfg = replace(cfg, sensor_noise_std=float(value), train_size=cfg.sweep_train_size, val_size=cfg.sweep_val_size, epochs=cfg.sweep_epochs, baseline_epochs=cfg.sweep_epochs, run_methods=("cabissm", "deeponet", "fnoenc"))
            tag = f"noise_{float(value):.3f}".replace(".", "p")
        print("\n" + "=" * 100)
        print(f"Sweep run: {tag}")
        print("=" * 100)
        run_dir = out_dir / tag
        summary = run_main_experiment(scfg, problem, device, run_dir, tag=tag)
        all_summaries.append(summary)
        for method, metrics in summary.get("full_val_metrics", {}).items():
            row = {"sweep_type": sweep_type, "sweep_value": value, "method": method}
            row.update(metrics)
            rows.append(row)
    save_table_csv(rows, out_dir / f"{sweep_type}_sweep_metrics.csv")
    # plot MAE vs sweep value
    methods = sorted(set(r["method"] for r in rows))
    plt.figure(figsize=(8, 5))
    for method in methods:
        xs = [r["sweep_value"] for r in rows if r["method"] == method]
        ys = [r["mae"] for r in rows if r["method"] == method]
        plt.plot(xs, ys, marker="o", label=method)
    plt.xlabel("number of sensors" if sweep_type == "sensor" else "sensor noise std")
    plt.ylabel("MAE")
    plt.title(f"{problem.name}: {sweep_type} sweep")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{sweep_type}_sweep_mae.png", dpi=200)
    plt.close()
    summary = {
        "problem": problem.name,
        "run_mode": f"{sweep_type}_sweep",
        "coefficient_family": cfg.coefficient_family,
        "config": asdict(cfg),
        "rows": rows,
        "subruns": all_summaries,
    }
    save_json(summary, out_dir / f"{sweep_type}_sweep_summary.json")
    return summary


def run_all_modes_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path) -> Dict:
    """
    Runs all journal-level modes sequentially for one selected equation:
      1) main: CABiSSM + DeepONet + FNOEnc + VC-PINN-style
      2) ablation: architecture/loss ablations for CABiSSM
      3) sensor_sweep: sparse-sensor robustness
      4) noise_sweep: noisy-observation robustness
    Each mode gets its own subfolder. A combined final_summary.json, PDF, and ZIP are saved at the parent folder.
    """
    ensure_dir(out_dir)
    modes_to_run = ["main", "ablation", "sensor_sweep", "noise_sweep"]
    mode_summaries: Dict[str, Dict] = {}
    for mode in modes_to_run:
        print("\n" + "#" * 120)
        print(f"RUNNING MODE: {mode.upper()} for problem {problem.name}")
        print("#" * 120)
        mcfg = replace(cfg, run_mode=mode)
        mode_dir = out_dir / mode
        if mode == "main":
            mode_summaries[mode] = run_main_experiment(mcfg, problem, device, mode_dir, tag="main")
        elif mode == "ablation":
            mode_summaries[mode] = run_ablation_experiment(mcfg, problem, device, mode_dir)
        elif mode == "sensor_sweep":
            mode_summaries[mode] = run_sweep_experiment(mcfg, problem, device, mode_dir, sweep_type="sensor")
        elif mode == "noise_sweep":
            mode_summaries[mode] = run_sweep_experiment(mcfg, problem, device, mode_dir, sweep_type="noise")
        else:
            raise ValueError(mode)
    combined = {
        "problem": problem.name,
        "run_mode": "all_modes",
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "modes": mode_summaries,
    }
    save_json(combined, out_dir / "all_modes_summary.json")
    return combined


# ============================================================
# Main
# ============================================================
def main():
    cfg = CFG()
    set_seed(cfg.seed)
    torch.set_float32_matmul_precision("high")
    device = get_device()
    print("Device:", device)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    problem = get_problem(cfg.problem_name)
    root_dir = Path(cfg.out_root)
    family_tag = cfg.coefficient_family if problem.name in ["vkdv_paper", "vsg"] else "default"
    out_dir = root_dir / f"{problem.name}_{family_tag}_{cfg.run_mode}"
    ensure_dir(out_dir)
    print("=" * 100)
    print(f"Running problem       : {problem.name}")
    print(f"Run mode              : {cfg.run_mode}")
    print(f"Coefficient family    : {family_tag}")
    print(f"Target axis           : {problem.target_axis}")
    print(f"Coefficient range     : {problem.coeff_min} to {problem.coeff_max}")
    print(f"Output folder         : {out_dir.resolve()}")
    print("=" * 100)
    start_all = time.time()
    if cfg.run_mode == "main":
        summary = run_main_experiment(cfg, problem, device, out_dir, tag="main")
    elif cfg.run_mode == "ablation":
        summary = run_ablation_experiment(cfg, problem, device, out_dir)
    elif cfg.run_mode == "sensor_sweep":
        summary = run_sweep_experiment(cfg, problem, device, out_dir, sweep_type="sensor")
    elif cfg.run_mode == "noise_sweep":
        summary = run_sweep_experiment(cfg, problem, device, out_dir, sweep_type="noise")
    elif cfg.run_mode == "all_modes":
        summary = run_all_modes_experiment(cfg, problem, device, out_dir)
    else:
        raise ValueError(f"Unknown run_mode: {cfg.run_mode}")
    summary["total_wall_time_sec"] = time.time() - start_all
    save_json(summary, out_dir / "final_summary.json")
    pdf_path = None
    zip_path = None
    if cfg.make_pdf_summary:
        pdf_path = save_pdf_report(summary, out_dir)
        print(f"PDF report saved: {pdf_path}")
    if cfg.make_zip:
        zip_path = make_results_zip(out_dir)
        print(f"ZIP saved: {zip_path}")
    print("\nExperiment completed.")
    print("Artifacts saved in:", out_dir.resolve())
    try:
        from IPython.display import FileLink, display
        if pdf_path is not None:
            display(FileLink(str(pdf_path)))
        if zip_path is not None:
            display(FileLink(str(zip_path)))
    except Exception:
        pass


if __name__ == "__main__":
    main()


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Running problem       : wave
Run mode              : main
Coefficient family    : default
Target axis           : x
Coefficient range     : 0.75 to 1.45
Output folder         : /kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/wave_default_main
[wave] Building train_main dataset: 16384 samples, 128 simulation batches
  [wave/train_main] batch 12/128 | elapsed 0.8s
  [wave/train_main] batch 24/128 | elapsed 1.1s
  [wave/train_main] batch 36/128 | elapsed 1.3s
  [wave/train_main] batch 48/128 | elapsed 1.6s
  [wave/train_main] batch 60/128 | elapsed 1.9s
  [wave/train_main] batch 72/128 | elapsed 2.2s
  [wave/train_main] batch 84/128 | elapsed 2.4s
  [wave/train_main] batch 96/128 | elapsed 2.7s
  [wave/train_main] batch 108/128 | elapsed 3.0s
  [wave/train_main] batch 120/128 | elapsed 3.3s
  [wave/train_main] batch 128/128 | elapsed 3.5s
[wave] Building val_main dataset: 2048 samples, 16 simulation batches
  [wave/va

/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/wave_default_main/summary_report.pdf

/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/wave_default_main.zip

In [2]:
# ============================================================
# Publishable inverse-PDE pipeline
# CABiSSM vs DeepONet vs FNOEnc vs VC-PINN-style
# Includes: equation selection, baselines, ablations, robustness sweeps,
# runtime analysis, normalized metrics, variance-ratio metrics, PDF + ZIP export.
#
# Recommended Kaggle use:
#   1) Paste this full script into one Kaggle notebook cell.
#   2) Edit ONLY the USER SETTINGS block.
#   3) Run one problem / one experiment mode at a time.
# ============================================================

import csv
import json
import math
import random
import time
import zipfile
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# USER SETTINGS: change this block only
# ============================================================
@dataclass
class CFG:
    # ------------------------------
    # Main experiment selection
    # ------------------------------
    # Available problems:
    #   "wave"         : variable wave speed c(x)
    #   "adr"          : variable diffusion D(x)
    #   "vkdv"         : random time-coefficient KdV-style g(t)
    #   "vkdv_paper"   : VC-PINN-paper-inspired vKdV coefficient families
    #   "vsg"          : variable-coefficient Sine-Gordon h(t)
    problem_name: str = "vkdv"

    # Run modes:
    #   "main"          : main model + baselines
    #   "ablation"      : CABiSSM ablation study
    #   "sensor_sweep"  : sparse sensor robustness sweep
    #   "noise_sweep"   : observation-noise robustness sweep
    #   "all_modes"     : run main + ablation + sensor_sweep + noise_sweep sequentially for one equation
    run_mode: str = "main"

    # For vkdv_paper and vsg. Options are problem-dependent.
    # vkdv_paper coefficient_family: "linear", "cubic", "cos", "exp_decay", "mixed_random"
    # vsg coefficient_family       : "linear", "quadratic", "cos", "mixed_random"
    coefficient_family: str = "mixed_random"

    # Main methods.
    # For main runs, keep all enabled.
    run_methods: Tuple[str, ...] = ("cabissm", "deeponet", "fnoenc", "vc_pinn")

    # ------------------------------
    # Data sizes
    # ------------------------------
    train_size: int = 16384
    val_size: int = 2048
    n_sensors: int = 128
    sensor_noise_std: float = 0.01
    sim_batch_size: int = 128

    # Grids
    n_x: int = 129
    n_t: int = 301
    n_q: int = 129

    # ------------------------------
    # Amortized model training
    # ------------------------------
    batch_size: int = 64
    epochs: int = 50
    baseline_epochs: int = 50
    lr: float = 3e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    lambda_grad: float = 0.5
    lambda_tv: float = 1e-4
    early_stop_patience: int = 12

    # Runtime
    seed: int = 42
    num_workers: int = 2
    use_amp: bool = True
    amp_dtype: str = "bfloat16"  # good on RTX PRO 6000 / H100; FFT sections force FP32 internally
    use_compile: bool = False
    out_root: str = "/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline"

    # ------------------------------
    # CABiSSM architecture
    # ------------------------------
    d_model: int = 128
    n_heads: int = 8
    n_layers: int = 4
    state_dim: int = 8
    mlp_ratio: int = 4
    dropout: float = 0.0
    use_fourier_features: bool = True
    use_cross_attention: bool = True
    use_ssm: bool = True

    # ------------------------------
    # DeepONet baseline
    # ------------------------------
    deeponet_width: int = 128
    deeponet_p: int = 128

    # ------------------------------
    # FNO-Encoder baseline
    # ------------------------------
    fno_width: int = 48
    fno_modes_t: int = 16
    fno_modes_x: int = 16
    fno_layers: int = 4
    fno_t_bins: int = 129
    fno_x_bins: int = 129

    # ------------------------------
    # VC-PINN-style baseline
    # ------------------------------
    # This is sample-wise, so full validation is expensive.
    vc_eval_samples: int = 16
    vc_hidden: int = 128
    vc_blocks_u: int = 4
    vc_blocks_c: int = 3
    vc_layers_per_block: int = 2
    vc_adam_steps: int = 2000
    vc_lbfgs_steps: int = 100
    vc_lr: float = 1e-3
    vc_colloc_f: int = 4096
    vc_colloc_ic: int = 512
    vc_colloc_bc: int = 512
    vc_w_data: float = 10.0
    vc_w_pde: float = 1.0
    vc_w_ic: float = 10.0
    vc_w_bc: float = 10.0
    vc_w_cbc: float = 10.0
    vc_w_c_smooth: float = 1e-4
    vc_log_every: int = 500

    # ------------------------------
    # Journal-strength extra experiments
    # ------------------------------
    sensor_sweep_values: Tuple[int, ...] = (16, 32, 64, 128)
    noise_sweep_values: Tuple[float, ...] = (0.0, 0.01, 0.03, 0.05)

    # For sweeps, use smaller sizes if you want a quick result.
    sweep_train_size: int = 8192
    sweep_val_size: int = 1024
    sweep_epochs: int = 35

    # Ablation variants.
    # These are trained only when run_mode == "ablation".
    ablation_names: Tuple[str, ...] = (
        "full_cabissm",
        "no_ssm_attention_only",
        "no_cross_ssm_only",
        "no_fourier_features",
        "no_gradient_loss",
    )

    # Artifacts
    make_pdf_summary: bool = True
    make_zip: bool = True
    save_checkpoints: bool = True


# ============================================================
# Utilities
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_amp_dtype(cfg: CFG):
    return torch.bfloat16 if cfg.amp_dtype.lower() == "bfloat16" else torch.float16


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def gradient_1d(field: torch.Tensor) -> torch.Tensor:
    return field[:, 1:, :] - field[:, :-1, :]


def total_variation_1d(field: torch.Tensor) -> torch.Tensor:
    return (field[:, 1:, :] - field[:, :-1, :]).abs().mean()


@torch.no_grad()
def coeff_metrics(c_hat: torch.Tensor, c_true: torch.Tensor) -> Dict[str, float]:
    c_hat = c_hat.float()
    c_true = c_true.float()

    mse = F.mse_loss(c_hat, c_true)
    mae = F.l1_loss(c_hat, c_true)
    rel_l2 = torch.norm(c_hat - c_true) / (torch.norm(c_true) + 1e-8)
    grad_mae = F.l1_loss(gradient_1d(c_hat), gradient_1d(c_true))

    pred_mean = c_hat.mean()
    pred_std = c_hat.std()
    true_mean = c_true.mean()
    true_std = c_true.std()

    norm_mae = mae / (true_std + 1e-8)
    norm_rmse = torch.sqrt(mse) / (true_std + 1e-8)
    var_ratio = pred_std / (true_std + 1e-8)
    mean_bias = pred_mean - true_mean

    return {
        "mse": float(mse.item()),
        "mae": float(mae.item()),
        "rel_l2": float(rel_l2.item()),
        "grad_mae": float(grad_mae.item()),
        "norm_mae": float(norm_mae.item()),
        "norm_rmse": float(norm_rmse.item()),
        "var_ratio": float(var_ratio.item()),
        "mean_bias": float(mean_bias.item()),
        "pred_mean": float(pred_mean.item()),
        "pred_std": float(pred_std.item()),
        "true_mean": float(true_mean.item()),
        "true_std": float(true_std.item()),
    }


def mean_metric(metrics: List[Dict[str, float]], key: str) -> float:
    vals = [m[key] for m in metrics if key in m]
    return sum(vals) / max(len(vals), 1)


def save_json(obj: Dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def move_batch(batch: Dict[str, torch.Tensor], device: torch.device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# Dataset
# ============================================================
class InversePDEDataset(Dataset):
    def __init__(self, sensors: torch.Tensor, q_grid: torch.Tensor, coeff_true: torch.Tensor):
        self.sensors = sensors.float()
        self.q_grid = q_grid.float()
        self.coeff_true = coeff_true.float()

    def __len__(self):
        return self.sensors.shape[0]

    def __getitem__(self, idx):
        return {
            "sensors": self.sensors[idx],
            "q_grid": self.q_grid[idx],
            "coeff_true": self.coeff_true[idx],
        }


def make_loader(ds: Dataset, cfg: CFG, shuffle: bool):
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(cfg.num_workers > 0),
    )


# ============================================================
# PDE problem classes
# ============================================================
class BaseProblem:
    name = "base"
    target_axis = "x"  # "x" or "t"
    coeff_min = 0.0
    coeff_max = 1.0
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0

    def make_grids(self, cfg: CFG, device: torch.device):
        x = torch.linspace(self.x_min, self.x_max, cfg.n_x, device=device)
        t = torch.linspace(self.t_min, self.t_max, cfg.n_t, device=device)
        if self.target_axis == "x":
            q = torch.linspace(self.x_min, self.x_max, cfg.n_q, device=device)
        else:
            q = torch.linspace(self.t_min, self.t_max, cfg.n_q, device=device)
        return x, t, q

    def generate_coeff(self, cfg: CFG, batch_size: int, q: torch.Tensor, device: torch.device):
        raise NotImplementedError

    def solve(self, cfg: CFG, coeff: torch.Tensor, x: torch.Tensor, t: torch.Tensor, q: torch.Tensor):
        raise NotImplementedError

    def sample_sensors(self, cfg: CFG, U: torch.Tensor, x: torch.Tensor, t: torch.Tensor):
        B = U.shape[0]
        Ns = cfg.n_sensors
        device = U.device
        ix = torch.randint(0, len(x), (B, Ns), device=device)
        it = torch.randint(0, len(t), (B, Ns), device=device)
        b = torch.arange(B, device=device).unsqueeze(1).expand(B, Ns)
        u = U[b, it, ix]
        if cfg.sensor_noise_std > 0:
            u = u + cfg.sensor_noise_std * torch.randn_like(u)
        return torch.stack([x[ix], t[it], u], dim=-1)

    @torch.no_grad()
    def build_dataset(self, cfg: CFG, size: int, device: torch.device, tag: str):
        x, t, q = self.make_grids(cfg, device)
        sensors_all, coeff_all, q_all = [], [], []
        num_batches = math.ceil(size / cfg.sim_batch_size)
        print(f"[{self.name}] Building {tag} dataset: {size} samples, {num_batches} simulation batches")
        start = time.time()
        for bi in range(num_batches):
            bs = min(cfg.sim_batch_size, size - bi * cfg.sim_batch_size)
            coeff = self.generate_coeff(cfg, bs, q, device)
            U = self.solve(cfg, coeff, x, t, q)
            sensors = self.sample_sensors(cfg, U, x, t)
            q_grid = q.view(1, -1, 1).repeat(bs, 1, 1)
            sensors_all.append(sensors.cpu())
            coeff_all.append(coeff.unsqueeze(-1).cpu())
            q_all.append(q_grid.cpu())
            if (bi + 1) % max(1, num_batches // 10) == 0 or (bi + 1) == num_batches:
                print(f"  [{self.name}/{tag}] batch {bi + 1}/{num_batches} | elapsed {time.time() - start:.1f}s")
        return InversePDEDataset(torch.cat(sensors_all), torch.cat(q_all), torch.cat(coeff_all))

    def coeff_input(self, x_col: torch.Tensor, t_col: torch.Tensor) -> torch.Tensor:
        return x_col if self.target_axis == "x" else t_col

    def pde_residual(self, model, x_col: torch.Tensor, t_col: torch.Tensor):
        raise NotImplementedError

    def aux_losses(self, model, sample: Dict[str, torch.Tensor], cfg: CFG, device: torch.device):
        return {}


class WaveProblem(BaseProblem):
    name = "wave"
    target_axis = "x"
    coeff_min = 0.75
    coeff_max = 1.45
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    beta_ic = 3

    def initial_condition(self, x):
        return torch.sin(math.pi * x) + 0.5 * torch.sin(self.beta_ic * math.pi * x)

    def smooth(self, field, kernel_size=9):
        pad = kernel_size // 2
        z = field.unsqueeze(1)
        z = F.pad(z, (pad, pad), mode="replicate")
        z = F.avg_pool1d(z, kernel_size=kernel_size, stride=1)
        return z.squeeze(1)

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            mode = random.choice(["sin", "gauss", "piecewise"])
            base = random.uniform(0.95, 1.15)
            xx = q_cpu.clone()
            if mode == "sin":
                c = base
                c = c + random.uniform(0.08, 0.20) * torch.sin(2 * math.pi * xx + random.uniform(0, 2 * math.pi))
                c = c + random.uniform(0.03, 0.10) * torch.sin(4 * math.pi * xx + random.uniform(0, 2 * math.pi))
            elif mode == "gauss":
                c = torch.full_like(xx, base)
                for _ in range(random.randint(1, 3)):
                    amp = random.uniform(-0.18, 0.18)
                    ctr = random.uniform(0.1, 0.9)
                    wid = random.uniform(0.03, 0.12)
                    c = c + amp * torch.exp(-0.5 * ((xx - ctr) / wid) ** 2)
            else:
                n_segments = random.randint(3, 6)
                edges = sorted(random.sample(range(8, cfg.n_q - 8), n_segments - 1))
                edges = [0] + edges + [cfg.n_q]
                c = torch.empty_like(xx)
                cur = base
                for s in range(len(edges) - 1):
                    cur = max(self.coeff_min, min(self.coeff_max, cur + random.uniform(-0.18, 0.18)))
                    c[edges[s]:edges[s + 1]] = cur
            fields.append(c)
        c = torch.stack(fields).to(device)
        c = self.smooth(c)
        return c.clamp(self.coeff_min, self.coeff_max)

    def div_operator(self, u, a_half, dx):
        out = torch.zeros_like(u)
        out[:, 1:-1] = (
            a_half[:, 1:] * (u[:, 2:] - u[:, 1:-1])
            - a_half[:, :-1] * (u[:, 1:-1] - u[:, :-2])
        ) / (dx * dx)
        return out

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        coeff_x = coeff if coeff.shape[1] == len(x) else F.interpolate(coeff.unsqueeze(1), size=len(x), mode="linear", align_corners=True).squeeze(1)
        B, Nx = coeff_x.shape
        Nt = len(t)
        dx = float((self.x_max - self.x_min) / (Nx - 1))
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        if dt > 0.95 * dx / self.coeff_max:
            print("Warning: wave CFL may be high. Increase n_t or reduce coeff_max.")
        a = coeff_x * coeff_x
        a_half = 0.5 * (a[:, :-1] + a[:, 1:])
        u0 = self.initial_condition(x).unsqueeze(0).repeat(B, 1)
        U = torch.zeros(B, Nt, Nx, device=x.device)
        U[:, 0] = u0
        Lu0 = self.div_operator(u0, a_half, dx)
        u1 = u0.clone()
        u1[:, 1:-1] = u0[:, 1:-1] + 0.5 * dt * dt * Lu0[:, 1:-1]
        u1[:, 0] = 0.0
        u1[:, -1] = 0.0
        U[:, 1] = u1
        up, uc = u0, u1
        for n in range(1, Nt - 1):
            Lu = self.div_operator(uc, a_half, dx)
            un = 2 * uc - up + dt * dt * Lu
            un[:, 0] = 0.0
            un[:, -1] = 0.0
            U[:, n + 1] = un
            up, uc = uc, un
        return U

    def pde_residual(self, model, x_col, t_col):
        u = model.u(x_col, t_col)
        c = model.coeff(x_col)
        a = c * c
        u_t = autograd_grad(u, t_col)
        u_tt = autograd_grad(u_t, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        a_x = autograd_grad(a, x_col)
        return u_tt - (a_x * u_x + a * u_xx)

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device).requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        u_ic = model.u(x_ic, t_ic)
        u_true = self.initial_condition(x_ic.squeeze(-1)).unsqueeze(-1)
        u_t_ic = autograd_grad(u_ic, t_ic)
        losses["ic"] = F.mse_loss(u_ic, u_true) + F.mse_loss(u_t_ic, torch.zeros_like(u_t_ic))
        t_bc = torch.rand(cfg.vc_colloc_bc, 1, device=device).requires_grad_(True)
        x0 = torch.zeros_like(t_bc).requires_grad_(True)
        x1 = torch.ones_like(t_bc).requires_grad_(True)
        losses["bc"] = F.mse_loss(model.u(x0, t_bc), torch.zeros_like(t_bc)) + F.mse_loss(model.u(x1, t_bc), torch.zeros_like(t_bc))
        return losses


class ADRProblem(BaseProblem):
    name = "adr"
    target_axis = "x"
    coeff_min = 0.0015
    coeff_max = 0.0060
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    v = 0.4
    lam = 1.0

    def initial_condition(self, x):
        return torch.sin(math.pi * x)

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            base = random.uniform(0.003, 0.0045)
            D = base + random.uniform(0.0004, 0.0012) * torch.sin(2 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            mode = random.choice(["smooth", "bumps", "piecewise"])
            if mode == "bumps":
                for _ in range(random.randint(1, 3)):
                    ctr = random.uniform(0.15, 0.85)
                    wid = random.uniform(0.035, 0.15)
                    amp = random.uniform(-0.0012, 0.0012)
                    D = D + amp * torch.exp(-0.5 * ((q_cpu - ctr) / wid) ** 2)
            elif mode == "piecewise":
                jump = torch.zeros_like(q_cpu)
                ctr = random.uniform(0.25, 0.75)
                jump[q_cpu > ctr] = random.uniform(-0.001, 0.001)
                D = D + jump
            fields.append(D)
        D = torch.stack(fields).to(device)
        D = F.avg_pool1d(F.pad(D.unsqueeze(1), (4, 4), mode="replicate"), kernel_size=9, stride=1).squeeze(1)
        return D.clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        D = coeff if coeff.shape[1] == len(x) else F.interpolate(coeff.unsqueeze(1), size=len(x), mode="linear", align_corners=True).squeeze(1)
        B, Nx = D.shape
        Nt = len(t)
        dx = float((self.x_max - self.x_min) / (Nx - 1))
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        D_half = 0.5 * (D[:, :-1] + D[:, 1:])
        U = torch.zeros(B, Nt, Nx, device=x.device)
        U[:, 0] = self.initial_condition(x).unsqueeze(0).repeat(B, 1)
        for n in range(Nt - 1):
            u = U[:, n]
            diff = torch.zeros_like(u)
            diff[:, 1:-1] = (
                D_half[:, 1:] * (u[:, 2:] - u[:, 1:-1])
                - D_half[:, :-1] * (u[:, 1:-1] - u[:, :-2])
            ) / (dx * dx)
            ux_up = torch.zeros_like(u)
            ux_up[:, 1:] = (u[:, 1:] - u[:, :-1]) / dx
            reaction = self.lam * u * (1.0 - u)
            un = u + dt * (diff - self.v * ux_up + reaction)
            un[:, 0] = 0.0
            un[:, -1] = 0.0
            U[:, n + 1] = un.clamp(-3.0, 3.0)
        return U

    def pde_residual(self, model, x_col, t_col):
        u = model.u(x_col, t_col)
        D = model.coeff(x_col)
        u_t = autograd_grad(u, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        D_x = autograd_grad(D, x_col)
        return u_t - (D_x * u_x + D * u_xx - self.v * u_x + self.lam * u * (1.0 - u))

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device).requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_condition(x_ic.squeeze(-1)).unsqueeze(-1))
        t_bc = torch.rand(cfg.vc_colloc_bc, 1, device=device).requires_grad_(True)
        x0 = torch.zeros_like(t_bc).requires_grad_(True)
        x1 = torch.ones_like(t_bc).requires_grad_(True)
        losses["bc"] = F.mse_loss(model.u(x0, t_bc), torch.zeros_like(t_bc)) + F.mse_loss(model.u(x1, t_bc), torch.zeros_like(t_bc))
        return losses


class VKdVProblem(BaseProblem):
    name = "vkdv"
    target_axis = "t"
    coeff_min = 0.50
    coeff_max = 1.50
    x_min = -1.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    kappa = 0.75
    x0 = -0.55

    def sech2(self, z):
        return 1.0 / torch.cosh(z).pow(2)

    def initial_profile(self, x):
        return 2.0 * self.kappa * self.kappa * self.sech2(self.kappa * (x - self.x0))

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            base = random.uniform(0.85, 1.15)
            g = base
            g = g + random.uniform(0.08, 0.22) * torch.sin(2 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            g = g + random.uniform(0.03, 0.12) * torch.sin(4 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            if random.random() < 0.5:
                g = g * torch.exp(-random.uniform(0.0, 0.4) * q_cpu)
            fields.append(g)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        g_t = coeff if coeff.shape[1] == len(t) else F.interpolate(coeff.unsqueeze(1), size=len(t), mode="linear", align_corners=True).squeeze(1)
        B, Nt = g_t.shape
        Nx = len(x)
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        tau = torch.zeros_like(g_t)
        tau[:, 1:] = torch.cumsum(0.5 * (g_t[:, 1:] + g_t[:, :-1]) * dt, dim=1)
        X = x.view(1, 1, Nx)
        Tau = tau.view(B, Nt, 1)
        center = self.x0 + 4.0 * self.kappa * self.kappa * Tau
        U = 2.0 * self.kappa * self.kappa * self.sech2(self.kappa * (X - center))
        return U

    def pde_residual(self, model, x_col, t_col):
        # u_t + 6 g(t) u u_x + g(t) u_xxx = 0
        u = model.u(x_col, t_col)
        g = model.coeff(t_col)
        u_t = autograd_grad(u, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        u_xxx = autograd_grad(u_xx, x_col)
        return u_t + 6.0 * g * u * u_x + g * u_xxx

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device) * (self.x_max - self.x_min) + self.x_min
        x_ic.requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_profile(x_ic).detach())
        return losses


class VKdVPaperProblem(VKdVProblem):
    name = "vkdv_paper"
    # Same PDE form as VKdVProblem in this code, but coefficients follow fixed families
    # inspired by VC-PINN vKdV experiments: linear, cubic, cosine, decaying oscillation.

    def generate_coeff(self, cfg, batch_size, q, device):
        t_cpu = q.detach().cpu()
        fields = []
        for _ in range(batch_size):
            fam = cfg.coefficient_family
            if fam == "mixed_random":
                fam = random.choice(["linear", "cubic", "cos", "exp_decay"])
            if fam == "linear":
                # scaled linear, with small random slope/offset
                a = random.uniform(0.25, 0.55)
                b = random.uniform(0.75, 1.05)
                g = b + a * t_cpu
            elif fam == "cubic":
                a = random.uniform(0.20, 0.50)
                b = random.uniform(0.80, 1.05)
                g = b + a * (t_cpu ** 3)
            elif fam == "cos":
                base = random.uniform(0.95, 1.10)
                amp = random.uniform(0.15, 0.35)
                phase = random.uniform(0, 2 * math.pi)
                g = base + amp * torch.cos(2 * math.pi * t_cpu + phase)
            elif fam == "exp_decay":
                base = random.uniform(0.90, 1.15)
                amp = random.uniform(0.15, 0.35)
                decay = random.uniform(0.8, 1.8)
                freq = random.uniform(1.0, 2.5)
                phase = random.uniform(0, 2 * math.pi)
                g = base + amp * torch.exp(-decay * t_cpu) * torch.cos(2 * math.pi * freq * t_cpu + phase)
            else:
                raise ValueError(f"Unknown vkdv_paper coefficient_family: {cfg.coefficient_family}")
            fields.append(g)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)


class VSGProblem(BaseProblem):
    name = "vsg"
    target_axis = "t"
    coeff_min = 0.50
    coeff_max = 1.50
    x_min = -1.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    k = 1.0

    def generate_coeff(self, cfg, batch_size, q, device):
        t_cpu = q.detach().cpu()
        fields = []
        for _ in range(batch_size):
            fam = cfg.coefficient_family
            if fam == "mixed_random":
                fam = random.choice(["linear", "quadratic", "cos"])
            if fam == "linear":
                h = random.uniform(0.7, 1.0) + random.uniform(0.2, 0.5) * t_cpu
            elif fam == "quadratic":
                h = random.uniform(0.7, 1.0) + random.uniform(0.2, 0.5) * (t_cpu ** 2)
            elif fam == "cos":
                h = random.uniform(0.95, 1.10) + random.uniform(0.15, 0.35) * torch.cos(2 * math.pi * t_cpu + random.uniform(0, 2 * math.pi))
            else:
                raise ValueError(f"Unknown vsg coefficient_family: {cfg.coefficient_family}")
            fields.append(h)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        h_t = coeff if coeff.shape[1] == len(t) else F.interpolate(coeff.unsqueeze(1), size=len(t), mode="linear", align_corners=True).squeeze(1)
        B, Nt = h_t.shape
        Nx = len(x)
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        omega = torch.zeros_like(h_t)
        omega[:, 1:] = torch.cumsum(0.5 * (h_t[:, 1:] + h_t[:, :-1]) * dt / self.k, dim=1)
        X = x.view(1, 1, Nx)
        Om = omega.view(B, Nt, 1)
        U = 4.0 * torch.atan(torch.exp(self.k * X - Om))
        return U

    def initial_profile(self, x):
        return 4.0 * torch.atan(torch.exp(self.k * x))

    def pde_residual(self, model, x_col, t_col):
        # u_xt + h(t) sin(u) = 0
        u = model.u(x_col, t_col)
        h = model.coeff(t_col)
        u_x = autograd_grad(u, x_col)
        u_xt = autograd_grad(u_x, t_col)
        return u_xt + h * torch.sin(u)

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device) * (self.x_max - self.x_min) + self.x_min
        x_ic.requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_profile(x_ic).detach())
        return losses


def get_problem(name: str) -> BaseProblem:
    if name == "wave":
        return WaveProblem()
    if name == "adr":
        return ADRProblem()
    if name == "vkdv":
        return VKdVProblem()
    if name == "vkdv_paper":
        return VKdVPaperProblem()
    if name == "vsg":
        return VSGProblem()
    raise ValueError(f"Unknown problem_name: {name}")


# ============================================================
# Model components
# ============================================================
class FourierFeatures(nn.Module):
    def __init__(self, in_dim: int, num_bands: int = 8, scale: float = 4.0, enabled: bool = True):
        super().__init__()
        self.in_dim = in_dim
        self.num_bands = num_bands
        self.enabled = enabled
        if enabled:
            freqs = torch.linspace(1.0, num_bands, num_bands) * scale
            self.register_buffer("freqs", freqs)
        else:
            self.register_buffer("freqs", torch.empty(0))

    @property
    def out_dim(self):
        if not self.enabled:
            return self.in_dim
        return self.in_dim + 2 * self.in_dim * self.num_bands

    def forward(self, x):
        if not self.enabled:
            return x
        outs = [x]
        for i in range(self.in_dim):
            xi = x[..., i:i + 1]
            w = self.freqs.view(*([1] * (x.ndim - 1)), -1)
            outs.append(torch.sin(2 * math.pi * xi * w))
            outs.append(torch.cos(2 * math.pi * xi * w))
        return torch.cat(outs, dim=-1)


class FeedForward(nn.Module):
    def __init__(self, d_model, mlp_ratio=4, dropout=0.0):
        super().__init__()
        hidden = d_model * mlp_ratio
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class SensorEncoder(nn.Module):
    def __init__(self, cfg: CFG, u_mean: float, u_std: float):
        super().__init__()
        ff_on = cfg.use_fourier_features
        self.coord_ff = FourierFeatures(2, num_bands=8, scale=4.0, enabled=ff_on)
        self.val_ff = FourierFeatures(1, num_bands=4, scale=4.0, enabled=ff_on)
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        in_dim = self.coord_ff.out_dim + self.val_ff.out_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, sensors):
        coords = sensors[..., :2]
        vals = (sensors[..., 2:3] - self.u_mean) / self.u_std
        z = torch.cat([self.coord_ff(coords), self.val_ff(vals)], dim=-1)
        return self.norm(self.net(z))


class QueryEmbedder(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.ff = FourierFeatures(1, num_bands=16, scale=4.0, enabled=cfg.use_fourier_features)
        self.net = nn.Sequential(
            nn.Linear(self.ff.out_dim, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, q):
        return self.norm(self.net(self.ff(q)))


class CrossAttentionBlock(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.norm_q = nn.LayerNorm(cfg.d_model)
        self.norm_kv = nn.LayerNorm(cfg.d_model)
        self.attn = nn.MultiheadAttention(cfg.d_model, cfg.n_heads, batch_first=True, dropout=cfg.dropout)
        self.norm_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)

    def forward(self, q, kv):
        qn = self.norm_q(q)
        kvn = self.norm_kv(kv)
        attn_out, _ = self.attn(qn, kvn, kvn, need_weights=False)
        x = q + attn_out
        x = x + self.ffn(self.norm_ffn(x))
        return x


class NoCrossConditioning(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )

    def forward(self, h, sensor_tokens):
        ctx = sensor_tokens.mean(dim=1, keepdim=True)
        return h + self.proj(ctx)


class DiagonalSelectiveSSM(nn.Module):
    """
    Lightweight selective diagonal SSM, not full Mamba.
    The update is input-dependent through dt and gate:
        h_k = alpha_k h_{k-1} + (1-alpha_k) B u_k
        y_k = C h_k + D u_k
    """
    def __init__(self, d_model, state_dim, dropout=0.0):
        super().__init__()
        self.d_model = d_model
        self.state_dim = state_dim
        self.in_proj = nn.Linear(d_model, d_model)
        self.gate_proj = nn.Linear(d_model, d_model)
        self.dt_proj = nn.Linear(d_model, d_model * state_dim)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.A_log = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.B = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.C = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.D = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        Bsz, L, Dm = x.shape
        u = self.in_proj(x)
        gate = torch.sigmoid(self.gate_proj(x))
        dt = F.softplus(self.dt_proj(x)).view(Bsz, L, Dm, self.state_dim) + 1e-4
        A = F.softplus(self.A_log).unsqueeze(0).unsqueeze(0)
        alpha = torch.exp(-dt * A)
        Bp = self.B.unsqueeze(0)
        Cp = self.C.unsqueeze(0)
        Dp = self.D.unsqueeze(0)
        h = torch.zeros(Bsz, Dm, self.state_dim, device=x.device, dtype=x.dtype)
        ys = []
        for k in range(L):
            uk = u[:, k, :].unsqueeze(-1)
            h = alpha[:, k] * h + (1.0 - alpha[:, k]) * (uk * Bp)
            yk = (h * Cp).sum(dim=-1) + Dp * u[:, k, :]
            ys.append(yk)
        y = torch.stack(ys, dim=1)
        y = gate * y
        return self.out_proj(self.dropout(y))


class BiSSMBlock(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.norm_f = nn.LayerNorm(cfg.d_model)
        self.norm_b = nn.LayerNorm(cfg.d_model)
        self.fwd = DiagonalSelectiveSSM(cfg.d_model, cfg.state_dim, cfg.dropout)
        self.bwd = DiagonalSelectiveSSM(cfg.d_model, cfg.state_dim, cfg.dropout)
        self.mix = nn.Linear(2 * cfg.d_model, cfg.d_model)
        self.norm_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)

    def forward(self, x):
        xf = self.fwd(self.norm_f(x))
        xb = torch.flip(self.bwd(torch.flip(self.norm_b(x), dims=[1])), dims=[1])
        x = x + self.mix(torch.cat([xf, xb], dim=-1))
        x = x + self.ffn(self.norm_ffn(x))
        return x


class CABiSSMInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.cfg = cfg
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.sensor_encoder = SensorEncoder(cfg, u_mean, u_std)
        self.query_embedder = QueryEmbedder(cfg)
        self.cross_blocks = nn.ModuleList()
        self.nocross_blocks = nn.ModuleList()
        self.ssm_blocks = nn.ModuleList()
        self.ffn_blocks = nn.ModuleList()
        for _ in range(cfg.n_layers):
            self.cross_blocks.append(CrossAttentionBlock(cfg))
            self.nocross_blocks.append(NoCrossConditioning(cfg))
            self.ssm_blocks.append(BiSSMBlock(cfg))
            self.ffn_blocks.append(nn.Sequential(nn.LayerNorm(cfg.d_model), FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)))
        self.norm = nn.LayerNorm(cfg.d_model)
        self.decoder = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(), nn.Linear(cfg.d_model, 1))

    def forward(self, sensors, q_grid):
        sensor_tokens = self.sensor_encoder(sensors)
        h = self.query_embedder(q_grid)
        for i in range(self.cfg.n_layers):
            if self.cfg.use_cross_attention:
                h = self.cross_blocks[i](h, sensor_tokens)
            else:
                h = self.nocross_blocks[i](h, sensor_tokens)
            if self.cfg.use_ssm:
                h = self.ssm_blocks[i](h)
            else:
                h = h + self.ffn_blocks[i](h)
        raw = self.decoder(self.norm(h))
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


class DeepONetInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        self.coord_ff = FourierFeatures(2, 8, 4.0, enabled=cfg.use_fourier_features)
        self.val_ff = FourierFeatures(1, 4, 4.0, enabled=cfg.use_fourier_features)
        sensor_in = self.coord_ff.out_dim + self.val_ff.out_dim
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_in, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_width),
        )
        self.branch = nn.Sequential(
            nn.Linear(2 * cfg.deeponet_width, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_p),
        )
        self.trunk_ff = FourierFeatures(1, 16, 4.0, enabled=cfg.use_fourier_features)
        self.trunk = nn.Sequential(
            nn.Linear(self.trunk_ff.out_dim, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_p),
        )
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, sensors, q_grid):
        vals = (sensors[..., 2:3] - self.u_mean) / self.u_std
        z = torch.cat([self.coord_ff(sensors[..., :2]), self.val_ff(vals)], dim=-1)
        z = self.sensor_mlp(z)
        z = torch.cat([z.mean(dim=1), z.max(dim=1).values], dim=-1)
        b = self.branch(z)
        tr = self.trunk(self.trunk_ff(q_grid))
        raw = (b.unsqueeze(1) * tr).sum(dim=-1, keepdim=True) / math.sqrt(b.shape[-1]) + self.bias
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_t, modes_x):
        super().__init__()
        scale = 1 / max(1, in_channels * out_channels)
        self.modes_t = modes_t
        self.modes_x = modes_x
        self.weights_pos = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes_t, modes_x, dtype=torch.cfloat))
        self.weights_neg = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes_t, modes_x, dtype=torch.cfloat))

    def compl_mul2d(self, x, w):
        return torch.einsum("bixy,ioxy->boxy", x, w)

    def forward(self, x):
        original_dtype = x.dtype
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            B, C, T, X = x.shape
            x_ft = torch.fft.rfft2(x, dim=(-2, -1))
            out_ft = torch.zeros(B, self.weights_pos.shape[1], T, X // 2 + 1, device=x.device, dtype=torch.cfloat)
            mt = min(self.modes_t, T)
            mx = min(self.modes_x, X // 2 + 1)
            out_ft[:, :, :mt, :mx] = self.compl_mul2d(x_ft[:, :, :mt, :mx], self.weights_pos[:, :, :mt, :mx])
            out_ft[:, :, -mt:, :mx] = self.compl_mul2d(x_ft[:, :, -mt:, :mx], self.weights_neg[:, :, :mt, :mx])
            y = torch.fft.irfft2(out_ft, s=(T, X), dim=(-2, -1))
        return y.to(original_dtype)


class FNOBlock(nn.Module):
    def __init__(self, width, modes_t, modes_x):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes_t, modes_x)
        self.point = nn.Conv2d(width, width, 1)
        self.norm = nn.GroupNorm(8, width)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.point(x)))


def sensors_to_grid(cfg: CFG, problem: BaseProblem, sensors, u_mean, u_std):
    device = sensors.device
    dtype = sensors.dtype
    B, Ns, _ = sensors.shape
    T, X = cfg.fno_t_bins, cfg.fno_x_bins
    x = sensors[..., 0]
    t = sensors[..., 1]
    u = (sensors[..., 2] - u_mean.to(device)) / u_std.to(device)
    ix = torch.round((x - problem.x_min) / (problem.x_max - problem.x_min) * (X - 1)).long().clamp(0, X - 1)
    it = torch.round((t - problem.t_min) / (problem.t_max - problem.t_min) * (T - 1)).long().clamp(0, T - 1)
    obs = torch.zeros(B, T, X, device=device, dtype=dtype)
    cnt = torch.zeros(B, T, X, device=device, dtype=dtype)
    b = torch.arange(B, device=device).view(B, 1).expand(B, Ns)
    flat = (b * T * X + it * X + ix).reshape(-1)
    obs.reshape(-1).scatter_add_(0, flat, u.reshape(-1).to(dtype))
    cnt.reshape(-1).scatter_add_(0, flat, torch.ones_like(u, dtype=dtype).reshape(-1))
    mask = (cnt > 0).to(dtype)
    obs = obs / cnt.clamp_min(1.0)
    return torch.stack([obs, mask], dim=1)


class FNOEncoderInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.cfg = cfg
        self.problem = problem
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        self.lift = nn.Conv2d(4, cfg.fno_width, 1)
        self.blocks = nn.ModuleList([FNOBlock(cfg.fno_width, cfg.fno_modes_t, cfg.fno_modes_x) for _ in range(cfg.fno_layers)])
        self.q_ff = FourierFeatures(1, 16, 4.0, enabled=cfg.use_fourier_features)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fno_width + self.q_ff.out_dim, cfg.fno_width), nn.GELU(),
            nn.Linear(cfg.fno_width, cfg.fno_width), nn.GELU(),
            nn.Linear(cfg.fno_width, 1),
        )

    def forward(self, sensors, q_grid):
        B = sensors.shape[0]
        T, X = self.cfg.fno_t_bins, self.cfg.fno_x_bins
        grid = sensors_to_grid(self.cfg, self.problem, sensors, self.u_mean, self.u_std)
        x_coord = torch.linspace(self.problem.x_min, self.problem.x_max, X, device=sensors.device, dtype=sensors.dtype).view(1, 1, 1, X).expand(B, 1, T, X)
        t_coord = torch.linspace(self.problem.t_min, self.problem.t_max, T, device=sensors.device, dtype=sensors.dtype).view(1, 1, T, 1).expand(B, 1, T, X)
        z = torch.cat([grid, x_coord, t_coord], dim=1)
        z = self.lift(z)
        for blk in self.blocks:
            z = blk(z)
        latent = z.mean(dim=(-2, -1))
        latent = latent.unsqueeze(1).expand(B, q_grid.shape[1], latent.shape[-1])
        raw = self.decoder(torch.cat([latent, self.q_ff(q_grid)], dim=-1))
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


# ============================================================
# Training helpers
# ============================================================
def compute_train_stats(ds: InversePDEDataset):
    u = ds.sensors[..., 2]
    c = ds.coeff_true
    g = gradient_1d(c)
    return {
        "u_mean": float(u.mean().item()),
        "u_std": float(u.std().item() + 1e-6),
        "c_mean": float(c.mean().item()),
        "c_std": float(c.std().item() + 1e-6),
        "g_std": float(g.std().item() + 1e-6),
    }


def supervised_loss(c_hat, c_true, stats, cfg):
    loss_field = F.mse_loss((c_hat - c_true) / stats["c_std"], torch.zeros_like(c_hat))
    if cfg.lambda_grad > 0:
        loss_grad = F.mse_loss((gradient_1d(c_hat) - gradient_1d(c_true)) / stats["g_std"], torch.zeros_like(gradient_1d(c_hat)))
    else:
        loss_grad = torch.tensor(0.0, device=c_hat.device)
    loss_tv = total_variation_1d(c_hat)
    return loss_field + cfg.lambda_grad * loss_grad + cfg.lambda_tv * loss_tv, {
        "loss_field": float(loss_field.detach().item()),
        "loss_grad": float(loss_grad.detach().item()),
        "loss_tv": float(loss_tv.detach().item()),
    }


def build_model(model_key: str, cfg: CFG, problem: BaseProblem, stats: Dict[str, float]) -> nn.Module:
    if model_key == "cabissm":
        return CABiSSMInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    if model_key == "deeponet":
        return DeepONetInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    if model_key == "fnoenc":
        return FNOEncoderInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    raise ValueError(model_key)


def nice_name(model_key: str) -> str:
    return {"cabissm": "CABiSSM", "deeponet": "DeepONet", "fnoenc": "FNOEnc"}.get(model_key, model_key)


def train_amortized_model(cfg, problem, model, model_name, train_ds, val_ds, stats, device, out_dir, epochs):
    train_loader = make_loader(train_ds, cfg, True)
    val_loader = make_loader(val_ds, cfg, False)
    model = model.to(device)
    if cfg.use_compile and hasattr(torch, "compile"):
        model = torch.compile(model)
    print(f"[{problem.name}] {model_name} parameters: {count_params(model):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    amp_dtype = get_amp_dtype(cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.use_amp and device.type == "cuda" and amp_dtype == torch.float16))
    history = {"train_loss": [], "val_loss": [], "val_mae": [], "val_rel_l2": [], "val_grad_mae": [], "val_norm_mae": [], "val_var_ratio": []}
    best_val = float("inf")
    best_metrics = None
    best_epoch = -1
    best_state = None
    no_improve = 0
    best_path = out_dir / f"{problem.name}_{model_name.lower()}_best.pt"
    start_train = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        train_total, train_count = 0.0, 0
        for batch in train_loader:
            batch = move_batch(batch, device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(cfg.use_amp and device.type == "cuda")):
                pred = model(batch["sensors"], batch["q_grid"])
                loss, _ = supervised_loss(pred, batch["coeff_true"], stats, cfg)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                opt.step()
            bs = batch["sensors"].shape[0]
            train_total += float(loss.detach()) * bs
            train_count += bs
        sched.step()
        model.eval()
        val_total, val_count = 0.0, 0
        metric_total: Dict[str, float] = {}
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch(batch, device)
                with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(cfg.use_amp and device.type == "cuda")):
                    pred = model(batch["sensors"], batch["q_grid"])
                    loss, _ = supervised_loss(pred, batch["coeff_true"], stats, cfg)
                bs = batch["sensors"].shape[0]
                val_total += float(loss.detach()) * bs
                val_count += bs
                m = coeff_metrics(pred.float(), batch["coeff_true"].float())
                for k, v in m.items():
                    metric_total[k] = metric_total.get(k, 0.0) + v * bs
        train_loss = train_total / max(train_count, 1)
        val_loss = val_total / max(val_count, 1)
        metrics = {k: v / max(val_count, 1) for k, v in metric_total.items()}
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(metrics["mae"])
        history["val_rel_l2"].append(metrics["rel_l2"])
        history["val_grad_mae"].append(metrics["grad_mae"])
        history["val_norm_mae"].append(metrics["norm_mae"])
        history["val_var_ratio"].append(metrics["var_ratio"])
        print(
            f"[{problem.name}] {model_name} ep {ep:03d} | train {train_loss:.3e} | val {val_loss:.3e} | "
            f"mae {metrics['mae']:.3e} | relL2 {metrics['rel_l2']:.3e} | grad {metrics['grad_mae']:.3e} | "
            f"normMAE {metrics['norm_mae']:.3e} | varRatio {metrics['var_ratio']:.3f}"
        )
        if val_loss < best_val:
            best_val = val_loss
            best_metrics = metrics
            best_epoch = ep
            no_improve = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if cfg.save_checkpoints:
                torch.save({"model": best_state, "metrics": metrics, "best_val": best_val, "epoch": ep, "history": history, "stats": stats}, best_path)
        else:
            no_improve += 1
        if model_name.startswith("CABiSSM") and no_improve >= cfg.early_stop_patience:
            print(f"[{problem.name}] Early stopping {model_name} at epoch {ep}; best epoch {best_epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    train_time = time.time() - start_train
    return model, history, best_metrics or {}, best_val, best_epoch, train_time


@torch.no_grad()
def evaluate_subset(model, ds, indices, device):
    model.eval()
    metrics, preds = [], []
    for idx in indices:
        s = ds[idx]
        sensors = s["sensors"].unsqueeze(0).to(device)
        q = s["q_grid"].unsqueeze(0).to(device)
        true = s["coeff_true"].unsqueeze(0).to(device)
        pred = model(sensors, q)
        metrics.append(coeff_metrics(pred.float(), true.float()))
        preds.append(pred.squeeze(0).cpu())
    return metrics, preds


@torch.no_grad()
def benchmark_inference(model, ds, cfg, device, n_batches=10):
    loader = make_loader(ds, cfg, False)
    model.eval()
    times = []
    count = 0
    # warmup
    for i, batch in enumerate(loader):
        batch = move_batch(batch, device)
        _ = model(batch["sensors"], batch["q_grid"])
        if i >= 2:
            break
    if device.type == "cuda":
        torch.cuda.synchronize()
    for i, batch in enumerate(loader):
        if i >= n_batches:
            break
        batch = move_batch(batch, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        _ = model(batch["sensors"], batch["q_grid"])
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - start
        times.append(elapsed)
        count += batch["sensors"].shape[0]
    total = sum(times)
    return {"inference_time_sec_total": total, "inference_samples": count, "inference_ms_per_sample": 1000.0 * total / max(count, 1)}


# ============================================================
# VC-PINN-style baseline
# ============================================================
def autograd_grad(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True, retain_graph=True)[0]


class PreActResidualBlock(nn.Module):
    def __init__(self, width, layers_per_block=2):
        super().__init__()
        layers = []
        for _ in range(layers_per_block):
            layers += [nn.Tanh(), nn.Linear(width, width)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.net(x)


class ResNetMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, blocks, layers_per_block):
        super().__init__()
        self.input = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([PreActResidualBlock(hidden, layers_per_block) for _ in range(blocks)])
        self.output = nn.Linear(hidden, out_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.input(x)
        for block in self.blocks:
            h = block(h)
        return self.output(h)


class VCPINNModel(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem):
        super().__init__()
        self.problem = problem
        self.u_net = ResNetMLP(2, cfg.vc_hidden, 1, cfg.vc_blocks_u, cfg.vc_layers_per_block)
        self.c_net = ResNetMLP(1, cfg.vc_hidden, 1, cfg.vc_blocks_c, cfg.vc_layers_per_block)

    def u(self, x, t):
        return self.u_net(torch.cat([x, t], dim=-1))

    def coeff(self, q):
        raw = self.c_net(q)
        return self.problem.coeff_min + (self.problem.coeff_max - self.problem.coeff_min) * torch.sigmoid(raw)


def vc_pinn_loss(problem, model, sample, cfg, device, colloc):
    sensors = sample["sensors"].to(device).float()
    q_grid = sample["q_grid"].to(device).float()
    coeff_true = sample["coeff_true"].to(device).float()
    x_s = sensors[:, 0:1]
    t_s = sensors[:, 1:2]
    u_s = sensors[:, 2:3]
    loss_data = F.mse_loss(model.u(x_s, t_s), u_s)
    x_f = colloc["x_f"].clone().detach().requires_grad_(True)
    t_f = colloc["t_f"].clone().detach().requires_grad_(True)
    loss_pde = torch.mean(problem.pde_residual(model, x_f, t_f) ** 2)
    aux = problem.aux_losses(model, sample, cfg, device)
    loss_ic = aux.get("ic", torch.tensor(0.0, device=device))
    loss_bc = aux.get("bc", torch.tensor(0.0, device=device))
    c0_pred = model.coeff(q_grid[0:1])
    c1_pred = model.coeff(q_grid[-1:])
    loss_cbc = F.mse_loss(c0_pred, coeff_true[0:1]) + F.mse_loss(c1_pred, coeff_true[-1:])
    c_grid = model.coeff(q_grid)
    loss_smooth = total_variation_1d(c_grid.unsqueeze(0))
    loss = (
        cfg.vc_w_data * loss_data + cfg.vc_w_pde * loss_pde + cfg.vc_w_ic * loss_ic +
        cfg.vc_w_bc * loss_bc + cfg.vc_w_cbc * loss_cbc + cfg.vc_w_c_smooth * loss_smooth
    )
    parts = {"data": float(loss_data.detach()), "pde": float(loss_pde.detach()), "ic": float(loss_ic.detach()), "bc": float(loss_bc.detach()), "cbc": float(loss_cbc.detach()), "smooth": float(loss_smooth.detach())}
    return loss, parts


@torch.no_grad()
def eval_vc_coeff(model, sample, device):
    q_grid = sample["q_grid"].to(device).float()
    coeff_true = sample["coeff_true"].to(device).float()
    pred = model.coeff(q_grid)
    return pred.cpu(), coeff_metrics(pred.unsqueeze(0), coeff_true.unsqueeze(0))


def run_vc_pinn_sample(problem, cfg, sample, device):
    model = VCPINNModel(cfg, problem).to(device).float()
    opt = torch.optim.Adam(model.parameters(), lr=cfg.vc_lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.vc_adam_steps)
    colloc = {
        "x_f": torch.rand(cfg.vc_colloc_f, 1, device=device) * (problem.x_max - problem.x_min) + problem.x_min,
        "t_f": torch.rand(cfg.vc_colloc_f, 1, device=device) * (problem.t_max - problem.t_min) + problem.t_min,
    }
    best_state, best_loss = None, float("inf")
    hist = []
    for step in range(1, cfg.vc_adam_steps + 1):
        opt.zero_grad(set_to_none=True)
        loss, parts = vc_pinn_loss(problem, model, sample, cfg, device, colloc)
        loss.backward()
        opt.step()
        sched.step()
        lv = float(loss.detach())
        hist.append(lv)
        if lv < best_loss:
            best_loss = lv
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if step == 1 or step % cfg.vc_log_every == 0 or step == cfg.vc_adam_steps:
            with torch.no_grad():
                _, m = eval_vc_coeff(model, sample, device)
            print(f"    [{problem.name}] VC-PINN {step:04d}/{cfg.vc_adam_steps} | loss {lv:.3e} | data {parts['data']:.2e} | pde {parts['pde']:.2e} | mae {m['mae']:.3e} | relL2 {m['rel_l2']:.3e}")
    if best_state is not None:
        model.load_state_dict(best_state)
    if cfg.vc_lbfgs_steps > 0:
        print(f"    [{problem.name}] VC-PINN L-BFGS refinement: {cfg.vc_lbfgs_steps} iters")
        lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=cfg.vc_lbfgs_steps, max_eval=2 * cfg.vc_lbfgs_steps, history_size=50, line_search_fn="strong_wolfe")
        def closure():
            lbfgs.zero_grad(set_to_none=True)
            loss, _ = vc_pinn_loss(problem, model, sample, cfg, device, colloc)
            loss.backward()
            return loss
        lbfgs.step(closure)
    pred, metrics = eval_vc_coeff(model, sample, device)
    return pred, metrics, hist


def run_vc_pinn_baseline(problem, cfg, val_ds, device, indices):
    metrics, preds, histories = [], [], []
    print(f"[{problem.name}] Running VC-PINN-style on {len(indices)} samples")
    total = time.time()
    for i, idx in enumerate(indices):
        print(f"  [{problem.name}] VC sample {i + 1}/{len(indices)} index={idx}")
        start = time.time()
        pred, m, hist = run_vc_pinn_sample(problem, cfg, val_ds[idx], device)
        m["wall_time_sec"] = time.time() - start
        metrics.append(m)
        preds.append(pred)
        histories.append(hist)
        print(f"  [{problem.name}] done idx={idx} | time {m['wall_time_sec']:.1f}s | MAE {m['mae']:.3e} | relL2 {m['rel_l2']:.3e}")
    print(f"[{problem.name}] VC-PINN total time: {time.time() - total:.1f}s")
    return metrics, preds, histories


# ============================================================
# Plotting and artifact saving
# ============================================================
def save_curves(history, out_dir, problem_name, method_name):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="train loss")
    plt.plot(epochs, history["val_loss"], label="val loss")
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"{problem_name}: {method_name} loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{problem_name}_{method_name}_loss.png", dpi=200)
    plt.close()
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_mae"], label="MAE")
    plt.plot(epochs, history["val_rel_l2"], label="relL2")
    plt.plot(epochs, history["val_grad_mae"], label="gradMAE")
    plt.plot(epochs, history["val_norm_mae"], label="normMAE")
    plt.xlabel("epoch")
    plt.ylabel("metric")
    plt.title(f"{problem_name}: {method_name} validation metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{problem_name}_{method_name}_metrics.png", dpi=200)
    plt.close()


def save_reconstruction_plot(problem, val_ds, indices, method_preds, out_dir, tag=""):
    if not method_preds:
        return
    n = len(indices)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))
    if n == 1:
        axes = axes[None, :]
    for row, idx in enumerate(indices):
        s = val_ds[idx]
        q = s["q_grid"].squeeze(-1)
        true = s["coeff_true"].squeeze(-1)
        sensors = s["sensors"]
        ax0 = axes[row, 0]
        sc = ax0.scatter(sensors[:, 0], sensors[:, 1], c=sensors[:, 2], s=16)
        ax0.set_xlabel("x")
        ax0.set_ylabel("t")
        ax0.set_title(f"{problem.name}: sparse sensors sample {idx}")
        plt.colorbar(sc, ax=ax0, fraction=0.046, pad=0.04)
        ax1 = axes[row, 1]
        ax1.plot(q, true, label="true coefficient", linewidth=2)
        for name, preds in method_preds.items():
            ax1.plot(q, preds[row].squeeze(-1), label=name, linewidth=2)
        ax1.set_xlabel(problem.target_axis)
        ax1.set_ylabel("coefficient")
        ax1.set_title(f"{problem.name}: coefficient reconstruction")
        ax1.legend(fontsize=8)
    plt.tight_layout()
    suffix = f"_{tag}" if tag else ""
    plt.savefig(out_dir / f"{problem.name}{suffix}_reconstructions.png", dpi=200)
    plt.close()


def save_barplot(problem_name, method_metrics, out_dir, tag=""):
    if not method_metrics:
        return
    keys = ["mae", "rel_l2", "grad_mae", "norm_mae", "var_ratio"]
    methods = list(method_metrics.keys())
    x = list(range(len(keys)))
    width = 0.8 / max(len(methods), 1)
    plt.figure(figsize=(12, 5))
    for i, method in enumerate(methods):
        vals = [mean_metric(method_metrics[method], k) for k in keys]
        offset = (i - (len(methods) - 1) / 2) * width
        plt.bar([j + offset for j in x], vals, width=width, label=method)
    plt.xticks(x, keys)
    plt.ylabel("metric value")
    plt.title(f"{problem_name}: comparison metrics")
    plt.legend()
    plt.tight_layout()
    suffix = f"_{tag}" if tag else ""
    plt.savefig(out_dir / f"{problem_name}{suffix}_barplot.png", dpi=200)
    plt.close()


def save_table_csv(rows: List[Dict[str, Any]], path: Path):
    if not rows:
        return
    keys = sorted({k for r in rows for k in r.keys()})
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def save_summary_tables(summary: Dict, out_dir: Path):
    full_rows = []
    for method, metrics in summary.get("full_val_metrics", {}).items():
        row = {"method": method}
        row.update(metrics)
        full_rows.append(row)
    save_table_csv(full_rows, out_dir / "full_val_metrics.csv")
    subset_rows = []
    for method, metrics in summary.get("subset_avg", {}).items():
        row = {"method": method}
        row.update(metrics)
        subset_rows.append(row)
    save_table_csv(subset_rows, out_dir / "subset_avg_metrics.csv")
    runtime_rows = []
    for method, metrics in summary.get("runtime", {}).items():
        row = {"method": method}
        row.update(metrics)
        runtime_rows.append(row)
    save_table_csv(runtime_rows, out_dir / "runtime_metrics.csv")


def add_text_page(pdf, title: str, lines: List[str], max_lines=42):
    for start in range(0, max(len(lines), 1), max_lines):
        chunk = lines[start:start + max_lines]
        fig = plt.figure(figsize=(11, 8.5))
        ax = fig.add_subplot(111)
        ax.axis("off")
        ax.text(0.02, 0.97, title if start == 0 else title + " (continued)", fontsize=16, fontweight="bold", va="top", family="monospace")
        y = 0.91
        for line in chunk:
            ax.text(0.02, y, str(line)[:150], fontsize=9, va="top", family="monospace")
            y -= 0.021
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)


def add_image_page(pdf, image_path: Path, title: str):
    if not image_path.exists():
        return
    try:
        img = plt.imread(str(image_path))
    except Exception:
        return
    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_subplot(111)
    ax.imshow(img)
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def save_pdf_report(summary: Dict, out_dir: Path):
    pdf_path = out_dir / "summary_report.pdf"
    with PdfPages(pdf_path) as pdf:
        lines = [
            f"Problem: {summary.get('problem')}",
            f"Run mode: {summary.get('run_mode')}",
            f"Coefficient family: {summary.get('coefficient_family')}",
            f"Target axis: {summary.get('target_axis')}",
            f"Coefficient range: {summary.get('coefficient_range')}",
            "",
            "Train stats:",
        ]
        for k, v in summary.get("train_stats", {}).items():
            lines.append(f"  {k}: {v}")
        add_text_page(pdf, "Experiment overview", lines)

        if "modes" in summary:
            mode_lines = []
            for mode_name, mode_summary in summary.get("modes", {}).items():
                mode_lines.append("=" * 80)
                mode_lines.append(f"MODE: {mode_name}")
                mode_lines.append("=" * 80)
                if "full_val_metrics" in mode_summary:
                    mode_lines.append("Full validation metrics:")
                    for method, metrics in mode_summary.get("full_val_metrics", {}).items():
                        mae = metrics.get("mae", None)
                        rel = metrics.get("rel_l2", None)
                        grad = metrics.get("grad_mae", None)
                        varr = metrics.get("var_ratio", None)
                        mode_lines.append(f"  {method}: MAE={mae}, relL2={rel}, gradMAE={grad}, varRatio={varr}")
                if "subset_avg" in mode_summary:
                    mode_lines.append("Subset averages:")
                    for method, metrics in mode_summary.get("subset_avg", {}).items():
                        mae = metrics.get("mae", None)
                        rel = metrics.get("rel_l2", None)
                        grad = metrics.get("grad_mae", None)
                        varr = metrics.get("var_ratio", None)
                        mode_lines.append(f"  {method}: MAE={mae}, relL2={rel}, gradMAE={grad}, varRatio={varr}")
                mode_lines.append("")
            add_text_page(pdf, "All-modes combined summary", mode_lines)

        full_lines = []
        for method, metrics in summary.get("full_val_metrics", {}).items():
            full_lines.append(method)
            for k, v in metrics.items():
                full_lines.append(f"  {k:<18}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            full_lines.append("")
        add_text_page(pdf, "Full validation metrics", full_lines)

        subset_lines = []
        for method, metrics in summary.get("subset_avg", {}).items():
            subset_lines.append(method)
            for k, v in metrics.items():
                subset_lines.append(f"  {k:<18}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            subset_lines.append("")
        add_text_page(pdf, "Subset metrics", subset_lines)

        runtime_lines = []
        for method, metrics in summary.get("runtime", {}).items():
            runtime_lines.append(method)
            for k, v in metrics.items():
                runtime_lines.append(f"  {k:<25}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            runtime_lines.append("")
        add_text_page(pdf, "Runtime metrics", runtime_lines)

        for png in sorted(out_dir.rglob("*.png")):
            add_image_page(pdf, png, str(png.relative_to(out_dir)))
    return pdf_path


def make_results_zip(out_dir: Path):
    zip_path = out_dir.parent / f"{out_dir.name}.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file in out_dir.rglob("*"):
            if file.is_file() and file != zip_path:
                zf.write(file, arcname=file.relative_to(out_dir.parent))
    return zip_path


# ============================================================
# Experiment runners
# ============================================================
def run_main_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path, tag: str = "main") -> Dict:
    ensure_dir(out_dir)
    train_ds = problem.build_dataset(cfg, cfg.train_size, device, tag=f"train_{tag}")
    val_ds = problem.build_dataset(cfg, cfg.val_size, device, tag=f"val_{tag}")
    stats = compute_train_stats(train_ds)
    print(f"[{problem.name}] train stats: {stats}")

    full_val_metrics: Dict[str, Dict] = {}
    subset_metrics: Dict[str, List[Dict]] = {}
    subset_preds: Dict[str, List[torch.Tensor]] = {}
    histories: Dict[str, Any] = {}
    runtime: Dict[str, Dict] = {}
    best_epochs: Dict[str, int] = {}

    subset_indices = list(range(min(cfg.vc_eval_samples, len(val_ds))))

    for model_key in cfg.run_methods:
        if model_key == "vc_pinn":
            continue
        model_name = nice_name(model_key)
        model = build_model(model_key, cfg, problem, stats)
        model, hist, metrics, best_val, best_epoch, train_time = train_amortized_model(cfg, problem, model, model_name, train_ds, val_ds, stats, device, out_dir, cfg.epochs if model_key == "cabissm" else cfg.baseline_epochs)
        histories[model_name] = hist
        full_val_metrics[model_name] = metrics
        best_epochs[model_name] = best_epoch
        runtime[model_name] = {"train_time_sec": train_time, "train_time_min": train_time / 60.0}
        runtime[model_name].update(benchmark_inference(model, val_ds, cfg, device, n_batches=10))
        save_curves(hist, out_dir, problem.name, model_name)
        m, p = evaluate_subset(model, val_ds, subset_indices, device)
        subset_metrics[model_name] = m
        subset_preds[model_name] = p

    if "vc_pinn" in cfg.run_methods:
        m, p, h = run_vc_pinn_baseline(problem, cfg, val_ds, device, subset_indices)
        subset_metrics["VC-PINN-style"] = m
        subset_preds["VC-PINN-style"] = p
        histories["VC-PINN-style"] = h
        runtime["VC-PINN-style"] = {
            "subset_total_time_sec": sum(x.get("wall_time_sec", 0.0) for x in m),
            "mean_time_per_sample_sec": mean_metric(m, "wall_time_sec"),
        }

    save_reconstruction_plot(problem, val_ds, subset_indices, subset_preds, out_dir, tag=tag)
    save_barplot(problem.name, subset_metrics, out_dir, tag=tag)

    summary = {
        "problem": problem.name,
        "run_mode": cfg.run_mode,
        "tag": tag,
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "train_stats": stats,
        "best_epochs": best_epochs,
        "full_val_metrics": full_val_metrics,
        "subset_avg": {method: {k: mean_metric(metrics, k) for k in metrics[0].keys()} for method, metrics in subset_metrics.items()},
        "runtime": runtime,
    }
    save_json(summary, out_dir / f"{tag}_summary.json")
    save_summary_tables(summary, out_dir)
    return summary


def ablation_cfg(base: CFG, name: str) -> CFG:
    cfg = replace(base)
    cfg.run_methods = ("cabissm",)
    cfg.epochs = base.epochs
    if name == "full_cabissm":
        pass
    elif name == "no_ssm_attention_only":
        cfg.use_ssm = False
    elif name == "no_cross_ssm_only":
        cfg.use_cross_attention = False
    elif name == "no_fourier_features":
        cfg.use_fourier_features = False
    elif name == "no_gradient_loss":
        cfg.lambda_grad = 0.0
    else:
        raise ValueError(name)
    return cfg


def run_ablation_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path) -> Dict:
    ensure_dir(out_dir)
    # Use the same dataset across ablations for fairness.
    train_ds = problem.build_dataset(cfg, cfg.train_size, device, tag="train_ablation_shared")
    val_ds = problem.build_dataset(cfg, cfg.val_size, device, tag="val_ablation_shared")
    base_stats = compute_train_stats(train_ds)
    subset_indices = list(range(min(cfg.vc_eval_samples, len(val_ds))))
    all_metrics, all_runtime, all_histories, subset_metrics, subset_preds = {}, {}, {}, {}, {}
    for abl in cfg.ablation_names:
        print("\n" + "=" * 100)
        print(f"Ablation: {abl}")
        print("=" * 100)
        acfg = ablation_cfg(cfg, abl)
        model_name = f"CABiSSM_{abl}"
        model = CABiSSMInverse(acfg, problem, base_stats["u_mean"], base_stats["u_std"])
        model, hist, metrics, best_val, best_epoch, train_time = train_amortized_model(acfg, problem, model, model_name, train_ds, val_ds, base_stats, device, out_dir, acfg.epochs)
        all_metrics[model_name] = metrics
        all_runtime[model_name] = {"train_time_sec": train_time, "train_time_min": train_time / 60.0}
        all_runtime[model_name].update(benchmark_inference(model, val_ds, acfg, device, n_batches=10))
        all_histories[model_name] = hist
        save_curves(hist, out_dir, problem.name, model_name)
        m, p = evaluate_subset(model, val_ds, subset_indices, device)
        subset_metrics[model_name] = m
        subset_preds[model_name] = p
    save_reconstruction_plot(problem, val_ds, subset_indices, subset_preds, out_dir, tag="ablation")
    save_barplot(problem.name, subset_metrics, out_dir, tag="ablation")
    summary = {
        "problem": problem.name,
        "run_mode": "ablation",
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "train_stats": base_stats,
        "full_val_metrics": all_metrics,
        "subset_avg": {method: {k: mean_metric(metrics, k) for k in metrics[0].keys()} for method, metrics in subset_metrics.items()},
        "runtime": all_runtime,
    }
    save_json(summary, out_dir / "ablation_summary.json")
    save_summary_tables(summary, out_dir)
    return summary


def run_sweep_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path, sweep_type: str) -> Dict:
    ensure_dir(out_dir)
    values = cfg.sensor_sweep_values if sweep_type == "sensor" else cfg.noise_sweep_values
    rows = []
    all_summaries = []
    for value in values:
        if sweep_type == "sensor":
            scfg = replace(cfg, n_sensors=int(value), train_size=cfg.sweep_train_size, val_size=cfg.sweep_val_size, epochs=cfg.sweep_epochs, baseline_epochs=cfg.sweep_epochs, run_methods=("cabissm", "deeponet", "fnoenc"))
            tag = f"sensor_{int(value)}"
        else:
            scfg = replace(cfg, sensor_noise_std=float(value), train_size=cfg.sweep_train_size, val_size=cfg.sweep_val_size, epochs=cfg.sweep_epochs, baseline_epochs=cfg.sweep_epochs, run_methods=("cabissm", "deeponet", "fnoenc"))
            tag = f"noise_{float(value):.3f}".replace(".", "p")
        print("\n" + "=" * 100)
        print(f"Sweep run: {tag}")
        print("=" * 100)
        run_dir = out_dir / tag
        summary = run_main_experiment(scfg, problem, device, run_dir, tag=tag)
        all_summaries.append(summary)
        for method, metrics in summary.get("full_val_metrics", {}).items():
            row = {"sweep_type": sweep_type, "sweep_value": value, "method": method}
            row.update(metrics)
            rows.append(row)
    save_table_csv(rows, out_dir / f"{sweep_type}_sweep_metrics.csv")
    # plot MAE vs sweep value
    methods = sorted(set(r["method"] for r in rows))
    plt.figure(figsize=(8, 5))
    for method in methods:
        xs = [r["sweep_value"] for r in rows if r["method"] == method]
        ys = [r["mae"] for r in rows if r["method"] == method]
        plt.plot(xs, ys, marker="o", label=method)
    plt.xlabel("number of sensors" if sweep_type == "sensor" else "sensor noise std")
    plt.ylabel("MAE")
    plt.title(f"{problem.name}: {sweep_type} sweep")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{sweep_type}_sweep_mae.png", dpi=200)
    plt.close()
    summary = {
        "problem": problem.name,
        "run_mode": f"{sweep_type}_sweep",
        "coefficient_family": cfg.coefficient_family,
        "config": asdict(cfg),
        "rows": rows,
        "subruns": all_summaries,
    }
    save_json(summary, out_dir / f"{sweep_type}_sweep_summary.json")
    return summary


def run_all_modes_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path) -> Dict:
    """
    Runs all journal-level modes sequentially for one selected equation:
      1) main: CABiSSM + DeepONet + FNOEnc + VC-PINN-style
      2) ablation: architecture/loss ablations for CABiSSM
      3) sensor_sweep: sparse-sensor robustness
      4) noise_sweep: noisy-observation robustness
    Each mode gets its own subfolder. A combined final_summary.json, PDF, and ZIP are saved at the parent folder.
    """
    ensure_dir(out_dir)
    modes_to_run = ["main", "ablation", "sensor_sweep", "noise_sweep"]
    mode_summaries: Dict[str, Dict] = {}
    for mode in modes_to_run:
        print("\n" + "#" * 120)
        print(f"RUNNING MODE: {mode.upper()} for problem {problem.name}")
        print("#" * 120)
        mcfg = replace(cfg, run_mode=mode)
        mode_dir = out_dir / mode
        if mode == "main":
            mode_summaries[mode] = run_main_experiment(mcfg, problem, device, mode_dir, tag="main")
        elif mode == "ablation":
            mode_summaries[mode] = run_ablation_experiment(mcfg, problem, device, mode_dir)
        elif mode == "sensor_sweep":
            mode_summaries[mode] = run_sweep_experiment(mcfg, problem, device, mode_dir, sweep_type="sensor")
        elif mode == "noise_sweep":
            mode_summaries[mode] = run_sweep_experiment(mcfg, problem, device, mode_dir, sweep_type="noise")
        else:
            raise ValueError(mode)
    combined = {
        "problem": problem.name,
        "run_mode": "all_modes",
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "modes": mode_summaries,
    }
    save_json(combined, out_dir / "all_modes_summary.json")
    return combined


# ============================================================
# Main
# ============================================================
def main():
    cfg = CFG()
    set_seed(cfg.seed)
    torch.set_float32_matmul_precision("high")
    device = get_device()
    print("Device:", device)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    problem = get_problem(cfg.problem_name)
    root_dir = Path(cfg.out_root)
    family_tag = cfg.coefficient_family if problem.name in ["vkdv_paper", "vsg"] else "default"
    out_dir = root_dir / f"{problem.name}_{family_tag}_{cfg.run_mode}"
    ensure_dir(out_dir)
    print("=" * 100)
    print(f"Running problem       : {problem.name}")
    print(f"Run mode              : {cfg.run_mode}")
    print(f"Coefficient family    : {family_tag}")
    print(f"Target axis           : {problem.target_axis}")
    print(f"Coefficient range     : {problem.coeff_min} to {problem.coeff_max}")
    print(f"Output folder         : {out_dir.resolve()}")
    print("=" * 100)
    start_all = time.time()
    if cfg.run_mode == "main":
        summary = run_main_experiment(cfg, problem, device, out_dir, tag="main")
    elif cfg.run_mode == "ablation":
        summary = run_ablation_experiment(cfg, problem, device, out_dir)
    elif cfg.run_mode == "sensor_sweep":
        summary = run_sweep_experiment(cfg, problem, device, out_dir, sweep_type="sensor")
    elif cfg.run_mode == "noise_sweep":
        summary = run_sweep_experiment(cfg, problem, device, out_dir, sweep_type="noise")
    elif cfg.run_mode == "all_modes":
        summary = run_all_modes_experiment(cfg, problem, device, out_dir)
    else:
        raise ValueError(f"Unknown run_mode: {cfg.run_mode}")
    summary["total_wall_time_sec"] = time.time() - start_all
    save_json(summary, out_dir / "final_summary.json")
    pdf_path = None
    zip_path = None
    if cfg.make_pdf_summary:
        pdf_path = save_pdf_report(summary, out_dir)
        print(f"PDF report saved: {pdf_path}")
    if cfg.make_zip:
        zip_path = make_results_zip(out_dir)
        print(f"ZIP saved: {zip_path}")
    print("\nExperiment completed.")
    print("Artifacts saved in:", out_dir.resolve())
    try:
        from IPython.display import FileLink, display
        if pdf_path is not None:
            display(FileLink(str(pdf_path)))
        if zip_path is not None:
            display(FileLink(str(zip_path)))
    except Exception:
        pass


if __name__ == "__main__":
    main()


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Running problem       : vkdv
Run mode              : main
Coefficient family    : default
Target axis           : t
Coefficient range     : 0.5 to 1.5
Output folder         : /kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/vkdv_default_main
[vkdv] Building train_main dataset: 16384 samples, 128 simulation batches
  [vkdv/train_main] batch 12/128 | elapsed 0.1s
  [vkdv/train_main] batch 24/128 | elapsed 0.2s
  [vkdv/train_main] batch 36/128 | elapsed 0.2s
  [vkdv/train_main] batch 48/128 | elapsed 0.3s
  [vkdv/train_main] batch 60/128 | elapsed 0.3s
  [vkdv/train_main] batch 72/128 | elapsed 0.4s
  [vkdv/train_main] batch 84/128 | elapsed 0.4s
  [vkdv/train_main] batch 96/128 | elapsed 0.5s
  [vkdv/train_main] batch 108/128 | elapsed 0.6s
  [vkdv/train_main] batch 120/128 | elapsed 0.6s
  [vkdv/train_main] batch 128/128 | elapsed 0.7s
[vkdv] Building val_main dataset: 2048 samples, 16 simulation batches
  [vkdv/val_

/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/vkdv_default_main/summary_report.pdf

/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/vkdv_default_main.zip

In [3]:
# ============================================================
# Publishable inverse-PDE pipeline
# CABiSSM vs DeepONet vs FNOEnc vs VC-PINN-style
# Includes: equation selection, baselines, ablations, robustness sweeps,
# runtime analysis, normalized metrics, variance-ratio metrics, PDF + ZIP export.
#
# Recommended Kaggle use:
#   1) Paste this full script into one Kaggle notebook cell.
#   2) Edit ONLY the USER SETTINGS block.
#   3) Run one problem / one experiment mode at a time.
# ============================================================

import csv
import json
import math
import random
import time
import zipfile
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# USER SETTINGS: change this block only
# ============================================================
@dataclass
class CFG:
    # ------------------------------
    # Main experiment selection
    # ------------------------------
    # Available problems:
    #   "wave"         : variable wave speed c(x)
    #   "adr"          : variable diffusion D(x)
    #   "vkdv"         : random time-coefficient KdV-style g(t)
    #   "vkdv_paper"   : VC-PINN-paper-inspired vKdV coefficient families
    #   "vsg"          : variable-coefficient Sine-Gordon h(t)
    problem_name: str = "vsg"

    # Run modes:
    #   "main"          : main model + baselines
    #   "ablation"      : CABiSSM ablation study
    #   "sensor_sweep"  : sparse sensor robustness sweep
    #   "noise_sweep"   : observation-noise robustness sweep
    #   "all_modes"     : run main + ablation + sensor_sweep + noise_sweep sequentially for one equation
    run_mode: str = "main"

    # For vkdv_paper and vsg. Options are problem-dependent.
    # vkdv_paper coefficient_family: "linear", "cubic", "cos", "exp_decay", "mixed_random"
    # vsg coefficient_family       : "linear", "quadratic", "cos", "mixed_random"
    coefficient_family: str = "mixed_random"

    # Main methods.
    # For main runs, keep all enabled.
    run_methods: Tuple[str, ...] = ("cabissm", "deeponet", "fnoenc", "vc_pinn")

    # ------------------------------
    # Data sizes
    # ------------------------------
    train_size: int = 16384
    val_size: int = 2048
    n_sensors: int = 128
    sensor_noise_std: float = 0.01
    sim_batch_size: int = 128

    # Grids
    n_x: int = 129
    n_t: int = 301
    n_q: int = 129

    # ------------------------------
    # Amortized model training
    # ------------------------------
    batch_size: int = 64
    epochs: int = 50
    baseline_epochs: int = 50
    lr: float = 3e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    lambda_grad: float = 0.5
    lambda_tv: float = 1e-4
    early_stop_patience: int = 12

    # Runtime
    seed: int = 42
    num_workers: int = 2
    use_amp: bool = True
    amp_dtype: str = "bfloat16"  # good on RTX PRO 6000 / H100; FFT sections force FP32 internally
    use_compile: bool = False
    out_root: str = "/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline"

    # ------------------------------
    # CABiSSM architecture
    # ------------------------------
    d_model: int = 128
    n_heads: int = 8
    n_layers: int = 4
    state_dim: int = 8
    mlp_ratio: int = 4
    dropout: float = 0.0
    use_fourier_features: bool = True
    use_cross_attention: bool = True
    use_ssm: bool = True

    # ------------------------------
    # DeepONet baseline
    # ------------------------------
    deeponet_width: int = 128
    deeponet_p: int = 128

    # ------------------------------
    # FNO-Encoder baseline
    # ------------------------------
    fno_width: int = 48
    fno_modes_t: int = 16
    fno_modes_x: int = 16
    fno_layers: int = 4
    fno_t_bins: int = 129
    fno_x_bins: int = 129

    # ------------------------------
    # VC-PINN-style baseline
    # ------------------------------
    # This is sample-wise, so full validation is expensive.
    vc_eval_samples: int = 16
    vc_hidden: int = 128
    vc_blocks_u: int = 4
    vc_blocks_c: int = 3
    vc_layers_per_block: int = 2
    vc_adam_steps: int = 2000
    vc_lbfgs_steps: int = 100
    vc_lr: float = 1e-3
    vc_colloc_f: int = 4096
    vc_colloc_ic: int = 512
    vc_colloc_bc: int = 512
    vc_w_data: float = 10.0
    vc_w_pde: float = 1.0
    vc_w_ic: float = 10.0
    vc_w_bc: float = 10.0
    vc_w_cbc: float = 10.0
    vc_w_c_smooth: float = 1e-4
    vc_log_every: int = 500

    # ------------------------------
    # Journal-strength extra experiments
    # ------------------------------
    sensor_sweep_values: Tuple[int, ...] = (16, 32, 64, 128)
    noise_sweep_values: Tuple[float, ...] = (0.0, 0.01, 0.03, 0.05)

    # For sweeps, use smaller sizes if you want a quick result.
    sweep_train_size: int = 8192
    sweep_val_size: int = 1024
    sweep_epochs: int = 35

    # Ablation variants.
    # These are trained only when run_mode == "ablation".
    ablation_names: Tuple[str, ...] = (
        "full_cabissm",
        "no_ssm_attention_only",
        "no_cross_ssm_only",
        "no_fourier_features",
        "no_gradient_loss",
    )

    # Artifacts
    make_pdf_summary: bool = True
    make_zip: bool = True
    save_checkpoints: bool = True


# ============================================================
# Utilities
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_amp_dtype(cfg: CFG):
    return torch.bfloat16 if cfg.amp_dtype.lower() == "bfloat16" else torch.float16


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def gradient_1d(field: torch.Tensor) -> torch.Tensor:
    return field[:, 1:, :] - field[:, :-1, :]


def total_variation_1d(field: torch.Tensor) -> torch.Tensor:
    return (field[:, 1:, :] - field[:, :-1, :]).abs().mean()


@torch.no_grad()
def coeff_metrics(c_hat: torch.Tensor, c_true: torch.Tensor) -> Dict[str, float]:
    c_hat = c_hat.float()
    c_true = c_true.float()

    mse = F.mse_loss(c_hat, c_true)
    mae = F.l1_loss(c_hat, c_true)
    rel_l2 = torch.norm(c_hat - c_true) / (torch.norm(c_true) + 1e-8)
    grad_mae = F.l1_loss(gradient_1d(c_hat), gradient_1d(c_true))

    pred_mean = c_hat.mean()
    pred_std = c_hat.std()
    true_mean = c_true.mean()
    true_std = c_true.std()

    norm_mae = mae / (true_std + 1e-8)
    norm_rmse = torch.sqrt(mse) / (true_std + 1e-8)
    var_ratio = pred_std / (true_std + 1e-8)
    mean_bias = pred_mean - true_mean

    return {
        "mse": float(mse.item()),
        "mae": float(mae.item()),
        "rel_l2": float(rel_l2.item()),
        "grad_mae": float(grad_mae.item()),
        "norm_mae": float(norm_mae.item()),
        "norm_rmse": float(norm_rmse.item()),
        "var_ratio": float(var_ratio.item()),
        "mean_bias": float(mean_bias.item()),
        "pred_mean": float(pred_mean.item()),
        "pred_std": float(pred_std.item()),
        "true_mean": float(true_mean.item()),
        "true_std": float(true_std.item()),
    }


def mean_metric(metrics: List[Dict[str, float]], key: str) -> float:
    vals = [m[key] for m in metrics if key in m]
    return sum(vals) / max(len(vals), 1)


def save_json(obj: Dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def move_batch(batch: Dict[str, torch.Tensor], device: torch.device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# Dataset
# ============================================================
class InversePDEDataset(Dataset):
    def __init__(self, sensors: torch.Tensor, q_grid: torch.Tensor, coeff_true: torch.Tensor):
        self.sensors = sensors.float()
        self.q_grid = q_grid.float()
        self.coeff_true = coeff_true.float()

    def __len__(self):
        return self.sensors.shape[0]

    def __getitem__(self, idx):
        return {
            "sensors": self.sensors[idx],
            "q_grid": self.q_grid[idx],
            "coeff_true": self.coeff_true[idx],
        }


def make_loader(ds: Dataset, cfg: CFG, shuffle: bool):
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(cfg.num_workers > 0),
    )


# ============================================================
# PDE problem classes
# ============================================================
class BaseProblem:
    name = "base"
    target_axis = "x"  # "x" or "t"
    coeff_min = 0.0
    coeff_max = 1.0
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0

    def make_grids(self, cfg: CFG, device: torch.device):
        x = torch.linspace(self.x_min, self.x_max, cfg.n_x, device=device)
        t = torch.linspace(self.t_min, self.t_max, cfg.n_t, device=device)
        if self.target_axis == "x":
            q = torch.linspace(self.x_min, self.x_max, cfg.n_q, device=device)
        else:
            q = torch.linspace(self.t_min, self.t_max, cfg.n_q, device=device)
        return x, t, q

    def generate_coeff(self, cfg: CFG, batch_size: int, q: torch.Tensor, device: torch.device):
        raise NotImplementedError

    def solve(self, cfg: CFG, coeff: torch.Tensor, x: torch.Tensor, t: torch.Tensor, q: torch.Tensor):
        raise NotImplementedError

    def sample_sensors(self, cfg: CFG, U: torch.Tensor, x: torch.Tensor, t: torch.Tensor):
        B = U.shape[0]
        Ns = cfg.n_sensors
        device = U.device
        ix = torch.randint(0, len(x), (B, Ns), device=device)
        it = torch.randint(0, len(t), (B, Ns), device=device)
        b = torch.arange(B, device=device).unsqueeze(1).expand(B, Ns)
        u = U[b, it, ix]
        if cfg.sensor_noise_std > 0:
            u = u + cfg.sensor_noise_std * torch.randn_like(u)
        return torch.stack([x[ix], t[it], u], dim=-1)

    @torch.no_grad()
    def build_dataset(self, cfg: CFG, size: int, device: torch.device, tag: str):
        x, t, q = self.make_grids(cfg, device)
        sensors_all, coeff_all, q_all = [], [], []
        num_batches = math.ceil(size / cfg.sim_batch_size)
        print(f"[{self.name}] Building {tag} dataset: {size} samples, {num_batches} simulation batches")
        start = time.time()
        for bi in range(num_batches):
            bs = min(cfg.sim_batch_size, size - bi * cfg.sim_batch_size)
            coeff = self.generate_coeff(cfg, bs, q, device)
            U = self.solve(cfg, coeff, x, t, q)
            sensors = self.sample_sensors(cfg, U, x, t)
            q_grid = q.view(1, -1, 1).repeat(bs, 1, 1)
            sensors_all.append(sensors.cpu())
            coeff_all.append(coeff.unsqueeze(-1).cpu())
            q_all.append(q_grid.cpu())
            if (bi + 1) % max(1, num_batches // 10) == 0 or (bi + 1) == num_batches:
                print(f"  [{self.name}/{tag}] batch {bi + 1}/{num_batches} | elapsed {time.time() - start:.1f}s")
        return InversePDEDataset(torch.cat(sensors_all), torch.cat(q_all), torch.cat(coeff_all))

    def coeff_input(self, x_col: torch.Tensor, t_col: torch.Tensor) -> torch.Tensor:
        return x_col if self.target_axis == "x" else t_col

    def pde_residual(self, model, x_col: torch.Tensor, t_col: torch.Tensor):
        raise NotImplementedError

    def aux_losses(self, model, sample: Dict[str, torch.Tensor], cfg: CFG, device: torch.device):
        return {}


class WaveProblem(BaseProblem):
    name = "wave"
    target_axis = "x"
    coeff_min = 0.75
    coeff_max = 1.45
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    beta_ic = 3

    def initial_condition(self, x):
        return torch.sin(math.pi * x) + 0.5 * torch.sin(self.beta_ic * math.pi * x)

    def smooth(self, field, kernel_size=9):
        pad = kernel_size // 2
        z = field.unsqueeze(1)
        z = F.pad(z, (pad, pad), mode="replicate")
        z = F.avg_pool1d(z, kernel_size=kernel_size, stride=1)
        return z.squeeze(1)

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            mode = random.choice(["sin", "gauss", "piecewise"])
            base = random.uniform(0.95, 1.15)
            xx = q_cpu.clone()
            if mode == "sin":
                c = base
                c = c + random.uniform(0.08, 0.20) * torch.sin(2 * math.pi * xx + random.uniform(0, 2 * math.pi))
                c = c + random.uniform(0.03, 0.10) * torch.sin(4 * math.pi * xx + random.uniform(0, 2 * math.pi))
            elif mode == "gauss":
                c = torch.full_like(xx, base)
                for _ in range(random.randint(1, 3)):
                    amp = random.uniform(-0.18, 0.18)
                    ctr = random.uniform(0.1, 0.9)
                    wid = random.uniform(0.03, 0.12)
                    c = c + amp * torch.exp(-0.5 * ((xx - ctr) / wid) ** 2)
            else:
                n_segments = random.randint(3, 6)
                edges = sorted(random.sample(range(8, cfg.n_q - 8), n_segments - 1))
                edges = [0] + edges + [cfg.n_q]
                c = torch.empty_like(xx)
                cur = base
                for s in range(len(edges) - 1):
                    cur = max(self.coeff_min, min(self.coeff_max, cur + random.uniform(-0.18, 0.18)))
                    c[edges[s]:edges[s + 1]] = cur
            fields.append(c)
        c = torch.stack(fields).to(device)
        c = self.smooth(c)
        return c.clamp(self.coeff_min, self.coeff_max)

    def div_operator(self, u, a_half, dx):
        out = torch.zeros_like(u)
        out[:, 1:-1] = (
            a_half[:, 1:] * (u[:, 2:] - u[:, 1:-1])
            - a_half[:, :-1] * (u[:, 1:-1] - u[:, :-2])
        ) / (dx * dx)
        return out

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        coeff_x = coeff if coeff.shape[1] == len(x) else F.interpolate(coeff.unsqueeze(1), size=len(x), mode="linear", align_corners=True).squeeze(1)
        B, Nx = coeff_x.shape
        Nt = len(t)
        dx = float((self.x_max - self.x_min) / (Nx - 1))
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        if dt > 0.95 * dx / self.coeff_max:
            print("Warning: wave CFL may be high. Increase n_t or reduce coeff_max.")
        a = coeff_x * coeff_x
        a_half = 0.5 * (a[:, :-1] + a[:, 1:])
        u0 = self.initial_condition(x).unsqueeze(0).repeat(B, 1)
        U = torch.zeros(B, Nt, Nx, device=x.device)
        U[:, 0] = u0
        Lu0 = self.div_operator(u0, a_half, dx)
        u1 = u0.clone()
        u1[:, 1:-1] = u0[:, 1:-1] + 0.5 * dt * dt * Lu0[:, 1:-1]
        u1[:, 0] = 0.0
        u1[:, -1] = 0.0
        U[:, 1] = u1
        up, uc = u0, u1
        for n in range(1, Nt - 1):
            Lu = self.div_operator(uc, a_half, dx)
            un = 2 * uc - up + dt * dt * Lu
            un[:, 0] = 0.0
            un[:, -1] = 0.0
            U[:, n + 1] = un
            up, uc = uc, un
        return U

    def pde_residual(self, model, x_col, t_col):
        u = model.u(x_col, t_col)
        c = model.coeff(x_col)
        a = c * c
        u_t = autograd_grad(u, t_col)
        u_tt = autograd_grad(u_t, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        a_x = autograd_grad(a, x_col)
        return u_tt - (a_x * u_x + a * u_xx)

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device).requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        u_ic = model.u(x_ic, t_ic)
        u_true = self.initial_condition(x_ic.squeeze(-1)).unsqueeze(-1)
        u_t_ic = autograd_grad(u_ic, t_ic)
        losses["ic"] = F.mse_loss(u_ic, u_true) + F.mse_loss(u_t_ic, torch.zeros_like(u_t_ic))
        t_bc = torch.rand(cfg.vc_colloc_bc, 1, device=device).requires_grad_(True)
        x0 = torch.zeros_like(t_bc).requires_grad_(True)
        x1 = torch.ones_like(t_bc).requires_grad_(True)
        losses["bc"] = F.mse_loss(model.u(x0, t_bc), torch.zeros_like(t_bc)) + F.mse_loss(model.u(x1, t_bc), torch.zeros_like(t_bc))
        return losses


class ADRProblem(BaseProblem):
    name = "adr"
    target_axis = "x"
    coeff_min = 0.0015
    coeff_max = 0.0060
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    v = 0.4
    lam = 1.0

    def initial_condition(self, x):
        return torch.sin(math.pi * x)

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            base = random.uniform(0.003, 0.0045)
            D = base + random.uniform(0.0004, 0.0012) * torch.sin(2 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            mode = random.choice(["smooth", "bumps", "piecewise"])
            if mode == "bumps":
                for _ in range(random.randint(1, 3)):
                    ctr = random.uniform(0.15, 0.85)
                    wid = random.uniform(0.035, 0.15)
                    amp = random.uniform(-0.0012, 0.0012)
                    D = D + amp * torch.exp(-0.5 * ((q_cpu - ctr) / wid) ** 2)
            elif mode == "piecewise":
                jump = torch.zeros_like(q_cpu)
                ctr = random.uniform(0.25, 0.75)
                jump[q_cpu > ctr] = random.uniform(-0.001, 0.001)
                D = D + jump
            fields.append(D)
        D = torch.stack(fields).to(device)
        D = F.avg_pool1d(F.pad(D.unsqueeze(1), (4, 4), mode="replicate"), kernel_size=9, stride=1).squeeze(1)
        return D.clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        D = coeff if coeff.shape[1] == len(x) else F.interpolate(coeff.unsqueeze(1), size=len(x), mode="linear", align_corners=True).squeeze(1)
        B, Nx = D.shape
        Nt = len(t)
        dx = float((self.x_max - self.x_min) / (Nx - 1))
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        D_half = 0.5 * (D[:, :-1] + D[:, 1:])
        U = torch.zeros(B, Nt, Nx, device=x.device)
        U[:, 0] = self.initial_condition(x).unsqueeze(0).repeat(B, 1)
        for n in range(Nt - 1):
            u = U[:, n]
            diff = torch.zeros_like(u)
            diff[:, 1:-1] = (
                D_half[:, 1:] * (u[:, 2:] - u[:, 1:-1])
                - D_half[:, :-1] * (u[:, 1:-1] - u[:, :-2])
            ) / (dx * dx)
            ux_up = torch.zeros_like(u)
            ux_up[:, 1:] = (u[:, 1:] - u[:, :-1]) / dx
            reaction = self.lam * u * (1.0 - u)
            un = u + dt * (diff - self.v * ux_up + reaction)
            un[:, 0] = 0.0
            un[:, -1] = 0.0
            U[:, n + 1] = un.clamp(-3.0, 3.0)
        return U

    def pde_residual(self, model, x_col, t_col):
        u = model.u(x_col, t_col)
        D = model.coeff(x_col)
        u_t = autograd_grad(u, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        D_x = autograd_grad(D, x_col)
        return u_t - (D_x * u_x + D * u_xx - self.v * u_x + self.lam * u * (1.0 - u))

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device).requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_condition(x_ic.squeeze(-1)).unsqueeze(-1))
        t_bc = torch.rand(cfg.vc_colloc_bc, 1, device=device).requires_grad_(True)
        x0 = torch.zeros_like(t_bc).requires_grad_(True)
        x1 = torch.ones_like(t_bc).requires_grad_(True)
        losses["bc"] = F.mse_loss(model.u(x0, t_bc), torch.zeros_like(t_bc)) + F.mse_loss(model.u(x1, t_bc), torch.zeros_like(t_bc))
        return losses


class VKdVProblem(BaseProblem):
    name = "vkdv"
    target_axis = "t"
    coeff_min = 0.50
    coeff_max = 1.50
    x_min = -1.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    kappa = 0.75
    x0 = -0.55

    def sech2(self, z):
        return 1.0 / torch.cosh(z).pow(2)

    def initial_profile(self, x):
        return 2.0 * self.kappa * self.kappa * self.sech2(self.kappa * (x - self.x0))

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            base = random.uniform(0.85, 1.15)
            g = base
            g = g + random.uniform(0.08, 0.22) * torch.sin(2 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            g = g + random.uniform(0.03, 0.12) * torch.sin(4 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            if random.random() < 0.5:
                g = g * torch.exp(-random.uniform(0.0, 0.4) * q_cpu)
            fields.append(g)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        g_t = coeff if coeff.shape[1] == len(t) else F.interpolate(coeff.unsqueeze(1), size=len(t), mode="linear", align_corners=True).squeeze(1)
        B, Nt = g_t.shape
        Nx = len(x)
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        tau = torch.zeros_like(g_t)
        tau[:, 1:] = torch.cumsum(0.5 * (g_t[:, 1:] + g_t[:, :-1]) * dt, dim=1)
        X = x.view(1, 1, Nx)
        Tau = tau.view(B, Nt, 1)
        center = self.x0 + 4.0 * self.kappa * self.kappa * Tau
        U = 2.0 * self.kappa * self.kappa * self.sech2(self.kappa * (X - center))
        return U

    def pde_residual(self, model, x_col, t_col):
        # u_t + 6 g(t) u u_x + g(t) u_xxx = 0
        u = model.u(x_col, t_col)
        g = model.coeff(t_col)
        u_t = autograd_grad(u, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        u_xxx = autograd_grad(u_xx, x_col)
        return u_t + 6.0 * g * u * u_x + g * u_xxx

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device) * (self.x_max - self.x_min) + self.x_min
        x_ic.requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_profile(x_ic).detach())
        return losses


class VKdVPaperProblem(VKdVProblem):
    name = "vkdv_paper"
    # Same PDE form as VKdVProblem in this code, but coefficients follow fixed families
    # inspired by VC-PINN vKdV experiments: linear, cubic, cosine, decaying oscillation.

    def generate_coeff(self, cfg, batch_size, q, device):
        t_cpu = q.detach().cpu()
        fields = []
        for _ in range(batch_size):
            fam = cfg.coefficient_family
            if fam == "mixed_random":
                fam = random.choice(["linear", "cubic", "cos", "exp_decay"])
            if fam == "linear":
                # scaled linear, with small random slope/offset
                a = random.uniform(0.25, 0.55)
                b = random.uniform(0.75, 1.05)
                g = b + a * t_cpu
            elif fam == "cubic":
                a = random.uniform(0.20, 0.50)
                b = random.uniform(0.80, 1.05)
                g = b + a * (t_cpu ** 3)
            elif fam == "cos":
                base = random.uniform(0.95, 1.10)
                amp = random.uniform(0.15, 0.35)
                phase = random.uniform(0, 2 * math.pi)
                g = base + amp * torch.cos(2 * math.pi * t_cpu + phase)
            elif fam == "exp_decay":
                base = random.uniform(0.90, 1.15)
                amp = random.uniform(0.15, 0.35)
                decay = random.uniform(0.8, 1.8)
                freq = random.uniform(1.0, 2.5)
                phase = random.uniform(0, 2 * math.pi)
                g = base + amp * torch.exp(-decay * t_cpu) * torch.cos(2 * math.pi * freq * t_cpu + phase)
            else:
                raise ValueError(f"Unknown vkdv_paper coefficient_family: {cfg.coefficient_family}")
            fields.append(g)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)


class VSGProblem(BaseProblem):
    name = "vsg"
    target_axis = "t"
    coeff_min = 0.50
    coeff_max = 1.50
    x_min = -1.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    k = 1.0

    def generate_coeff(self, cfg, batch_size, q, device):
        t_cpu = q.detach().cpu()
        fields = []
        for _ in range(batch_size):
            fam = cfg.coefficient_family
            if fam == "mixed_random":
                fam = random.choice(["linear", "quadratic", "cos"])
            if fam == "linear":
                h = random.uniform(0.7, 1.0) + random.uniform(0.2, 0.5) * t_cpu
            elif fam == "quadratic":
                h = random.uniform(0.7, 1.0) + random.uniform(0.2, 0.5) * (t_cpu ** 2)
            elif fam == "cos":
                h = random.uniform(0.95, 1.10) + random.uniform(0.15, 0.35) * torch.cos(2 * math.pi * t_cpu + random.uniform(0, 2 * math.pi))
            else:
                raise ValueError(f"Unknown vsg coefficient_family: {cfg.coefficient_family}")
            fields.append(h)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        h_t = coeff if coeff.shape[1] == len(t) else F.interpolate(coeff.unsqueeze(1), size=len(t), mode="linear", align_corners=True).squeeze(1)
        B, Nt = h_t.shape
        Nx = len(x)
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        omega = torch.zeros_like(h_t)
        omega[:, 1:] = torch.cumsum(0.5 * (h_t[:, 1:] + h_t[:, :-1]) * dt / self.k, dim=1)
        X = x.view(1, 1, Nx)
        Om = omega.view(B, Nt, 1)
        U = 4.0 * torch.atan(torch.exp(self.k * X - Om))
        return U

    def initial_profile(self, x):
        return 4.0 * torch.atan(torch.exp(self.k * x))

    def pde_residual(self, model, x_col, t_col):
        # u_xt + h(t) sin(u) = 0
        u = model.u(x_col, t_col)
        h = model.coeff(t_col)
        u_x = autograd_grad(u, x_col)
        u_xt = autograd_grad(u_x, t_col)
        return u_xt + h * torch.sin(u)

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device) * (self.x_max - self.x_min) + self.x_min
        x_ic.requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_profile(x_ic).detach())
        return losses


def get_problem(name: str) -> BaseProblem:
    if name == "wave":
        return WaveProblem()
    if name == "adr":
        return ADRProblem()
    if name == "vkdv":
        return VKdVProblem()
    if name == "vkdv_paper":
        return VKdVPaperProblem()
    if name == "vsg":
        return VSGProblem()
    raise ValueError(f"Unknown problem_name: {name}")


# ============================================================
# Model components
# ============================================================
class FourierFeatures(nn.Module):
    def __init__(self, in_dim: int, num_bands: int = 8, scale: float = 4.0, enabled: bool = True):
        super().__init__()
        self.in_dim = in_dim
        self.num_bands = num_bands
        self.enabled = enabled
        if enabled:
            freqs = torch.linspace(1.0, num_bands, num_bands) * scale
            self.register_buffer("freqs", freqs)
        else:
            self.register_buffer("freqs", torch.empty(0))

    @property
    def out_dim(self):
        if not self.enabled:
            return self.in_dim
        return self.in_dim + 2 * self.in_dim * self.num_bands

    def forward(self, x):
        if not self.enabled:
            return x
        outs = [x]
        for i in range(self.in_dim):
            xi = x[..., i:i + 1]
            w = self.freqs.view(*([1] * (x.ndim - 1)), -1)
            outs.append(torch.sin(2 * math.pi * xi * w))
            outs.append(torch.cos(2 * math.pi * xi * w))
        return torch.cat(outs, dim=-1)


class FeedForward(nn.Module):
    def __init__(self, d_model, mlp_ratio=4, dropout=0.0):
        super().__init__()
        hidden = d_model * mlp_ratio
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class SensorEncoder(nn.Module):
    def __init__(self, cfg: CFG, u_mean: float, u_std: float):
        super().__init__()
        ff_on = cfg.use_fourier_features
        self.coord_ff = FourierFeatures(2, num_bands=8, scale=4.0, enabled=ff_on)
        self.val_ff = FourierFeatures(1, num_bands=4, scale=4.0, enabled=ff_on)
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        in_dim = self.coord_ff.out_dim + self.val_ff.out_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, sensors):
        coords = sensors[..., :2]
        vals = (sensors[..., 2:3] - self.u_mean) / self.u_std
        z = torch.cat([self.coord_ff(coords), self.val_ff(vals)], dim=-1)
        return self.norm(self.net(z))


class QueryEmbedder(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.ff = FourierFeatures(1, num_bands=16, scale=4.0, enabled=cfg.use_fourier_features)
        self.net = nn.Sequential(
            nn.Linear(self.ff.out_dim, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, q):
        return self.norm(self.net(self.ff(q)))


class CrossAttentionBlock(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.norm_q = nn.LayerNorm(cfg.d_model)
        self.norm_kv = nn.LayerNorm(cfg.d_model)
        self.attn = nn.MultiheadAttention(cfg.d_model, cfg.n_heads, batch_first=True, dropout=cfg.dropout)
        self.norm_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)

    def forward(self, q, kv):
        qn = self.norm_q(q)
        kvn = self.norm_kv(kv)
        attn_out, _ = self.attn(qn, kvn, kvn, need_weights=False)
        x = q + attn_out
        x = x + self.ffn(self.norm_ffn(x))
        return x


class NoCrossConditioning(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )

    def forward(self, h, sensor_tokens):
        ctx = sensor_tokens.mean(dim=1, keepdim=True)
        return h + self.proj(ctx)


class DiagonalSelectiveSSM(nn.Module):
    """
    Lightweight selective diagonal SSM, not full Mamba.
    The update is input-dependent through dt and gate:
        h_k = alpha_k h_{k-1} + (1-alpha_k) B u_k
        y_k = C h_k + D u_k
    """
    def __init__(self, d_model, state_dim, dropout=0.0):
        super().__init__()
        self.d_model = d_model
        self.state_dim = state_dim
        self.in_proj = nn.Linear(d_model, d_model)
        self.gate_proj = nn.Linear(d_model, d_model)
        self.dt_proj = nn.Linear(d_model, d_model * state_dim)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.A_log = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.B = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.C = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.D = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        Bsz, L, Dm = x.shape
        u = self.in_proj(x)
        gate = torch.sigmoid(self.gate_proj(x))
        dt = F.softplus(self.dt_proj(x)).view(Bsz, L, Dm, self.state_dim) + 1e-4
        A = F.softplus(self.A_log).unsqueeze(0).unsqueeze(0)
        alpha = torch.exp(-dt * A)
        Bp = self.B.unsqueeze(0)
        Cp = self.C.unsqueeze(0)
        Dp = self.D.unsqueeze(0)
        h = torch.zeros(Bsz, Dm, self.state_dim, device=x.device, dtype=x.dtype)
        ys = []
        for k in range(L):
            uk = u[:, k, :].unsqueeze(-1)
            h = alpha[:, k] * h + (1.0 - alpha[:, k]) * (uk * Bp)
            yk = (h * Cp).sum(dim=-1) + Dp * u[:, k, :]
            ys.append(yk)
        y = torch.stack(ys, dim=1)
        y = gate * y
        return self.out_proj(self.dropout(y))


class BiSSMBlock(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.norm_f = nn.LayerNorm(cfg.d_model)
        self.norm_b = nn.LayerNorm(cfg.d_model)
        self.fwd = DiagonalSelectiveSSM(cfg.d_model, cfg.state_dim, cfg.dropout)
        self.bwd = DiagonalSelectiveSSM(cfg.d_model, cfg.state_dim, cfg.dropout)
        self.mix = nn.Linear(2 * cfg.d_model, cfg.d_model)
        self.norm_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)

    def forward(self, x):
        xf = self.fwd(self.norm_f(x))
        xb = torch.flip(self.bwd(torch.flip(self.norm_b(x), dims=[1])), dims=[1])
        x = x + self.mix(torch.cat([xf, xb], dim=-1))
        x = x + self.ffn(self.norm_ffn(x))
        return x


class CABiSSMInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.cfg = cfg
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.sensor_encoder = SensorEncoder(cfg, u_mean, u_std)
        self.query_embedder = QueryEmbedder(cfg)
        self.cross_blocks = nn.ModuleList()
        self.nocross_blocks = nn.ModuleList()
        self.ssm_blocks = nn.ModuleList()
        self.ffn_blocks = nn.ModuleList()
        for _ in range(cfg.n_layers):
            self.cross_blocks.append(CrossAttentionBlock(cfg))
            self.nocross_blocks.append(NoCrossConditioning(cfg))
            self.ssm_blocks.append(BiSSMBlock(cfg))
            self.ffn_blocks.append(nn.Sequential(nn.LayerNorm(cfg.d_model), FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)))
        self.norm = nn.LayerNorm(cfg.d_model)
        self.decoder = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(), nn.Linear(cfg.d_model, 1))

    def forward(self, sensors, q_grid):
        sensor_tokens = self.sensor_encoder(sensors)
        h = self.query_embedder(q_grid)
        for i in range(self.cfg.n_layers):
            if self.cfg.use_cross_attention:
                h = self.cross_blocks[i](h, sensor_tokens)
            else:
                h = self.nocross_blocks[i](h, sensor_tokens)
            if self.cfg.use_ssm:
                h = self.ssm_blocks[i](h)
            else:
                h = h + self.ffn_blocks[i](h)
        raw = self.decoder(self.norm(h))
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


class DeepONetInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        self.coord_ff = FourierFeatures(2, 8, 4.0, enabled=cfg.use_fourier_features)
        self.val_ff = FourierFeatures(1, 4, 4.0, enabled=cfg.use_fourier_features)
        sensor_in = self.coord_ff.out_dim + self.val_ff.out_dim
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_in, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_width),
        )
        self.branch = nn.Sequential(
            nn.Linear(2 * cfg.deeponet_width, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_p),
        )
        self.trunk_ff = FourierFeatures(1, 16, 4.0, enabled=cfg.use_fourier_features)
        self.trunk = nn.Sequential(
            nn.Linear(self.trunk_ff.out_dim, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_p),
        )
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, sensors, q_grid):
        vals = (sensors[..., 2:3] - self.u_mean) / self.u_std
        z = torch.cat([self.coord_ff(sensors[..., :2]), self.val_ff(vals)], dim=-1)
        z = self.sensor_mlp(z)
        z = torch.cat([z.mean(dim=1), z.max(dim=1).values], dim=-1)
        b = self.branch(z)
        tr = self.trunk(self.trunk_ff(q_grid))
        raw = (b.unsqueeze(1) * tr).sum(dim=-1, keepdim=True) / math.sqrt(b.shape[-1]) + self.bias
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_t, modes_x):
        super().__init__()
        scale = 1 / max(1, in_channels * out_channels)
        self.modes_t = modes_t
        self.modes_x = modes_x
        self.weights_pos = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes_t, modes_x, dtype=torch.cfloat))
        self.weights_neg = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes_t, modes_x, dtype=torch.cfloat))

    def compl_mul2d(self, x, w):
        return torch.einsum("bixy,ioxy->boxy", x, w)

    def forward(self, x):
        original_dtype = x.dtype
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            B, C, T, X = x.shape
            x_ft = torch.fft.rfft2(x, dim=(-2, -1))
            out_ft = torch.zeros(B, self.weights_pos.shape[1], T, X // 2 + 1, device=x.device, dtype=torch.cfloat)
            mt = min(self.modes_t, T)
            mx = min(self.modes_x, X // 2 + 1)
            out_ft[:, :, :mt, :mx] = self.compl_mul2d(x_ft[:, :, :mt, :mx], self.weights_pos[:, :, :mt, :mx])
            out_ft[:, :, -mt:, :mx] = self.compl_mul2d(x_ft[:, :, -mt:, :mx], self.weights_neg[:, :, :mt, :mx])
            y = torch.fft.irfft2(out_ft, s=(T, X), dim=(-2, -1))
        return y.to(original_dtype)


class FNOBlock(nn.Module):
    def __init__(self, width, modes_t, modes_x):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes_t, modes_x)
        self.point = nn.Conv2d(width, width, 1)
        self.norm = nn.GroupNorm(8, width)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.point(x)))


def sensors_to_grid(cfg: CFG, problem: BaseProblem, sensors, u_mean, u_std):
    device = sensors.device
    dtype = sensors.dtype
    B, Ns, _ = sensors.shape
    T, X = cfg.fno_t_bins, cfg.fno_x_bins
    x = sensors[..., 0]
    t = sensors[..., 1]
    u = (sensors[..., 2] - u_mean.to(device)) / u_std.to(device)
    ix = torch.round((x - problem.x_min) / (problem.x_max - problem.x_min) * (X - 1)).long().clamp(0, X - 1)
    it = torch.round((t - problem.t_min) / (problem.t_max - problem.t_min) * (T - 1)).long().clamp(0, T - 1)
    obs = torch.zeros(B, T, X, device=device, dtype=dtype)
    cnt = torch.zeros(B, T, X, device=device, dtype=dtype)
    b = torch.arange(B, device=device).view(B, 1).expand(B, Ns)
    flat = (b * T * X + it * X + ix).reshape(-1)
    obs.reshape(-1).scatter_add_(0, flat, u.reshape(-1).to(dtype))
    cnt.reshape(-1).scatter_add_(0, flat, torch.ones_like(u, dtype=dtype).reshape(-1))
    mask = (cnt > 0).to(dtype)
    obs = obs / cnt.clamp_min(1.0)
    return torch.stack([obs, mask], dim=1)


class FNOEncoderInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.cfg = cfg
        self.problem = problem
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        self.lift = nn.Conv2d(4, cfg.fno_width, 1)
        self.blocks = nn.ModuleList([FNOBlock(cfg.fno_width, cfg.fno_modes_t, cfg.fno_modes_x) for _ in range(cfg.fno_layers)])
        self.q_ff = FourierFeatures(1, 16, 4.0, enabled=cfg.use_fourier_features)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fno_width + self.q_ff.out_dim, cfg.fno_width), nn.GELU(),
            nn.Linear(cfg.fno_width, cfg.fno_width), nn.GELU(),
            nn.Linear(cfg.fno_width, 1),
        )

    def forward(self, sensors, q_grid):
        B = sensors.shape[0]
        T, X = self.cfg.fno_t_bins, self.cfg.fno_x_bins
        grid = sensors_to_grid(self.cfg, self.problem, sensors, self.u_mean, self.u_std)
        x_coord = torch.linspace(self.problem.x_min, self.problem.x_max, X, device=sensors.device, dtype=sensors.dtype).view(1, 1, 1, X).expand(B, 1, T, X)
        t_coord = torch.linspace(self.problem.t_min, self.problem.t_max, T, device=sensors.device, dtype=sensors.dtype).view(1, 1, T, 1).expand(B, 1, T, X)
        z = torch.cat([grid, x_coord, t_coord], dim=1)
        z = self.lift(z)
        for blk in self.blocks:
            z = blk(z)
        latent = z.mean(dim=(-2, -1))
        latent = latent.unsqueeze(1).expand(B, q_grid.shape[1], latent.shape[-1])
        raw = self.decoder(torch.cat([latent, self.q_ff(q_grid)], dim=-1))
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


# ============================================================
# Training helpers
# ============================================================
def compute_train_stats(ds: InversePDEDataset):
    u = ds.sensors[..., 2]
    c = ds.coeff_true
    g = gradient_1d(c)
    return {
        "u_mean": float(u.mean().item()),
        "u_std": float(u.std().item() + 1e-6),
        "c_mean": float(c.mean().item()),
        "c_std": float(c.std().item() + 1e-6),
        "g_std": float(g.std().item() + 1e-6),
    }


def supervised_loss(c_hat, c_true, stats, cfg):
    loss_field = F.mse_loss((c_hat - c_true) / stats["c_std"], torch.zeros_like(c_hat))
    if cfg.lambda_grad > 0:
        loss_grad = F.mse_loss((gradient_1d(c_hat) - gradient_1d(c_true)) / stats["g_std"], torch.zeros_like(gradient_1d(c_hat)))
    else:
        loss_grad = torch.tensor(0.0, device=c_hat.device)
    loss_tv = total_variation_1d(c_hat)
    return loss_field + cfg.lambda_grad * loss_grad + cfg.lambda_tv * loss_tv, {
        "loss_field": float(loss_field.detach().item()),
        "loss_grad": float(loss_grad.detach().item()),
        "loss_tv": float(loss_tv.detach().item()),
    }


def build_model(model_key: str, cfg: CFG, problem: BaseProblem, stats: Dict[str, float]) -> nn.Module:
    if model_key == "cabissm":
        return CABiSSMInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    if model_key == "deeponet":
        return DeepONetInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    if model_key == "fnoenc":
        return FNOEncoderInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    raise ValueError(model_key)


def nice_name(model_key: str) -> str:
    return {"cabissm": "CABiSSM", "deeponet": "DeepONet", "fnoenc": "FNOEnc"}.get(model_key, model_key)


def train_amortized_model(cfg, problem, model, model_name, train_ds, val_ds, stats, device, out_dir, epochs):
    train_loader = make_loader(train_ds, cfg, True)
    val_loader = make_loader(val_ds, cfg, False)
    model = model.to(device)
    if cfg.use_compile and hasattr(torch, "compile"):
        model = torch.compile(model)
    print(f"[{problem.name}] {model_name} parameters: {count_params(model):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    amp_dtype = get_amp_dtype(cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.use_amp and device.type == "cuda" and amp_dtype == torch.float16))
    history = {"train_loss": [], "val_loss": [], "val_mae": [], "val_rel_l2": [], "val_grad_mae": [], "val_norm_mae": [], "val_var_ratio": []}
    best_val = float("inf")
    best_metrics = None
    best_epoch = -1
    best_state = None
    no_improve = 0
    best_path = out_dir / f"{problem.name}_{model_name.lower()}_best.pt"
    start_train = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        train_total, train_count = 0.0, 0
        for batch in train_loader:
            batch = move_batch(batch, device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(cfg.use_amp and device.type == "cuda")):
                pred = model(batch["sensors"], batch["q_grid"])
                loss, _ = supervised_loss(pred, batch["coeff_true"], stats, cfg)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                opt.step()
            bs = batch["sensors"].shape[0]
            train_total += float(loss.detach()) * bs
            train_count += bs
        sched.step()
        model.eval()
        val_total, val_count = 0.0, 0
        metric_total: Dict[str, float] = {}
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch(batch, device)
                with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(cfg.use_amp and device.type == "cuda")):
                    pred = model(batch["sensors"], batch["q_grid"])
                    loss, _ = supervised_loss(pred, batch["coeff_true"], stats, cfg)
                bs = batch["sensors"].shape[0]
                val_total += float(loss.detach()) * bs
                val_count += bs
                m = coeff_metrics(pred.float(), batch["coeff_true"].float())
                for k, v in m.items():
                    metric_total[k] = metric_total.get(k, 0.0) + v * bs
        train_loss = train_total / max(train_count, 1)
        val_loss = val_total / max(val_count, 1)
        metrics = {k: v / max(val_count, 1) for k, v in metric_total.items()}
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(metrics["mae"])
        history["val_rel_l2"].append(metrics["rel_l2"])
        history["val_grad_mae"].append(metrics["grad_mae"])
        history["val_norm_mae"].append(metrics["norm_mae"])
        history["val_var_ratio"].append(metrics["var_ratio"])
        print(
            f"[{problem.name}] {model_name} ep {ep:03d} | train {train_loss:.3e} | val {val_loss:.3e} | "
            f"mae {metrics['mae']:.3e} | relL2 {metrics['rel_l2']:.3e} | grad {metrics['grad_mae']:.3e} | "
            f"normMAE {metrics['norm_mae']:.3e} | varRatio {metrics['var_ratio']:.3f}"
        )
        if val_loss < best_val:
            best_val = val_loss
            best_metrics = metrics
            best_epoch = ep
            no_improve = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if cfg.save_checkpoints:
                torch.save({"model": best_state, "metrics": metrics, "best_val": best_val, "epoch": ep, "history": history, "stats": stats}, best_path)
        else:
            no_improve += 1
        if model_name.startswith("CABiSSM") and no_improve >= cfg.early_stop_patience:
            print(f"[{problem.name}] Early stopping {model_name} at epoch {ep}; best epoch {best_epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    train_time = time.time() - start_train
    return model, history, best_metrics or {}, best_val, best_epoch, train_time


@torch.no_grad()
def evaluate_subset(model, ds, indices, device):
    model.eval()
    metrics, preds = [], []
    for idx in indices:
        s = ds[idx]
        sensors = s["sensors"].unsqueeze(0).to(device)
        q = s["q_grid"].unsqueeze(0).to(device)
        true = s["coeff_true"].unsqueeze(0).to(device)
        pred = model(sensors, q)
        metrics.append(coeff_metrics(pred.float(), true.float()))
        preds.append(pred.squeeze(0).cpu())
    return metrics, preds


@torch.no_grad()
def benchmark_inference(model, ds, cfg, device, n_batches=10):
    loader = make_loader(ds, cfg, False)
    model.eval()
    times = []
    count = 0
    # warmup
    for i, batch in enumerate(loader):
        batch = move_batch(batch, device)
        _ = model(batch["sensors"], batch["q_grid"])
        if i >= 2:
            break
    if device.type == "cuda":
        torch.cuda.synchronize()
    for i, batch in enumerate(loader):
        if i >= n_batches:
            break
        batch = move_batch(batch, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        _ = model(batch["sensors"], batch["q_grid"])
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - start
        times.append(elapsed)
        count += batch["sensors"].shape[0]
    total = sum(times)
    return {"inference_time_sec_total": total, "inference_samples": count, "inference_ms_per_sample": 1000.0 * total / max(count, 1)}


# ============================================================
# VC-PINN-style baseline
# ============================================================
def autograd_grad(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True, retain_graph=True)[0]


class PreActResidualBlock(nn.Module):
    def __init__(self, width, layers_per_block=2):
        super().__init__()
        layers = []
        for _ in range(layers_per_block):
            layers += [nn.Tanh(), nn.Linear(width, width)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.net(x)


class ResNetMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, blocks, layers_per_block):
        super().__init__()
        self.input = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([PreActResidualBlock(hidden, layers_per_block) for _ in range(blocks)])
        self.output = nn.Linear(hidden, out_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.input(x)
        for block in self.blocks:
            h = block(h)
        return self.output(h)


class VCPINNModel(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem):
        super().__init__()
        self.problem = problem
        self.u_net = ResNetMLP(2, cfg.vc_hidden, 1, cfg.vc_blocks_u, cfg.vc_layers_per_block)
        self.c_net = ResNetMLP(1, cfg.vc_hidden, 1, cfg.vc_blocks_c, cfg.vc_layers_per_block)

    def u(self, x, t):
        return self.u_net(torch.cat([x, t], dim=-1))

    def coeff(self, q):
        raw = self.c_net(q)
        return self.problem.coeff_min + (self.problem.coeff_max - self.problem.coeff_min) * torch.sigmoid(raw)


def vc_pinn_loss(problem, model, sample, cfg, device, colloc):
    sensors = sample["sensors"].to(device).float()
    q_grid = sample["q_grid"].to(device).float()
    coeff_true = sample["coeff_true"].to(device).float()
    x_s = sensors[:, 0:1]
    t_s = sensors[:, 1:2]
    u_s = sensors[:, 2:3]
    loss_data = F.mse_loss(model.u(x_s, t_s), u_s)
    x_f = colloc["x_f"].clone().detach().requires_grad_(True)
    t_f = colloc["t_f"].clone().detach().requires_grad_(True)
    loss_pde = torch.mean(problem.pde_residual(model, x_f, t_f) ** 2)
    aux = problem.aux_losses(model, sample, cfg, device)
    loss_ic = aux.get("ic", torch.tensor(0.0, device=device))
    loss_bc = aux.get("bc", torch.tensor(0.0, device=device))
    c0_pred = model.coeff(q_grid[0:1])
    c1_pred = model.coeff(q_grid[-1:])
    loss_cbc = F.mse_loss(c0_pred, coeff_true[0:1]) + F.mse_loss(c1_pred, coeff_true[-1:])
    c_grid = model.coeff(q_grid)
    loss_smooth = total_variation_1d(c_grid.unsqueeze(0))
    loss = (
        cfg.vc_w_data * loss_data + cfg.vc_w_pde * loss_pde + cfg.vc_w_ic * loss_ic +
        cfg.vc_w_bc * loss_bc + cfg.vc_w_cbc * loss_cbc + cfg.vc_w_c_smooth * loss_smooth
    )
    parts = {"data": float(loss_data.detach()), "pde": float(loss_pde.detach()), "ic": float(loss_ic.detach()), "bc": float(loss_bc.detach()), "cbc": float(loss_cbc.detach()), "smooth": float(loss_smooth.detach())}
    return loss, parts


@torch.no_grad()
def eval_vc_coeff(model, sample, device):
    q_grid = sample["q_grid"].to(device).float()
    coeff_true = sample["coeff_true"].to(device).float()
    pred = model.coeff(q_grid)
    return pred.cpu(), coeff_metrics(pred.unsqueeze(0), coeff_true.unsqueeze(0))


def run_vc_pinn_sample(problem, cfg, sample, device):
    model = VCPINNModel(cfg, problem).to(device).float()
    opt = torch.optim.Adam(model.parameters(), lr=cfg.vc_lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.vc_adam_steps)
    colloc = {
        "x_f": torch.rand(cfg.vc_colloc_f, 1, device=device) * (problem.x_max - problem.x_min) + problem.x_min,
        "t_f": torch.rand(cfg.vc_colloc_f, 1, device=device) * (problem.t_max - problem.t_min) + problem.t_min,
    }
    best_state, best_loss = None, float("inf")
    hist = []
    for step in range(1, cfg.vc_adam_steps + 1):
        opt.zero_grad(set_to_none=True)
        loss, parts = vc_pinn_loss(problem, model, sample, cfg, device, colloc)
        loss.backward()
        opt.step()
        sched.step()
        lv = float(loss.detach())
        hist.append(lv)
        if lv < best_loss:
            best_loss = lv
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if step == 1 or step % cfg.vc_log_every == 0 or step == cfg.vc_adam_steps:
            with torch.no_grad():
                _, m = eval_vc_coeff(model, sample, device)
            print(f"    [{problem.name}] VC-PINN {step:04d}/{cfg.vc_adam_steps} | loss {lv:.3e} | data {parts['data']:.2e} | pde {parts['pde']:.2e} | mae {m['mae']:.3e} | relL2 {m['rel_l2']:.3e}")
    if best_state is not None:
        model.load_state_dict(best_state)
    if cfg.vc_lbfgs_steps > 0:
        print(f"    [{problem.name}] VC-PINN L-BFGS refinement: {cfg.vc_lbfgs_steps} iters")
        lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=cfg.vc_lbfgs_steps, max_eval=2 * cfg.vc_lbfgs_steps, history_size=50, line_search_fn="strong_wolfe")
        def closure():
            lbfgs.zero_grad(set_to_none=True)
            loss, _ = vc_pinn_loss(problem, model, sample, cfg, device, colloc)
            loss.backward()
            return loss
        lbfgs.step(closure)
    pred, metrics = eval_vc_coeff(model, sample, device)
    return pred, metrics, hist


def run_vc_pinn_baseline(problem, cfg, val_ds, device, indices):
    metrics, preds, histories = [], [], []
    print(f"[{problem.name}] Running VC-PINN-style on {len(indices)} samples")
    total = time.time()
    for i, idx in enumerate(indices):
        print(f"  [{problem.name}] VC sample {i + 1}/{len(indices)} index={idx}")
        start = time.time()
        pred, m, hist = run_vc_pinn_sample(problem, cfg, val_ds[idx], device)
        m["wall_time_sec"] = time.time() - start
        metrics.append(m)
        preds.append(pred)
        histories.append(hist)
        print(f"  [{problem.name}] done idx={idx} | time {m['wall_time_sec']:.1f}s | MAE {m['mae']:.3e} | relL2 {m['rel_l2']:.3e}")
    print(f"[{problem.name}] VC-PINN total time: {time.time() - total:.1f}s")
    return metrics, preds, histories


# ============================================================
# Plotting and artifact saving
# ============================================================
def save_curves(history, out_dir, problem_name, method_name):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="train loss")
    plt.plot(epochs, history["val_loss"], label="val loss")
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"{problem_name}: {method_name} loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{problem_name}_{method_name}_loss.png", dpi=200)
    plt.close()
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_mae"], label="MAE")
    plt.plot(epochs, history["val_rel_l2"], label="relL2")
    plt.plot(epochs, history["val_grad_mae"], label="gradMAE")
    plt.plot(epochs, history["val_norm_mae"], label="normMAE")
    plt.xlabel("epoch")
    plt.ylabel("metric")
    plt.title(f"{problem_name}: {method_name} validation metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{problem_name}_{method_name}_metrics.png", dpi=200)
    plt.close()


def save_reconstruction_plot(problem, val_ds, indices, method_preds, out_dir, tag=""):
    if not method_preds:
        return
    n = len(indices)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))
    if n == 1:
        axes = axes[None, :]
    for row, idx in enumerate(indices):
        s = val_ds[idx]
        q = s["q_grid"].squeeze(-1)
        true = s["coeff_true"].squeeze(-1)
        sensors = s["sensors"]
        ax0 = axes[row, 0]
        sc = ax0.scatter(sensors[:, 0], sensors[:, 1], c=sensors[:, 2], s=16)
        ax0.set_xlabel("x")
        ax0.set_ylabel("t")
        ax0.set_title(f"{problem.name}: sparse sensors sample {idx}")
        plt.colorbar(sc, ax=ax0, fraction=0.046, pad=0.04)
        ax1 = axes[row, 1]
        ax1.plot(q, true, label="true coefficient", linewidth=2)
        for name, preds in method_preds.items():
            ax1.plot(q, preds[row].squeeze(-1), label=name, linewidth=2)
        ax1.set_xlabel(problem.target_axis)
        ax1.set_ylabel("coefficient")
        ax1.set_title(f"{problem.name}: coefficient reconstruction")
        ax1.legend(fontsize=8)
    plt.tight_layout()
    suffix = f"_{tag}" if tag else ""
    plt.savefig(out_dir / f"{problem.name}{suffix}_reconstructions.png", dpi=200)
    plt.close()


def save_barplot(problem_name, method_metrics, out_dir, tag=""):
    if not method_metrics:
        return
    keys = ["mae", "rel_l2", "grad_mae", "norm_mae", "var_ratio"]
    methods = list(method_metrics.keys())
    x = list(range(len(keys)))
    width = 0.8 / max(len(methods), 1)
    plt.figure(figsize=(12, 5))
    for i, method in enumerate(methods):
        vals = [mean_metric(method_metrics[method], k) for k in keys]
        offset = (i - (len(methods) - 1) / 2) * width
        plt.bar([j + offset for j in x], vals, width=width, label=method)
    plt.xticks(x, keys)
    plt.ylabel("metric value")
    plt.title(f"{problem_name}: comparison metrics")
    plt.legend()
    plt.tight_layout()
    suffix = f"_{tag}" if tag else ""
    plt.savefig(out_dir / f"{problem_name}{suffix}_barplot.png", dpi=200)
    plt.close()


def save_table_csv(rows: List[Dict[str, Any]], path: Path):
    if not rows:
        return
    keys = sorted({k for r in rows for k in r.keys()})
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def save_summary_tables(summary: Dict, out_dir: Path):
    full_rows = []
    for method, metrics in summary.get("full_val_metrics", {}).items():
        row = {"method": method}
        row.update(metrics)
        full_rows.append(row)
    save_table_csv(full_rows, out_dir / "full_val_metrics.csv")
    subset_rows = []
    for method, metrics in summary.get("subset_avg", {}).items():
        row = {"method": method}
        row.update(metrics)
        subset_rows.append(row)
    save_table_csv(subset_rows, out_dir / "subset_avg_metrics.csv")
    runtime_rows = []
    for method, metrics in summary.get("runtime", {}).items():
        row = {"method": method}
        row.update(metrics)
        runtime_rows.append(row)
    save_table_csv(runtime_rows, out_dir / "runtime_metrics.csv")


def add_text_page(pdf, title: str, lines: List[str], max_lines=42):
    for start in range(0, max(len(lines), 1), max_lines):
        chunk = lines[start:start + max_lines]
        fig = plt.figure(figsize=(11, 8.5))
        ax = fig.add_subplot(111)
        ax.axis("off")
        ax.text(0.02, 0.97, title if start == 0 else title + " (continued)", fontsize=16, fontweight="bold", va="top", family="monospace")
        y = 0.91
        for line in chunk:
            ax.text(0.02, y, str(line)[:150], fontsize=9, va="top", family="monospace")
            y -= 0.021
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)


def add_image_page(pdf, image_path: Path, title: str):
    if not image_path.exists():
        return
    try:
        img = plt.imread(str(image_path))
    except Exception:
        return
    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_subplot(111)
    ax.imshow(img)
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def save_pdf_report(summary: Dict, out_dir: Path):
    pdf_path = out_dir / "summary_report.pdf"
    with PdfPages(pdf_path) as pdf:
        lines = [
            f"Problem: {summary.get('problem')}",
            f"Run mode: {summary.get('run_mode')}",
            f"Coefficient family: {summary.get('coefficient_family')}",
            f"Target axis: {summary.get('target_axis')}",
            f"Coefficient range: {summary.get('coefficient_range')}",
            "",
            "Train stats:",
        ]
        for k, v in summary.get("train_stats", {}).items():
            lines.append(f"  {k}: {v}")
        add_text_page(pdf, "Experiment overview", lines)

        if "modes" in summary:
            mode_lines = []
            for mode_name, mode_summary in summary.get("modes", {}).items():
                mode_lines.append("=" * 80)
                mode_lines.append(f"MODE: {mode_name}")
                mode_lines.append("=" * 80)
                if "full_val_metrics" in mode_summary:
                    mode_lines.append("Full validation metrics:")
                    for method, metrics in mode_summary.get("full_val_metrics", {}).items():
                        mae = metrics.get("mae", None)
                        rel = metrics.get("rel_l2", None)
                        grad = metrics.get("grad_mae", None)
                        varr = metrics.get("var_ratio", None)
                        mode_lines.append(f"  {method}: MAE={mae}, relL2={rel}, gradMAE={grad}, varRatio={varr}")
                if "subset_avg" in mode_summary:
                    mode_lines.append("Subset averages:")
                    for method, metrics in mode_summary.get("subset_avg", {}).items():
                        mae = metrics.get("mae", None)
                        rel = metrics.get("rel_l2", None)
                        grad = metrics.get("grad_mae", None)
                        varr = metrics.get("var_ratio", None)
                        mode_lines.append(f"  {method}: MAE={mae}, relL2={rel}, gradMAE={grad}, varRatio={varr}")
                mode_lines.append("")
            add_text_page(pdf, "All-modes combined summary", mode_lines)

        full_lines = []
        for method, metrics in summary.get("full_val_metrics", {}).items():
            full_lines.append(method)
            for k, v in metrics.items():
                full_lines.append(f"  {k:<18}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            full_lines.append("")
        add_text_page(pdf, "Full validation metrics", full_lines)

        subset_lines = []
        for method, metrics in summary.get("subset_avg", {}).items():
            subset_lines.append(method)
            for k, v in metrics.items():
                subset_lines.append(f"  {k:<18}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            subset_lines.append("")
        add_text_page(pdf, "Subset metrics", subset_lines)

        runtime_lines = []
        for method, metrics in summary.get("runtime", {}).items():
            runtime_lines.append(method)
            for k, v in metrics.items():
                runtime_lines.append(f"  {k:<25}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            runtime_lines.append("")
        add_text_page(pdf, "Runtime metrics", runtime_lines)

        for png in sorted(out_dir.rglob("*.png")):
            add_image_page(pdf, png, str(png.relative_to(out_dir)))
    return pdf_path


def make_results_zip(out_dir: Path):
    zip_path = out_dir.parent / f"{out_dir.name}.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file in out_dir.rglob("*"):
            if file.is_file() and file != zip_path:
                zf.write(file, arcname=file.relative_to(out_dir.parent))
    return zip_path


# ============================================================
# Experiment runners
# ============================================================
def run_main_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path, tag: str = "main") -> Dict:
    ensure_dir(out_dir)
    train_ds = problem.build_dataset(cfg, cfg.train_size, device, tag=f"train_{tag}")
    val_ds = problem.build_dataset(cfg, cfg.val_size, device, tag=f"val_{tag}")
    stats = compute_train_stats(train_ds)
    print(f"[{problem.name}] train stats: {stats}")

    full_val_metrics: Dict[str, Dict] = {}
    subset_metrics: Dict[str, List[Dict]] = {}
    subset_preds: Dict[str, List[torch.Tensor]] = {}
    histories: Dict[str, Any] = {}
    runtime: Dict[str, Dict] = {}
    best_epochs: Dict[str, int] = {}

    subset_indices = list(range(min(cfg.vc_eval_samples, len(val_ds))))

    for model_key in cfg.run_methods:
        if model_key == "vc_pinn":
            continue
        model_name = nice_name(model_key)
        model = build_model(model_key, cfg, problem, stats)
        model, hist, metrics, best_val, best_epoch, train_time = train_amortized_model(cfg, problem, model, model_name, train_ds, val_ds, stats, device, out_dir, cfg.epochs if model_key == "cabissm" else cfg.baseline_epochs)
        histories[model_name] = hist
        full_val_metrics[model_name] = metrics
        best_epochs[model_name] = best_epoch
        runtime[model_name] = {"train_time_sec": train_time, "train_time_min": train_time / 60.0}
        runtime[model_name].update(benchmark_inference(model, val_ds, cfg, device, n_batches=10))
        save_curves(hist, out_dir, problem.name, model_name)
        m, p = evaluate_subset(model, val_ds, subset_indices, device)
        subset_metrics[model_name] = m
        subset_preds[model_name] = p

    if "vc_pinn" in cfg.run_methods:
        m, p, h = run_vc_pinn_baseline(problem, cfg, val_ds, device, subset_indices)
        subset_metrics["VC-PINN-style"] = m
        subset_preds["VC-PINN-style"] = p
        histories["VC-PINN-style"] = h
        runtime["VC-PINN-style"] = {
            "subset_total_time_sec": sum(x.get("wall_time_sec", 0.0) for x in m),
            "mean_time_per_sample_sec": mean_metric(m, "wall_time_sec"),
        }

    save_reconstruction_plot(problem, val_ds, subset_indices, subset_preds, out_dir, tag=tag)
    save_barplot(problem.name, subset_metrics, out_dir, tag=tag)

    summary = {
        "problem": problem.name,
        "run_mode": cfg.run_mode,
        "tag": tag,
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "train_stats": stats,
        "best_epochs": best_epochs,
        "full_val_metrics": full_val_metrics,
        "subset_avg": {method: {k: mean_metric(metrics, k) for k in metrics[0].keys()} for method, metrics in subset_metrics.items()},
        "runtime": runtime,
    }
    save_json(summary, out_dir / f"{tag}_summary.json")
    save_summary_tables(summary, out_dir)
    return summary


def ablation_cfg(base: CFG, name: str) -> CFG:
    cfg = replace(base)
    cfg.run_methods = ("cabissm",)
    cfg.epochs = base.epochs
    if name == "full_cabissm":
        pass
    elif name == "no_ssm_attention_only":
        cfg.use_ssm = False
    elif name == "no_cross_ssm_only":
        cfg.use_cross_attention = False
    elif name == "no_fourier_features":
        cfg.use_fourier_features = False
    elif name == "no_gradient_loss":
        cfg.lambda_grad = 0.0
    else:
        raise ValueError(name)
    return cfg


def run_ablation_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path) -> Dict:
    ensure_dir(out_dir)
    # Use the same dataset across ablations for fairness.
    train_ds = problem.build_dataset(cfg, cfg.train_size, device, tag="train_ablation_shared")
    val_ds = problem.build_dataset(cfg, cfg.val_size, device, tag="val_ablation_shared")
    base_stats = compute_train_stats(train_ds)
    subset_indices = list(range(min(cfg.vc_eval_samples, len(val_ds))))
    all_metrics, all_runtime, all_histories, subset_metrics, subset_preds = {}, {}, {}, {}, {}
    for abl in cfg.ablation_names:
        print("\n" + "=" * 100)
        print(f"Ablation: {abl}")
        print("=" * 100)
        acfg = ablation_cfg(cfg, abl)
        model_name = f"CABiSSM_{abl}"
        model = CABiSSMInverse(acfg, problem, base_stats["u_mean"], base_stats["u_std"])
        model, hist, metrics, best_val, best_epoch, train_time = train_amortized_model(acfg, problem, model, model_name, train_ds, val_ds, base_stats, device, out_dir, acfg.epochs)
        all_metrics[model_name] = metrics
        all_runtime[model_name] = {"train_time_sec": train_time, "train_time_min": train_time / 60.0}
        all_runtime[model_name].update(benchmark_inference(model, val_ds, acfg, device, n_batches=10))
        all_histories[model_name] = hist
        save_curves(hist, out_dir, problem.name, model_name)
        m, p = evaluate_subset(model, val_ds, subset_indices, device)
        subset_metrics[model_name] = m
        subset_preds[model_name] = p
    save_reconstruction_plot(problem, val_ds, subset_indices, subset_preds, out_dir, tag="ablation")
    save_barplot(problem.name, subset_metrics, out_dir, tag="ablation")
    summary = {
        "problem": problem.name,
        "run_mode": "ablation",
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "train_stats": base_stats,
        "full_val_metrics": all_metrics,
        "subset_avg": {method: {k: mean_metric(metrics, k) for k in metrics[0].keys()} for method, metrics in subset_metrics.items()},
        "runtime": all_runtime,
    }
    save_json(summary, out_dir / "ablation_summary.json")
    save_summary_tables(summary, out_dir)
    return summary


def run_sweep_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path, sweep_type: str) -> Dict:
    ensure_dir(out_dir)
    values = cfg.sensor_sweep_values if sweep_type == "sensor" else cfg.noise_sweep_values
    rows = []
    all_summaries = []
    for value in values:
        if sweep_type == "sensor":
            scfg = replace(cfg, n_sensors=int(value), train_size=cfg.sweep_train_size, val_size=cfg.sweep_val_size, epochs=cfg.sweep_epochs, baseline_epochs=cfg.sweep_epochs, run_methods=("cabissm", "deeponet", "fnoenc"))
            tag = f"sensor_{int(value)}"
        else:
            scfg = replace(cfg, sensor_noise_std=float(value), train_size=cfg.sweep_train_size, val_size=cfg.sweep_val_size, epochs=cfg.sweep_epochs, baseline_epochs=cfg.sweep_epochs, run_methods=("cabissm", "deeponet", "fnoenc"))
            tag = f"noise_{float(value):.3f}".replace(".", "p")
        print("\n" + "=" * 100)
        print(f"Sweep run: {tag}")
        print("=" * 100)
        run_dir = out_dir / tag
        summary = run_main_experiment(scfg, problem, device, run_dir, tag=tag)
        all_summaries.append(summary)
        for method, metrics in summary.get("full_val_metrics", {}).items():
            row = {"sweep_type": sweep_type, "sweep_value": value, "method": method}
            row.update(metrics)
            rows.append(row)
    save_table_csv(rows, out_dir / f"{sweep_type}_sweep_metrics.csv")
    # plot MAE vs sweep value
    methods = sorted(set(r["method"] for r in rows))
    plt.figure(figsize=(8, 5))
    for method in methods:
        xs = [r["sweep_value"] for r in rows if r["method"] == method]
        ys = [r["mae"] for r in rows if r["method"] == method]
        plt.plot(xs, ys, marker="o", label=method)
    plt.xlabel("number of sensors" if sweep_type == "sensor" else "sensor noise std")
    plt.ylabel("MAE")
    plt.title(f"{problem.name}: {sweep_type} sweep")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{sweep_type}_sweep_mae.png", dpi=200)
    plt.close()
    summary = {
        "problem": problem.name,
        "run_mode": f"{sweep_type}_sweep",
        "coefficient_family": cfg.coefficient_family,
        "config": asdict(cfg),
        "rows": rows,
        "subruns": all_summaries,
    }
    save_json(summary, out_dir / f"{sweep_type}_sweep_summary.json")
    return summary


def run_all_modes_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path) -> Dict:
    """
    Runs all journal-level modes sequentially for one selected equation:
      1) main: CABiSSM + DeepONet + FNOEnc + VC-PINN-style
      2) ablation: architecture/loss ablations for CABiSSM
      3) sensor_sweep: sparse-sensor robustness
      4) noise_sweep: noisy-observation robustness
    Each mode gets its own subfolder. A combined final_summary.json, PDF, and ZIP are saved at the parent folder.
    """
    ensure_dir(out_dir)
    modes_to_run = ["main", "ablation", "sensor_sweep", "noise_sweep"]
    mode_summaries: Dict[str, Dict] = {}
    for mode in modes_to_run:
        print("\n" + "#" * 120)
        print(f"RUNNING MODE: {mode.upper()} for problem {problem.name}")
        print("#" * 120)
        mcfg = replace(cfg, run_mode=mode)
        mode_dir = out_dir / mode
        if mode == "main":
            mode_summaries[mode] = run_main_experiment(mcfg, problem, device, mode_dir, tag="main")
        elif mode == "ablation":
            mode_summaries[mode] = run_ablation_experiment(mcfg, problem, device, mode_dir)
        elif mode == "sensor_sweep":
            mode_summaries[mode] = run_sweep_experiment(mcfg, problem, device, mode_dir, sweep_type="sensor")
        elif mode == "noise_sweep":
            mode_summaries[mode] = run_sweep_experiment(mcfg, problem, device, mode_dir, sweep_type="noise")
        else:
            raise ValueError(mode)
    combined = {
        "problem": problem.name,
        "run_mode": "all_modes",
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "modes": mode_summaries,
    }
    save_json(combined, out_dir / "all_modes_summary.json")
    return combined


# ============================================================
# Main
# ============================================================
def main():
    cfg = CFG()
    set_seed(cfg.seed)
    torch.set_float32_matmul_precision("high")
    device = get_device()
    print("Device:", device)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    problem = get_problem(cfg.problem_name)
    root_dir = Path(cfg.out_root)
    family_tag = cfg.coefficient_family if problem.name in ["vkdv_paper", "vsg"] else "default"
    out_dir = root_dir / f"{problem.name}_{family_tag}_{cfg.run_mode}"
    ensure_dir(out_dir)
    print("=" * 100)
    print(f"Running problem       : {problem.name}")
    print(f"Run mode              : {cfg.run_mode}")
    print(f"Coefficient family    : {family_tag}")
    print(f"Target axis           : {problem.target_axis}")
    print(f"Coefficient range     : {problem.coeff_min} to {problem.coeff_max}")
    print(f"Output folder         : {out_dir.resolve()}")
    print("=" * 100)
    start_all = time.time()
    if cfg.run_mode == "main":
        summary = run_main_experiment(cfg, problem, device, out_dir, tag="main")
    elif cfg.run_mode == "ablation":
        summary = run_ablation_experiment(cfg, problem, device, out_dir)
    elif cfg.run_mode == "sensor_sweep":
        summary = run_sweep_experiment(cfg, problem, device, out_dir, sweep_type="sensor")
    elif cfg.run_mode == "noise_sweep":
        summary = run_sweep_experiment(cfg, problem, device, out_dir, sweep_type="noise")
    elif cfg.run_mode == "all_modes":
        summary = run_all_modes_experiment(cfg, problem, device, out_dir)
    else:
        raise ValueError(f"Unknown run_mode: {cfg.run_mode}")
    summary["total_wall_time_sec"] = time.time() - start_all
    save_json(summary, out_dir / "final_summary.json")
    pdf_path = None
    zip_path = None
    if cfg.make_pdf_summary:
        pdf_path = save_pdf_report(summary, out_dir)
        print(f"PDF report saved: {pdf_path}")
    if cfg.make_zip:
        zip_path = make_results_zip(out_dir)
        print(f"ZIP saved: {zip_path}")
    print("\nExperiment completed.")
    print("Artifacts saved in:", out_dir.resolve())
    try:
        from IPython.display import FileLink, display
        if pdf_path is not None:
            display(FileLink(str(pdf_path)))
        if zip_path is not None:
            display(FileLink(str(zip_path)))
    except Exception:
        pass


if __name__ == "__main__":
    main()


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Running problem       : vsg
Run mode              : main
Coefficient family    : mixed_random
Target axis           : t
Coefficient range     : 0.5 to 1.5
Output folder         : /kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/vsg_mixed_random_main
[vsg] Building train_main dataset: 16384 samples, 128 simulation batches
  [vsg/train_main] batch 12/128 | elapsed 0.0s
  [vsg/train_main] batch 24/128 | elapsed 0.1s
  [vsg/train_main] batch 36/128 | elapsed 0.1s
  [vsg/train_main] batch 48/128 | elapsed 0.1s
  [vsg/train_main] batch 60/128 | elapsed 0.1s
  [vsg/train_main] batch 72/128 | elapsed 0.1s
  [vsg/train_main] batch 84/128 | elapsed 0.1s
  [vsg/train_main] batch 96/128 | elapsed 0.2s
  [vsg/train_main] batch 108/128 | elapsed 0.2s
  [vsg/train_main] batch 120/128 | elapsed 0.2s
  [vsg/train_main] batch 128/128 | elapsed 0.2s
[vsg] Building val_main dataset: 2048 samples, 16 simulation batches
  [vsg/val_main] 

/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/vsg_mixed_random_main/summary_report.pdf

/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/vsg_mixed_random_main.zip

In [4]:
# ============================================================
# Publishable inverse-PDE pipeline
# CABiSSM vs DeepONet vs FNOEnc vs VC-PINN-style
# Includes: equation selection, baselines, ablations, robustness sweeps,
# runtime analysis, normalized metrics, variance-ratio metrics, PDF + ZIP export.
#
# Recommended Kaggle use:
#   1) Paste this full script into one Kaggle notebook cell.
#   2) Edit ONLY the USER SETTINGS block.
#   3) Run one problem / one experiment mode at a time.
# ============================================================

import csv
import json
import math
import random
import time
import zipfile
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# USER SETTINGS: change this block only
# ============================================================
@dataclass
class CFG:
    # ------------------------------
    # Main experiment selection
    # ------------------------------
    # Available problems:
    #   "wave"         : variable wave speed c(x)
    #   "adr"          : variable diffusion D(x)
    #   "vkdv"         : random time-coefficient KdV-style g(t)
    #   "vkdv_paper"   : VC-PINN-paper-inspired vKdV coefficient families
    #   "vsg"          : variable-coefficient Sine-Gordon h(t)
    problem_name: str = "adr"

    # Run modes:
    #   "main"          : main model + baselines
    #   "ablation"      : CABiSSM ablation study
    #   "sensor_sweep"  : sparse sensor robustness sweep
    #   "noise_sweep"   : observation-noise robustness sweep
    #   "all_modes"     : run main + ablation + sensor_sweep + noise_sweep sequentially for one equation
    run_mode: str = "main"

    # For vkdv_paper and vsg. Options are problem-dependent.
    # vkdv_paper coefficient_family: "linear", "cubic", "cos", "exp_decay", "mixed_random"
    # vsg coefficient_family       : "linear", "quadratic", "cos", "mixed_random"
    coefficient_family: str = "mixed_random"

    # Main methods.
    # For main runs, keep all enabled.
    run_methods: Tuple[str, ...] = ("cabissm", "deeponet", "fnoenc", "vc_pinn")

    # ------------------------------
    # Data sizes
    # ------------------------------
    train_size: int = 16384
    val_size: int = 2048
    n_sensors: int = 128
    sensor_noise_std: float = 0.01
    sim_batch_size: int = 128

    # Grids
    n_x: int = 129
    n_t: int = 301
    n_q: int = 129

    # ------------------------------
    # Amortized model training
    # ------------------------------
    batch_size: int = 64
    epochs: int = 50
    baseline_epochs: int = 50
    lr: float = 3e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    lambda_grad: float = 0.5
    lambda_tv: float = 1e-4
    early_stop_patience: int = 12

    # Runtime
    seed: int = 42
    num_workers: int = 2
    use_amp: bool = True
    amp_dtype: str = "bfloat16"  # good on RTX PRO 6000 / H100; FFT sections force FP32 internally
    use_compile: bool = False
    out_root: str = "/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline"

    # ------------------------------
    # CABiSSM architecture
    # ------------------------------
    d_model: int = 128
    n_heads: int = 8
    n_layers: int = 4
    state_dim: int = 8
    mlp_ratio: int = 4
    dropout: float = 0.0
    use_fourier_features: bool = True
    use_cross_attention: bool = True
    use_ssm: bool = True

    # ------------------------------
    # DeepONet baseline
    # ------------------------------
    deeponet_width: int = 128
    deeponet_p: int = 128

    # ------------------------------
    # FNO-Encoder baseline
    # ------------------------------
    fno_width: int = 48
    fno_modes_t: int = 16
    fno_modes_x: int = 16
    fno_layers: int = 4
    fno_t_bins: int = 129
    fno_x_bins: int = 129

    # ------------------------------
    # VC-PINN-style baseline
    # ------------------------------
    # This is sample-wise, so full validation is expensive.
    vc_eval_samples: int = 16
    vc_hidden: int = 128
    vc_blocks_u: int = 4
    vc_blocks_c: int = 3
    vc_layers_per_block: int = 2
    vc_adam_steps: int = 2000
    vc_lbfgs_steps: int = 100
    vc_lr: float = 1e-3
    vc_colloc_f: int = 4096
    vc_colloc_ic: int = 512
    vc_colloc_bc: int = 512
    vc_w_data: float = 10.0
    vc_w_pde: float = 1.0
    vc_w_ic: float = 10.0
    vc_w_bc: float = 10.0
    vc_w_cbc: float = 10.0
    vc_w_c_smooth: float = 1e-4
    vc_log_every: int = 500

    # ------------------------------
    # Journal-strength extra experiments
    # ------------------------------
    sensor_sweep_values: Tuple[int, ...] = (16, 32, 64, 128)
    noise_sweep_values: Tuple[float, ...] = (0.0, 0.01, 0.03, 0.05)

    # For sweeps, use smaller sizes if you want a quick result.
    sweep_train_size: int = 8192
    sweep_val_size: int = 1024
    sweep_epochs: int = 35

    # Ablation variants.
    # These are trained only when run_mode == "ablation".
    ablation_names: Tuple[str, ...] = (
        "full_cabissm",
        "no_ssm_attention_only",
        "no_cross_ssm_only",
        "no_fourier_features",
        "no_gradient_loss",
    )

    # Artifacts
    make_pdf_summary: bool = True
    make_zip: bool = True
    save_checkpoints: bool = True


# ============================================================
# Utilities
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_amp_dtype(cfg: CFG):
    return torch.bfloat16 if cfg.amp_dtype.lower() == "bfloat16" else torch.float16


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def gradient_1d(field: torch.Tensor) -> torch.Tensor:
    return field[:, 1:, :] - field[:, :-1, :]


def total_variation_1d(field: torch.Tensor) -> torch.Tensor:
    return (field[:, 1:, :] - field[:, :-1, :]).abs().mean()


@torch.no_grad()
def coeff_metrics(c_hat: torch.Tensor, c_true: torch.Tensor) -> Dict[str, float]:
    c_hat = c_hat.float()
    c_true = c_true.float()

    mse = F.mse_loss(c_hat, c_true)
    mae = F.l1_loss(c_hat, c_true)
    rel_l2 = torch.norm(c_hat - c_true) / (torch.norm(c_true) + 1e-8)
    grad_mae = F.l1_loss(gradient_1d(c_hat), gradient_1d(c_true))

    pred_mean = c_hat.mean()
    pred_std = c_hat.std()
    true_mean = c_true.mean()
    true_std = c_true.std()

    norm_mae = mae / (true_std + 1e-8)
    norm_rmse = torch.sqrt(mse) / (true_std + 1e-8)
    var_ratio = pred_std / (true_std + 1e-8)
    mean_bias = pred_mean - true_mean

    return {
        "mse": float(mse.item()),
        "mae": float(mae.item()),
        "rel_l2": float(rel_l2.item()),
        "grad_mae": float(grad_mae.item()),
        "norm_mae": float(norm_mae.item()),
        "norm_rmse": float(norm_rmse.item()),
        "var_ratio": float(var_ratio.item()),
        "mean_bias": float(mean_bias.item()),
        "pred_mean": float(pred_mean.item()),
        "pred_std": float(pred_std.item()),
        "true_mean": float(true_mean.item()),
        "true_std": float(true_std.item()),
    }


def mean_metric(metrics: List[Dict[str, float]], key: str) -> float:
    vals = [m[key] for m in metrics if key in m]
    return sum(vals) / max(len(vals), 1)


def save_json(obj: Dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def move_batch(batch: Dict[str, torch.Tensor], device: torch.device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# Dataset
# ============================================================
class InversePDEDataset(Dataset):
    def __init__(self, sensors: torch.Tensor, q_grid: torch.Tensor, coeff_true: torch.Tensor):
        self.sensors = sensors.float()
        self.q_grid = q_grid.float()
        self.coeff_true = coeff_true.float()

    def __len__(self):
        return self.sensors.shape[0]

    def __getitem__(self, idx):
        return {
            "sensors": self.sensors[idx],
            "q_grid": self.q_grid[idx],
            "coeff_true": self.coeff_true[idx],
        }


def make_loader(ds: Dataset, cfg: CFG, shuffle: bool):
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(cfg.num_workers > 0),
    )


# ============================================================
# PDE problem classes
# ============================================================
class BaseProblem:
    name = "base"
    target_axis = "x"  # "x" or "t"
    coeff_min = 0.0
    coeff_max = 1.0
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0

    def make_grids(self, cfg: CFG, device: torch.device):
        x = torch.linspace(self.x_min, self.x_max, cfg.n_x, device=device)
        t = torch.linspace(self.t_min, self.t_max, cfg.n_t, device=device)
        if self.target_axis == "x":
            q = torch.linspace(self.x_min, self.x_max, cfg.n_q, device=device)
        else:
            q = torch.linspace(self.t_min, self.t_max, cfg.n_q, device=device)
        return x, t, q

    def generate_coeff(self, cfg: CFG, batch_size: int, q: torch.Tensor, device: torch.device):
        raise NotImplementedError

    def solve(self, cfg: CFG, coeff: torch.Tensor, x: torch.Tensor, t: torch.Tensor, q: torch.Tensor):
        raise NotImplementedError

    def sample_sensors(self, cfg: CFG, U: torch.Tensor, x: torch.Tensor, t: torch.Tensor):
        B = U.shape[0]
        Ns = cfg.n_sensors
        device = U.device
        ix = torch.randint(0, len(x), (B, Ns), device=device)
        it = torch.randint(0, len(t), (B, Ns), device=device)
        b = torch.arange(B, device=device).unsqueeze(1).expand(B, Ns)
        u = U[b, it, ix]
        if cfg.sensor_noise_std > 0:
            u = u + cfg.sensor_noise_std * torch.randn_like(u)
        return torch.stack([x[ix], t[it], u], dim=-1)

    @torch.no_grad()
    def build_dataset(self, cfg: CFG, size: int, device: torch.device, tag: str):
        x, t, q = self.make_grids(cfg, device)
        sensors_all, coeff_all, q_all = [], [], []
        num_batches = math.ceil(size / cfg.sim_batch_size)
        print(f"[{self.name}] Building {tag} dataset: {size} samples, {num_batches} simulation batches")
        start = time.time()
        for bi in range(num_batches):
            bs = min(cfg.sim_batch_size, size - bi * cfg.sim_batch_size)
            coeff = self.generate_coeff(cfg, bs, q, device)
            U = self.solve(cfg, coeff, x, t, q)
            sensors = self.sample_sensors(cfg, U, x, t)
            q_grid = q.view(1, -1, 1).repeat(bs, 1, 1)
            sensors_all.append(sensors.cpu())
            coeff_all.append(coeff.unsqueeze(-1).cpu())
            q_all.append(q_grid.cpu())
            if (bi + 1) % max(1, num_batches // 10) == 0 or (bi + 1) == num_batches:
                print(f"  [{self.name}/{tag}] batch {bi + 1}/{num_batches} | elapsed {time.time() - start:.1f}s")
        return InversePDEDataset(torch.cat(sensors_all), torch.cat(q_all), torch.cat(coeff_all))

    def coeff_input(self, x_col: torch.Tensor, t_col: torch.Tensor) -> torch.Tensor:
        return x_col if self.target_axis == "x" else t_col

    def pde_residual(self, model, x_col: torch.Tensor, t_col: torch.Tensor):
        raise NotImplementedError

    def aux_losses(self, model, sample: Dict[str, torch.Tensor], cfg: CFG, device: torch.device):
        return {}


class WaveProblem(BaseProblem):
    name = "wave"
    target_axis = "x"
    coeff_min = 0.75
    coeff_max = 1.45
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    beta_ic = 3

    def initial_condition(self, x):
        return torch.sin(math.pi * x) + 0.5 * torch.sin(self.beta_ic * math.pi * x)

    def smooth(self, field, kernel_size=9):
        pad = kernel_size // 2
        z = field.unsqueeze(1)
        z = F.pad(z, (pad, pad), mode="replicate")
        z = F.avg_pool1d(z, kernel_size=kernel_size, stride=1)
        return z.squeeze(1)

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            mode = random.choice(["sin", "gauss", "piecewise"])
            base = random.uniform(0.95, 1.15)
            xx = q_cpu.clone()
            if mode == "sin":
                c = base
                c = c + random.uniform(0.08, 0.20) * torch.sin(2 * math.pi * xx + random.uniform(0, 2 * math.pi))
                c = c + random.uniform(0.03, 0.10) * torch.sin(4 * math.pi * xx + random.uniform(0, 2 * math.pi))
            elif mode == "gauss":
                c = torch.full_like(xx, base)
                for _ in range(random.randint(1, 3)):
                    amp = random.uniform(-0.18, 0.18)
                    ctr = random.uniform(0.1, 0.9)
                    wid = random.uniform(0.03, 0.12)
                    c = c + amp * torch.exp(-0.5 * ((xx - ctr) / wid) ** 2)
            else:
                n_segments = random.randint(3, 6)
                edges = sorted(random.sample(range(8, cfg.n_q - 8), n_segments - 1))
                edges = [0] + edges + [cfg.n_q]
                c = torch.empty_like(xx)
                cur = base
                for s in range(len(edges) - 1):
                    cur = max(self.coeff_min, min(self.coeff_max, cur + random.uniform(-0.18, 0.18)))
                    c[edges[s]:edges[s + 1]] = cur
            fields.append(c)
        c = torch.stack(fields).to(device)
        c = self.smooth(c)
        return c.clamp(self.coeff_min, self.coeff_max)

    def div_operator(self, u, a_half, dx):
        out = torch.zeros_like(u)
        out[:, 1:-1] = (
            a_half[:, 1:] * (u[:, 2:] - u[:, 1:-1])
            - a_half[:, :-1] * (u[:, 1:-1] - u[:, :-2])
        ) / (dx * dx)
        return out

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        coeff_x = coeff if coeff.shape[1] == len(x) else F.interpolate(coeff.unsqueeze(1), size=len(x), mode="linear", align_corners=True).squeeze(1)
        B, Nx = coeff_x.shape
        Nt = len(t)
        dx = float((self.x_max - self.x_min) / (Nx - 1))
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        if dt > 0.95 * dx / self.coeff_max:
            print("Warning: wave CFL may be high. Increase n_t or reduce coeff_max.")
        a = coeff_x * coeff_x
        a_half = 0.5 * (a[:, :-1] + a[:, 1:])
        u0 = self.initial_condition(x).unsqueeze(0).repeat(B, 1)
        U = torch.zeros(B, Nt, Nx, device=x.device)
        U[:, 0] = u0
        Lu0 = self.div_operator(u0, a_half, dx)
        u1 = u0.clone()
        u1[:, 1:-1] = u0[:, 1:-1] + 0.5 * dt * dt * Lu0[:, 1:-1]
        u1[:, 0] = 0.0
        u1[:, -1] = 0.0
        U[:, 1] = u1
        up, uc = u0, u1
        for n in range(1, Nt - 1):
            Lu = self.div_operator(uc, a_half, dx)
            un = 2 * uc - up + dt * dt * Lu
            un[:, 0] = 0.0
            un[:, -1] = 0.0
            U[:, n + 1] = un
            up, uc = uc, un
        return U

    def pde_residual(self, model, x_col, t_col):
        u = model.u(x_col, t_col)
        c = model.coeff(x_col)
        a = c * c
        u_t = autograd_grad(u, t_col)
        u_tt = autograd_grad(u_t, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        a_x = autograd_grad(a, x_col)
        return u_tt - (a_x * u_x + a * u_xx)

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device).requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        u_ic = model.u(x_ic, t_ic)
        u_true = self.initial_condition(x_ic.squeeze(-1)).unsqueeze(-1)
        u_t_ic = autograd_grad(u_ic, t_ic)
        losses["ic"] = F.mse_loss(u_ic, u_true) + F.mse_loss(u_t_ic, torch.zeros_like(u_t_ic))
        t_bc = torch.rand(cfg.vc_colloc_bc, 1, device=device).requires_grad_(True)
        x0 = torch.zeros_like(t_bc).requires_grad_(True)
        x1 = torch.ones_like(t_bc).requires_grad_(True)
        losses["bc"] = F.mse_loss(model.u(x0, t_bc), torch.zeros_like(t_bc)) + F.mse_loss(model.u(x1, t_bc), torch.zeros_like(t_bc))
        return losses


class ADRProblem(BaseProblem):
    name = "adr"
    target_axis = "x"
    coeff_min = 0.0015
    coeff_max = 0.0060
    x_min = 0.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    v = 0.4
    lam = 1.0

    def initial_condition(self, x):
        return torch.sin(math.pi * x)

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            base = random.uniform(0.003, 0.0045)
            D = base + random.uniform(0.0004, 0.0012) * torch.sin(2 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            mode = random.choice(["smooth", "bumps", "piecewise"])
            if mode == "bumps":
                for _ in range(random.randint(1, 3)):
                    ctr = random.uniform(0.15, 0.85)
                    wid = random.uniform(0.035, 0.15)
                    amp = random.uniform(-0.0012, 0.0012)
                    D = D + amp * torch.exp(-0.5 * ((q_cpu - ctr) / wid) ** 2)
            elif mode == "piecewise":
                jump = torch.zeros_like(q_cpu)
                ctr = random.uniform(0.25, 0.75)
                jump[q_cpu > ctr] = random.uniform(-0.001, 0.001)
                D = D + jump
            fields.append(D)
        D = torch.stack(fields).to(device)
        D = F.avg_pool1d(F.pad(D.unsqueeze(1), (4, 4), mode="replicate"), kernel_size=9, stride=1).squeeze(1)
        return D.clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        D = coeff if coeff.shape[1] == len(x) else F.interpolate(coeff.unsqueeze(1), size=len(x), mode="linear", align_corners=True).squeeze(1)
        B, Nx = D.shape
        Nt = len(t)
        dx = float((self.x_max - self.x_min) / (Nx - 1))
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        D_half = 0.5 * (D[:, :-1] + D[:, 1:])
        U = torch.zeros(B, Nt, Nx, device=x.device)
        U[:, 0] = self.initial_condition(x).unsqueeze(0).repeat(B, 1)
        for n in range(Nt - 1):
            u = U[:, n]
            diff = torch.zeros_like(u)
            diff[:, 1:-1] = (
                D_half[:, 1:] * (u[:, 2:] - u[:, 1:-1])
                - D_half[:, :-1] * (u[:, 1:-1] - u[:, :-2])
            ) / (dx * dx)
            ux_up = torch.zeros_like(u)
            ux_up[:, 1:] = (u[:, 1:] - u[:, :-1]) / dx
            reaction = self.lam * u * (1.0 - u)
            un = u + dt * (diff - self.v * ux_up + reaction)
            un[:, 0] = 0.0
            un[:, -1] = 0.0
            U[:, n + 1] = un.clamp(-3.0, 3.0)
        return U

    def pde_residual(self, model, x_col, t_col):
        u = model.u(x_col, t_col)
        D = model.coeff(x_col)
        u_t = autograd_grad(u, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        D_x = autograd_grad(D, x_col)
        return u_t - (D_x * u_x + D * u_xx - self.v * u_x + self.lam * u * (1.0 - u))

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device).requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_condition(x_ic.squeeze(-1)).unsqueeze(-1))
        t_bc = torch.rand(cfg.vc_colloc_bc, 1, device=device).requires_grad_(True)
        x0 = torch.zeros_like(t_bc).requires_grad_(True)
        x1 = torch.ones_like(t_bc).requires_grad_(True)
        losses["bc"] = F.mse_loss(model.u(x0, t_bc), torch.zeros_like(t_bc)) + F.mse_loss(model.u(x1, t_bc), torch.zeros_like(t_bc))
        return losses


class VKdVProblem(BaseProblem):
    name = "vkdv"
    target_axis = "t"
    coeff_min = 0.50
    coeff_max = 1.50
    x_min = -1.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    kappa = 0.75
    x0 = -0.55

    def sech2(self, z):
        return 1.0 / torch.cosh(z).pow(2)

    def initial_profile(self, x):
        return 2.0 * self.kappa * self.kappa * self.sech2(self.kappa * (x - self.x0))

    def generate_coeff(self, cfg, batch_size, q, device):
        fields = []
        q_cpu = q.detach().cpu()
        for _ in range(batch_size):
            base = random.uniform(0.85, 1.15)
            g = base
            g = g + random.uniform(0.08, 0.22) * torch.sin(2 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            g = g + random.uniform(0.03, 0.12) * torch.sin(4 * math.pi * q_cpu + random.uniform(0, 2 * math.pi))
            if random.random() < 0.5:
                g = g * torch.exp(-random.uniform(0.0, 0.4) * q_cpu)
            fields.append(g)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        g_t = coeff if coeff.shape[1] == len(t) else F.interpolate(coeff.unsqueeze(1), size=len(t), mode="linear", align_corners=True).squeeze(1)
        B, Nt = g_t.shape
        Nx = len(x)
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        tau = torch.zeros_like(g_t)
        tau[:, 1:] = torch.cumsum(0.5 * (g_t[:, 1:] + g_t[:, :-1]) * dt, dim=1)
        X = x.view(1, 1, Nx)
        Tau = tau.view(B, Nt, 1)
        center = self.x0 + 4.0 * self.kappa * self.kappa * Tau
        U = 2.0 * self.kappa * self.kappa * self.sech2(self.kappa * (X - center))
        return U

    def pde_residual(self, model, x_col, t_col):
        # u_t + 6 g(t) u u_x + g(t) u_xxx = 0
        u = model.u(x_col, t_col)
        g = model.coeff(t_col)
        u_t = autograd_grad(u, t_col)
        u_x = autograd_grad(u, x_col)
        u_xx = autograd_grad(u_x, x_col)
        u_xxx = autograd_grad(u_xx, x_col)
        return u_t + 6.0 * g * u * u_x + g * u_xxx

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device) * (self.x_max - self.x_min) + self.x_min
        x_ic.requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_profile(x_ic).detach())
        return losses


class VKdVPaperProblem(VKdVProblem):
    name = "vkdv_paper"
    # Same PDE form as VKdVProblem in this code, but coefficients follow fixed families
    # inspired by VC-PINN vKdV experiments: linear, cubic, cosine, decaying oscillation.

    def generate_coeff(self, cfg, batch_size, q, device):
        t_cpu = q.detach().cpu()
        fields = []
        for _ in range(batch_size):
            fam = cfg.coefficient_family
            if fam == "mixed_random":
                fam = random.choice(["linear", "cubic", "cos", "exp_decay"])
            if fam == "linear":
                # scaled linear, with small random slope/offset
                a = random.uniform(0.25, 0.55)
                b = random.uniform(0.75, 1.05)
                g = b + a * t_cpu
            elif fam == "cubic":
                a = random.uniform(0.20, 0.50)
                b = random.uniform(0.80, 1.05)
                g = b + a * (t_cpu ** 3)
            elif fam == "cos":
                base = random.uniform(0.95, 1.10)
                amp = random.uniform(0.15, 0.35)
                phase = random.uniform(0, 2 * math.pi)
                g = base + amp * torch.cos(2 * math.pi * t_cpu + phase)
            elif fam == "exp_decay":
                base = random.uniform(0.90, 1.15)
                amp = random.uniform(0.15, 0.35)
                decay = random.uniform(0.8, 1.8)
                freq = random.uniform(1.0, 2.5)
                phase = random.uniform(0, 2 * math.pi)
                g = base + amp * torch.exp(-decay * t_cpu) * torch.cos(2 * math.pi * freq * t_cpu + phase)
            else:
                raise ValueError(f"Unknown vkdv_paper coefficient_family: {cfg.coefficient_family}")
            fields.append(g)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)


class VSGProblem(BaseProblem):
    name = "vsg"
    target_axis = "t"
    coeff_min = 0.50
    coeff_max = 1.50
    x_min = -1.0
    x_max = 1.0
    t_min = 0.0
    t_max = 1.0
    k = 1.0

    def generate_coeff(self, cfg, batch_size, q, device):
        t_cpu = q.detach().cpu()
        fields = []
        for _ in range(batch_size):
            fam = cfg.coefficient_family
            if fam == "mixed_random":
                fam = random.choice(["linear", "quadratic", "cos"])
            if fam == "linear":
                h = random.uniform(0.7, 1.0) + random.uniform(0.2, 0.5) * t_cpu
            elif fam == "quadratic":
                h = random.uniform(0.7, 1.0) + random.uniform(0.2, 0.5) * (t_cpu ** 2)
            elif fam == "cos":
                h = random.uniform(0.95, 1.10) + random.uniform(0.15, 0.35) * torch.cos(2 * math.pi * t_cpu + random.uniform(0, 2 * math.pi))
            else:
                raise ValueError(f"Unknown vsg coefficient_family: {cfg.coefficient_family}")
            fields.append(h)
        return torch.stack(fields).to(device).clamp(self.coeff_min, self.coeff_max)

    @torch.no_grad()
    def solve(self, cfg, coeff, x, t, q):
        h_t = coeff if coeff.shape[1] == len(t) else F.interpolate(coeff.unsqueeze(1), size=len(t), mode="linear", align_corners=True).squeeze(1)
        B, Nt = h_t.shape
        Nx = len(x)
        dt = float((self.t_max - self.t_min) / (Nt - 1))
        omega = torch.zeros_like(h_t)
        omega[:, 1:] = torch.cumsum(0.5 * (h_t[:, 1:] + h_t[:, :-1]) * dt / self.k, dim=1)
        X = x.view(1, 1, Nx)
        Om = omega.view(B, Nt, 1)
        U = 4.0 * torch.atan(torch.exp(self.k * X - Om))
        return U

    def initial_profile(self, x):
        return 4.0 * torch.atan(torch.exp(self.k * x))

    def pde_residual(self, model, x_col, t_col):
        # u_xt + h(t) sin(u) = 0
        u = model.u(x_col, t_col)
        h = model.coeff(t_col)
        u_x = autograd_grad(u, x_col)
        u_xt = autograd_grad(u_x, t_col)
        return u_xt + h * torch.sin(u)

    def aux_losses(self, model, sample, cfg, device):
        losses = {}
        x_ic = torch.rand(cfg.vc_colloc_ic, 1, device=device) * (self.x_max - self.x_min) + self.x_min
        x_ic.requires_grad_(True)
        t_ic = torch.zeros_like(x_ic).requires_grad_(True)
        losses["ic"] = F.mse_loss(model.u(x_ic, t_ic), self.initial_profile(x_ic).detach())
        return losses


def get_problem(name: str) -> BaseProblem:
    if name == "wave":
        return WaveProblem()
    if name == "adr":
        return ADRProblem()
    if name == "vkdv":
        return VKdVProblem()
    if name == "vkdv_paper":
        return VKdVPaperProblem()
    if name == "vsg":
        return VSGProblem()
    raise ValueError(f"Unknown problem_name: {name}")


# ============================================================
# Model components
# ============================================================
class FourierFeatures(nn.Module):
    def __init__(self, in_dim: int, num_bands: int = 8, scale: float = 4.0, enabled: bool = True):
        super().__init__()
        self.in_dim = in_dim
        self.num_bands = num_bands
        self.enabled = enabled
        if enabled:
            freqs = torch.linspace(1.0, num_bands, num_bands) * scale
            self.register_buffer("freqs", freqs)
        else:
            self.register_buffer("freqs", torch.empty(0))

    @property
    def out_dim(self):
        if not self.enabled:
            return self.in_dim
        return self.in_dim + 2 * self.in_dim * self.num_bands

    def forward(self, x):
        if not self.enabled:
            return x
        outs = [x]
        for i in range(self.in_dim):
            xi = x[..., i:i + 1]
            w = self.freqs.view(*([1] * (x.ndim - 1)), -1)
            outs.append(torch.sin(2 * math.pi * xi * w))
            outs.append(torch.cos(2 * math.pi * xi * w))
        return torch.cat(outs, dim=-1)


class FeedForward(nn.Module):
    def __init__(self, d_model, mlp_ratio=4, dropout=0.0):
        super().__init__()
        hidden = d_model * mlp_ratio
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class SensorEncoder(nn.Module):
    def __init__(self, cfg: CFG, u_mean: float, u_std: float):
        super().__init__()
        ff_on = cfg.use_fourier_features
        self.coord_ff = FourierFeatures(2, num_bands=8, scale=4.0, enabled=ff_on)
        self.val_ff = FourierFeatures(1, num_bands=4, scale=4.0, enabled=ff_on)
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        in_dim = self.coord_ff.out_dim + self.val_ff.out_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, sensors):
        coords = sensors[..., :2]
        vals = (sensors[..., 2:3] - self.u_mean) / self.u_std
        z = torch.cat([self.coord_ff(coords), self.val_ff(vals)], dim=-1)
        return self.norm(self.net(z))


class QueryEmbedder(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.ff = FourierFeatures(1, num_bands=16, scale=4.0, enabled=cfg.use_fourier_features)
        self.net = nn.Sequential(
            nn.Linear(self.ff.out_dim, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, q):
        return self.norm(self.net(self.ff(q)))


class CrossAttentionBlock(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.norm_q = nn.LayerNorm(cfg.d_model)
        self.norm_kv = nn.LayerNorm(cfg.d_model)
        self.attn = nn.MultiheadAttention(cfg.d_model, cfg.n_heads, batch_first=True, dropout=cfg.dropout)
        self.norm_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)

    def forward(self, q, kv):
        qn = self.norm_q(q)
        kvn = self.norm_kv(kv)
        attn_out, _ = self.attn(qn, kvn, kvn, need_weights=False)
        x = q + attn_out
        x = x + self.ffn(self.norm_ffn(x))
        return x


class NoCrossConditioning(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )

    def forward(self, h, sensor_tokens):
        ctx = sensor_tokens.mean(dim=1, keepdim=True)
        return h + self.proj(ctx)


class DiagonalSelectiveSSM(nn.Module):
    """
    Lightweight selective diagonal SSM, not full Mamba.
    The update is input-dependent through dt and gate:
        h_k = alpha_k h_{k-1} + (1-alpha_k) B u_k
        y_k = C h_k + D u_k
    """
    def __init__(self, d_model, state_dim, dropout=0.0):
        super().__init__()
        self.d_model = d_model
        self.state_dim = state_dim
        self.in_proj = nn.Linear(d_model, d_model)
        self.gate_proj = nn.Linear(d_model, d_model)
        self.dt_proj = nn.Linear(d_model, d_model * state_dim)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.A_log = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.B = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.C = nn.Parameter(torch.randn(d_model, state_dim) * 0.02)
        self.D = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        Bsz, L, Dm = x.shape
        u = self.in_proj(x)
        gate = torch.sigmoid(self.gate_proj(x))
        dt = F.softplus(self.dt_proj(x)).view(Bsz, L, Dm, self.state_dim) + 1e-4
        A = F.softplus(self.A_log).unsqueeze(0).unsqueeze(0)
        alpha = torch.exp(-dt * A)
        Bp = self.B.unsqueeze(0)
        Cp = self.C.unsqueeze(0)
        Dp = self.D.unsqueeze(0)
        h = torch.zeros(Bsz, Dm, self.state_dim, device=x.device, dtype=x.dtype)
        ys = []
        for k in range(L):
            uk = u[:, k, :].unsqueeze(-1)
            h = alpha[:, k] * h + (1.0 - alpha[:, k]) * (uk * Bp)
            yk = (h * Cp).sum(dim=-1) + Dp * u[:, k, :]
            ys.append(yk)
        y = torch.stack(ys, dim=1)
        y = gate * y
        return self.out_proj(self.dropout(y))


class BiSSMBlock(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.norm_f = nn.LayerNorm(cfg.d_model)
        self.norm_b = nn.LayerNorm(cfg.d_model)
        self.fwd = DiagonalSelectiveSSM(cfg.d_model, cfg.state_dim, cfg.dropout)
        self.bwd = DiagonalSelectiveSSM(cfg.d_model, cfg.state_dim, cfg.dropout)
        self.mix = nn.Linear(2 * cfg.d_model, cfg.d_model)
        self.norm_ffn = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)

    def forward(self, x):
        xf = self.fwd(self.norm_f(x))
        xb = torch.flip(self.bwd(torch.flip(self.norm_b(x), dims=[1])), dims=[1])
        x = x + self.mix(torch.cat([xf, xb], dim=-1))
        x = x + self.ffn(self.norm_ffn(x))
        return x


class CABiSSMInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.cfg = cfg
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.sensor_encoder = SensorEncoder(cfg, u_mean, u_std)
        self.query_embedder = QueryEmbedder(cfg)
        self.cross_blocks = nn.ModuleList()
        self.nocross_blocks = nn.ModuleList()
        self.ssm_blocks = nn.ModuleList()
        self.ffn_blocks = nn.ModuleList()
        for _ in range(cfg.n_layers):
            self.cross_blocks.append(CrossAttentionBlock(cfg))
            self.nocross_blocks.append(NoCrossConditioning(cfg))
            self.ssm_blocks.append(BiSSMBlock(cfg))
            self.ffn_blocks.append(nn.Sequential(nn.LayerNorm(cfg.d_model), FeedForward(cfg.d_model, cfg.mlp_ratio, cfg.dropout)))
        self.norm = nn.LayerNorm(cfg.d_model)
        self.decoder = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(), nn.Linear(cfg.d_model, 1))

    def forward(self, sensors, q_grid):
        sensor_tokens = self.sensor_encoder(sensors)
        h = self.query_embedder(q_grid)
        for i in range(self.cfg.n_layers):
            if self.cfg.use_cross_attention:
                h = self.cross_blocks[i](h, sensor_tokens)
            else:
                h = self.nocross_blocks[i](h, sensor_tokens)
            if self.cfg.use_ssm:
                h = self.ssm_blocks[i](h)
            else:
                h = h + self.ffn_blocks[i](h)
        raw = self.decoder(self.norm(h))
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


class DeepONetInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        self.coord_ff = FourierFeatures(2, 8, 4.0, enabled=cfg.use_fourier_features)
        self.val_ff = FourierFeatures(1, 4, 4.0, enabled=cfg.use_fourier_features)
        sensor_in = self.coord_ff.out_dim + self.val_ff.out_dim
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_in, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_width),
        )
        self.branch = nn.Sequential(
            nn.Linear(2 * cfg.deeponet_width, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_p),
        )
        self.trunk_ff = FourierFeatures(1, 16, 4.0, enabled=cfg.use_fourier_features)
        self.trunk = nn.Sequential(
            nn.Linear(self.trunk_ff.out_dim, cfg.deeponet_width), nn.GELU(),
            nn.Linear(cfg.deeponet_width, cfg.deeponet_p),
        )
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, sensors, q_grid):
        vals = (sensors[..., 2:3] - self.u_mean) / self.u_std
        z = torch.cat([self.coord_ff(sensors[..., :2]), self.val_ff(vals)], dim=-1)
        z = self.sensor_mlp(z)
        z = torch.cat([z.mean(dim=1), z.max(dim=1).values], dim=-1)
        b = self.branch(z)
        tr = self.trunk(self.trunk_ff(q_grid))
        raw = (b.unsqueeze(1) * tr).sum(dim=-1, keepdim=True) / math.sqrt(b.shape[-1]) + self.bias
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_t, modes_x):
        super().__init__()
        scale = 1 / max(1, in_channels * out_channels)
        self.modes_t = modes_t
        self.modes_x = modes_x
        self.weights_pos = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes_t, modes_x, dtype=torch.cfloat))
        self.weights_neg = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes_t, modes_x, dtype=torch.cfloat))

    def compl_mul2d(self, x, w):
        return torch.einsum("bixy,ioxy->boxy", x, w)

    def forward(self, x):
        original_dtype = x.dtype
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            B, C, T, X = x.shape
            x_ft = torch.fft.rfft2(x, dim=(-2, -1))
            out_ft = torch.zeros(B, self.weights_pos.shape[1], T, X // 2 + 1, device=x.device, dtype=torch.cfloat)
            mt = min(self.modes_t, T)
            mx = min(self.modes_x, X // 2 + 1)
            out_ft[:, :, :mt, :mx] = self.compl_mul2d(x_ft[:, :, :mt, :mx], self.weights_pos[:, :, :mt, :mx])
            out_ft[:, :, -mt:, :mx] = self.compl_mul2d(x_ft[:, :, -mt:, :mx], self.weights_neg[:, :, :mt, :mx])
            y = torch.fft.irfft2(out_ft, s=(T, X), dim=(-2, -1))
        return y.to(original_dtype)


class FNOBlock(nn.Module):
    def __init__(self, width, modes_t, modes_x):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes_t, modes_x)
        self.point = nn.Conv2d(width, width, 1)
        self.norm = nn.GroupNorm(8, width)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.point(x)))


def sensors_to_grid(cfg: CFG, problem: BaseProblem, sensors, u_mean, u_std):
    device = sensors.device
    dtype = sensors.dtype
    B, Ns, _ = sensors.shape
    T, X = cfg.fno_t_bins, cfg.fno_x_bins
    x = sensors[..., 0]
    t = sensors[..., 1]
    u = (sensors[..., 2] - u_mean.to(device)) / u_std.to(device)
    ix = torch.round((x - problem.x_min) / (problem.x_max - problem.x_min) * (X - 1)).long().clamp(0, X - 1)
    it = torch.round((t - problem.t_min) / (problem.t_max - problem.t_min) * (T - 1)).long().clamp(0, T - 1)
    obs = torch.zeros(B, T, X, device=device, dtype=dtype)
    cnt = torch.zeros(B, T, X, device=device, dtype=dtype)
    b = torch.arange(B, device=device).view(B, 1).expand(B, Ns)
    flat = (b * T * X + it * X + ix).reshape(-1)
    obs.reshape(-1).scatter_add_(0, flat, u.reshape(-1).to(dtype))
    cnt.reshape(-1).scatter_add_(0, flat, torch.ones_like(u, dtype=dtype).reshape(-1))
    mask = (cnt > 0).to(dtype)
    obs = obs / cnt.clamp_min(1.0)
    return torch.stack([obs, mask], dim=1)


class FNOEncoderInverse(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem, u_mean: float, u_std: float):
        super().__init__()
        self.cfg = cfg
        self.problem = problem
        self.coeff_min = problem.coeff_min
        self.coeff_max = problem.coeff_max
        self.register_buffer("u_mean", torch.tensor([u_mean], dtype=torch.float32))
        self.register_buffer("u_std", torch.tensor([u_std], dtype=torch.float32))
        self.lift = nn.Conv2d(4, cfg.fno_width, 1)
        self.blocks = nn.ModuleList([FNOBlock(cfg.fno_width, cfg.fno_modes_t, cfg.fno_modes_x) for _ in range(cfg.fno_layers)])
        self.q_ff = FourierFeatures(1, 16, 4.0, enabled=cfg.use_fourier_features)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fno_width + self.q_ff.out_dim, cfg.fno_width), nn.GELU(),
            nn.Linear(cfg.fno_width, cfg.fno_width), nn.GELU(),
            nn.Linear(cfg.fno_width, 1),
        )

    def forward(self, sensors, q_grid):
        B = sensors.shape[0]
        T, X = self.cfg.fno_t_bins, self.cfg.fno_x_bins
        grid = sensors_to_grid(self.cfg, self.problem, sensors, self.u_mean, self.u_std)
        x_coord = torch.linspace(self.problem.x_min, self.problem.x_max, X, device=sensors.device, dtype=sensors.dtype).view(1, 1, 1, X).expand(B, 1, T, X)
        t_coord = torch.linspace(self.problem.t_min, self.problem.t_max, T, device=sensors.device, dtype=sensors.dtype).view(1, 1, T, 1).expand(B, 1, T, X)
        z = torch.cat([grid, x_coord, t_coord], dim=1)
        z = self.lift(z)
        for blk in self.blocks:
            z = blk(z)
        latent = z.mean(dim=(-2, -1))
        latent = latent.unsqueeze(1).expand(B, q_grid.shape[1], latent.shape[-1])
        raw = self.decoder(torch.cat([latent, self.q_ff(q_grid)], dim=-1))
        return self.coeff_min + (self.coeff_max - self.coeff_min) * torch.sigmoid(raw)


# ============================================================
# Training helpers
# ============================================================
def compute_train_stats(ds: InversePDEDataset):
    u = ds.sensors[..., 2]
    c = ds.coeff_true
    g = gradient_1d(c)
    return {
        "u_mean": float(u.mean().item()),
        "u_std": float(u.std().item() + 1e-6),
        "c_mean": float(c.mean().item()),
        "c_std": float(c.std().item() + 1e-6),
        "g_std": float(g.std().item() + 1e-6),
    }


def supervised_loss(c_hat, c_true, stats, cfg):
    loss_field = F.mse_loss((c_hat - c_true) / stats["c_std"], torch.zeros_like(c_hat))
    if cfg.lambda_grad > 0:
        loss_grad = F.mse_loss((gradient_1d(c_hat) - gradient_1d(c_true)) / stats["g_std"], torch.zeros_like(gradient_1d(c_hat)))
    else:
        loss_grad = torch.tensor(0.0, device=c_hat.device)
    loss_tv = total_variation_1d(c_hat)
    return loss_field + cfg.lambda_grad * loss_grad + cfg.lambda_tv * loss_tv, {
        "loss_field": float(loss_field.detach().item()),
        "loss_grad": float(loss_grad.detach().item()),
        "loss_tv": float(loss_tv.detach().item()),
    }


def build_model(model_key: str, cfg: CFG, problem: BaseProblem, stats: Dict[str, float]) -> nn.Module:
    if model_key == "cabissm":
        return CABiSSMInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    if model_key == "deeponet":
        return DeepONetInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    if model_key == "fnoenc":
        return FNOEncoderInverse(cfg, problem, stats["u_mean"], stats["u_std"])
    raise ValueError(model_key)


def nice_name(model_key: str) -> str:
    return {"cabissm": "CABiSSM", "deeponet": "DeepONet", "fnoenc": "FNOEnc"}.get(model_key, model_key)


def train_amortized_model(cfg, problem, model, model_name, train_ds, val_ds, stats, device, out_dir, epochs):
    train_loader = make_loader(train_ds, cfg, True)
    val_loader = make_loader(val_ds, cfg, False)
    model = model.to(device)
    if cfg.use_compile and hasattr(torch, "compile"):
        model = torch.compile(model)
    print(f"[{problem.name}] {model_name} parameters: {count_params(model):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    amp_dtype = get_amp_dtype(cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.use_amp and device.type == "cuda" and amp_dtype == torch.float16))
    history = {"train_loss": [], "val_loss": [], "val_mae": [], "val_rel_l2": [], "val_grad_mae": [], "val_norm_mae": [], "val_var_ratio": []}
    best_val = float("inf")
    best_metrics = None
    best_epoch = -1
    best_state = None
    no_improve = 0
    best_path = out_dir / f"{problem.name}_{model_name.lower()}_best.pt"
    start_train = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        train_total, train_count = 0.0, 0
        for batch in train_loader:
            batch = move_batch(batch, device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(cfg.use_amp and device.type == "cuda")):
                pred = model(batch["sensors"], batch["q_grid"])
                loss, _ = supervised_loss(pred, batch["coeff_true"], stats, cfg)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                opt.step()
            bs = batch["sensors"].shape[0]
            train_total += float(loss.detach()) * bs
            train_count += bs
        sched.step()
        model.eval()
        val_total, val_count = 0.0, 0
        metric_total: Dict[str, float] = {}
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch(batch, device)
                with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(cfg.use_amp and device.type == "cuda")):
                    pred = model(batch["sensors"], batch["q_grid"])
                    loss, _ = supervised_loss(pred, batch["coeff_true"], stats, cfg)
                bs = batch["sensors"].shape[0]
                val_total += float(loss.detach()) * bs
                val_count += bs
                m = coeff_metrics(pred.float(), batch["coeff_true"].float())
                for k, v in m.items():
                    metric_total[k] = metric_total.get(k, 0.0) + v * bs
        train_loss = train_total / max(train_count, 1)
        val_loss = val_total / max(val_count, 1)
        metrics = {k: v / max(val_count, 1) for k, v in metric_total.items()}
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(metrics["mae"])
        history["val_rel_l2"].append(metrics["rel_l2"])
        history["val_grad_mae"].append(metrics["grad_mae"])
        history["val_norm_mae"].append(metrics["norm_mae"])
        history["val_var_ratio"].append(metrics["var_ratio"])
        print(
            f"[{problem.name}] {model_name} ep {ep:03d} | train {train_loss:.3e} | val {val_loss:.3e} | "
            f"mae {metrics['mae']:.3e} | relL2 {metrics['rel_l2']:.3e} | grad {metrics['grad_mae']:.3e} | "
            f"normMAE {metrics['norm_mae']:.3e} | varRatio {metrics['var_ratio']:.3f}"
        )
        if val_loss < best_val:
            best_val = val_loss
            best_metrics = metrics
            best_epoch = ep
            no_improve = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if cfg.save_checkpoints:
                torch.save({"model": best_state, "metrics": metrics, "best_val": best_val, "epoch": ep, "history": history, "stats": stats}, best_path)
        else:
            no_improve += 1
        if model_name.startswith("CABiSSM") and no_improve >= cfg.early_stop_patience:
            print(f"[{problem.name}] Early stopping {model_name} at epoch {ep}; best epoch {best_epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    train_time = time.time() - start_train
    return model, history, best_metrics or {}, best_val, best_epoch, train_time


@torch.no_grad()
def evaluate_subset(model, ds, indices, device):
    model.eval()
    metrics, preds = [], []
    for idx in indices:
        s = ds[idx]
        sensors = s["sensors"].unsqueeze(0).to(device)
        q = s["q_grid"].unsqueeze(0).to(device)
        true = s["coeff_true"].unsqueeze(0).to(device)
        pred = model(sensors, q)
        metrics.append(coeff_metrics(pred.float(), true.float()))
        preds.append(pred.squeeze(0).cpu())
    return metrics, preds


@torch.no_grad()
def benchmark_inference(model, ds, cfg, device, n_batches=10):
    loader = make_loader(ds, cfg, False)
    model.eval()
    times = []
    count = 0
    # warmup
    for i, batch in enumerate(loader):
        batch = move_batch(batch, device)
        _ = model(batch["sensors"], batch["q_grid"])
        if i >= 2:
            break
    if device.type == "cuda":
        torch.cuda.synchronize()
    for i, batch in enumerate(loader):
        if i >= n_batches:
            break
        batch = move_batch(batch, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        _ = model(batch["sensors"], batch["q_grid"])
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - start
        times.append(elapsed)
        count += batch["sensors"].shape[0]
    total = sum(times)
    return {"inference_time_sec_total": total, "inference_samples": count, "inference_ms_per_sample": 1000.0 * total / max(count, 1)}


# ============================================================
# VC-PINN-style baseline
# ============================================================
def autograd_grad(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True, retain_graph=True)[0]


class PreActResidualBlock(nn.Module):
    def __init__(self, width, layers_per_block=2):
        super().__init__()
        layers = []
        for _ in range(layers_per_block):
            layers += [nn.Tanh(), nn.Linear(width, width)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.net(x)


class ResNetMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, blocks, layers_per_block):
        super().__init__()
        self.input = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([PreActResidualBlock(hidden, layers_per_block) for _ in range(blocks)])
        self.output = nn.Linear(hidden, out_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.input(x)
        for block in self.blocks:
            h = block(h)
        return self.output(h)


class VCPINNModel(nn.Module):
    def __init__(self, cfg: CFG, problem: BaseProblem):
        super().__init__()
        self.problem = problem
        self.u_net = ResNetMLP(2, cfg.vc_hidden, 1, cfg.vc_blocks_u, cfg.vc_layers_per_block)
        self.c_net = ResNetMLP(1, cfg.vc_hidden, 1, cfg.vc_blocks_c, cfg.vc_layers_per_block)

    def u(self, x, t):
        return self.u_net(torch.cat([x, t], dim=-1))

    def coeff(self, q):
        raw = self.c_net(q)
        return self.problem.coeff_min + (self.problem.coeff_max - self.problem.coeff_min) * torch.sigmoid(raw)


def vc_pinn_loss(problem, model, sample, cfg, device, colloc):
    sensors = sample["sensors"].to(device).float()
    q_grid = sample["q_grid"].to(device).float()
    coeff_true = sample["coeff_true"].to(device).float()
    x_s = sensors[:, 0:1]
    t_s = sensors[:, 1:2]
    u_s = sensors[:, 2:3]
    loss_data = F.mse_loss(model.u(x_s, t_s), u_s)
    x_f = colloc["x_f"].clone().detach().requires_grad_(True)
    t_f = colloc["t_f"].clone().detach().requires_grad_(True)
    loss_pde = torch.mean(problem.pde_residual(model, x_f, t_f) ** 2)
    aux = problem.aux_losses(model, sample, cfg, device)
    loss_ic = aux.get("ic", torch.tensor(0.0, device=device))
    loss_bc = aux.get("bc", torch.tensor(0.0, device=device))
    c0_pred = model.coeff(q_grid[0:1])
    c1_pred = model.coeff(q_grid[-1:])
    loss_cbc = F.mse_loss(c0_pred, coeff_true[0:1]) + F.mse_loss(c1_pred, coeff_true[-1:])
    c_grid = model.coeff(q_grid)
    loss_smooth = total_variation_1d(c_grid.unsqueeze(0))
    loss = (
        cfg.vc_w_data * loss_data + cfg.vc_w_pde * loss_pde + cfg.vc_w_ic * loss_ic +
        cfg.vc_w_bc * loss_bc + cfg.vc_w_cbc * loss_cbc + cfg.vc_w_c_smooth * loss_smooth
    )
    parts = {"data": float(loss_data.detach()), "pde": float(loss_pde.detach()), "ic": float(loss_ic.detach()), "bc": float(loss_bc.detach()), "cbc": float(loss_cbc.detach()), "smooth": float(loss_smooth.detach())}
    return loss, parts


@torch.no_grad()
def eval_vc_coeff(model, sample, device):
    q_grid = sample["q_grid"].to(device).float()
    coeff_true = sample["coeff_true"].to(device).float()
    pred = model.coeff(q_grid)
    return pred.cpu(), coeff_metrics(pred.unsqueeze(0), coeff_true.unsqueeze(0))


def run_vc_pinn_sample(problem, cfg, sample, device):
    model = VCPINNModel(cfg, problem).to(device).float()
    opt = torch.optim.Adam(model.parameters(), lr=cfg.vc_lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.vc_adam_steps)
    colloc = {
        "x_f": torch.rand(cfg.vc_colloc_f, 1, device=device) * (problem.x_max - problem.x_min) + problem.x_min,
        "t_f": torch.rand(cfg.vc_colloc_f, 1, device=device) * (problem.t_max - problem.t_min) + problem.t_min,
    }
    best_state, best_loss = None, float("inf")
    hist = []
    for step in range(1, cfg.vc_adam_steps + 1):
        opt.zero_grad(set_to_none=True)
        loss, parts = vc_pinn_loss(problem, model, sample, cfg, device, colloc)
        loss.backward()
        opt.step()
        sched.step()
        lv = float(loss.detach())
        hist.append(lv)
        if lv < best_loss:
            best_loss = lv
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if step == 1 or step % cfg.vc_log_every == 0 or step == cfg.vc_adam_steps:
            with torch.no_grad():
                _, m = eval_vc_coeff(model, sample, device)
            print(f"    [{problem.name}] VC-PINN {step:04d}/{cfg.vc_adam_steps} | loss {lv:.3e} | data {parts['data']:.2e} | pde {parts['pde']:.2e} | mae {m['mae']:.3e} | relL2 {m['rel_l2']:.3e}")
    if best_state is not None:
        model.load_state_dict(best_state)
    if cfg.vc_lbfgs_steps > 0:
        print(f"    [{problem.name}] VC-PINN L-BFGS refinement: {cfg.vc_lbfgs_steps} iters")
        lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=cfg.vc_lbfgs_steps, max_eval=2 * cfg.vc_lbfgs_steps, history_size=50, line_search_fn="strong_wolfe")
        def closure():
            lbfgs.zero_grad(set_to_none=True)
            loss, _ = vc_pinn_loss(problem, model, sample, cfg, device, colloc)
            loss.backward()
            return loss
        lbfgs.step(closure)
    pred, metrics = eval_vc_coeff(model, sample, device)
    return pred, metrics, hist


def run_vc_pinn_baseline(problem, cfg, val_ds, device, indices):
    metrics, preds, histories = [], [], []
    print(f"[{problem.name}] Running VC-PINN-style on {len(indices)} samples")
    total = time.time()
    for i, idx in enumerate(indices):
        print(f"  [{problem.name}] VC sample {i + 1}/{len(indices)} index={idx}")
        start = time.time()
        pred, m, hist = run_vc_pinn_sample(problem, cfg, val_ds[idx], device)
        m["wall_time_sec"] = time.time() - start
        metrics.append(m)
        preds.append(pred)
        histories.append(hist)
        print(f"  [{problem.name}] done idx={idx} | time {m['wall_time_sec']:.1f}s | MAE {m['mae']:.3e} | relL2 {m['rel_l2']:.3e}")
    print(f"[{problem.name}] VC-PINN total time: {time.time() - total:.1f}s")
    return metrics, preds, histories


# ============================================================
# Plotting and artifact saving
# ============================================================
def save_curves(history, out_dir, problem_name, method_name):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="train loss")
    plt.plot(epochs, history["val_loss"], label="val loss")
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"{problem_name}: {method_name} loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{problem_name}_{method_name}_loss.png", dpi=200)
    plt.close()
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_mae"], label="MAE")
    plt.plot(epochs, history["val_rel_l2"], label="relL2")
    plt.plot(epochs, history["val_grad_mae"], label="gradMAE")
    plt.plot(epochs, history["val_norm_mae"], label="normMAE")
    plt.xlabel("epoch")
    plt.ylabel("metric")
    plt.title(f"{problem_name}: {method_name} validation metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{problem_name}_{method_name}_metrics.png", dpi=200)
    plt.close()


def save_reconstruction_plot(problem, val_ds, indices, method_preds, out_dir, tag=""):
    if not method_preds:
        return
    n = len(indices)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))
    if n == 1:
        axes = axes[None, :]
    for row, idx in enumerate(indices):
        s = val_ds[idx]
        q = s["q_grid"].squeeze(-1)
        true = s["coeff_true"].squeeze(-1)
        sensors = s["sensors"]
        ax0 = axes[row, 0]
        sc = ax0.scatter(sensors[:, 0], sensors[:, 1], c=sensors[:, 2], s=16)
        ax0.set_xlabel("x")
        ax0.set_ylabel("t")
        ax0.set_title(f"{problem.name}: sparse sensors sample {idx}")
        plt.colorbar(sc, ax=ax0, fraction=0.046, pad=0.04)
        ax1 = axes[row, 1]
        ax1.plot(q, true, label="true coefficient", linewidth=2)
        for name, preds in method_preds.items():
            ax1.plot(q, preds[row].squeeze(-1), label=name, linewidth=2)
        ax1.set_xlabel(problem.target_axis)
        ax1.set_ylabel("coefficient")
        ax1.set_title(f"{problem.name}: coefficient reconstruction")
        ax1.legend(fontsize=8)
    plt.tight_layout()
    suffix = f"_{tag}" if tag else ""
    plt.savefig(out_dir / f"{problem.name}{suffix}_reconstructions.png", dpi=200)
    plt.close()


def save_barplot(problem_name, method_metrics, out_dir, tag=""):
    if not method_metrics:
        return
    keys = ["mae", "rel_l2", "grad_mae", "norm_mae", "var_ratio"]
    methods = list(method_metrics.keys())
    x = list(range(len(keys)))
    width = 0.8 / max(len(methods), 1)
    plt.figure(figsize=(12, 5))
    for i, method in enumerate(methods):
        vals = [mean_metric(method_metrics[method], k) for k in keys]
        offset = (i - (len(methods) - 1) / 2) * width
        plt.bar([j + offset for j in x], vals, width=width, label=method)
    plt.xticks(x, keys)
    plt.ylabel("metric value")
    plt.title(f"{problem_name}: comparison metrics")
    plt.legend()
    plt.tight_layout()
    suffix = f"_{tag}" if tag else ""
    plt.savefig(out_dir / f"{problem_name}{suffix}_barplot.png", dpi=200)
    plt.close()


def save_table_csv(rows: List[Dict[str, Any]], path: Path):
    if not rows:
        return
    keys = sorted({k for r in rows for k in r.keys()})
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def save_summary_tables(summary: Dict, out_dir: Path):
    full_rows = []
    for method, metrics in summary.get("full_val_metrics", {}).items():
        row = {"method": method}
        row.update(metrics)
        full_rows.append(row)
    save_table_csv(full_rows, out_dir / "full_val_metrics.csv")
    subset_rows = []
    for method, metrics in summary.get("subset_avg", {}).items():
        row = {"method": method}
        row.update(metrics)
        subset_rows.append(row)
    save_table_csv(subset_rows, out_dir / "subset_avg_metrics.csv")
    runtime_rows = []
    for method, metrics in summary.get("runtime", {}).items():
        row = {"method": method}
        row.update(metrics)
        runtime_rows.append(row)
    save_table_csv(runtime_rows, out_dir / "runtime_metrics.csv")


def add_text_page(pdf, title: str, lines: List[str], max_lines=42):
    for start in range(0, max(len(lines), 1), max_lines):
        chunk = lines[start:start + max_lines]
        fig = plt.figure(figsize=(11, 8.5))
        ax = fig.add_subplot(111)
        ax.axis("off")
        ax.text(0.02, 0.97, title if start == 0 else title + " (continued)", fontsize=16, fontweight="bold", va="top", family="monospace")
        y = 0.91
        for line in chunk:
            ax.text(0.02, y, str(line)[:150], fontsize=9, va="top", family="monospace")
            y -= 0.021
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)


def add_image_page(pdf, image_path: Path, title: str):
    if not image_path.exists():
        return
    try:
        img = plt.imread(str(image_path))
    except Exception:
        return
    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_subplot(111)
    ax.imshow(img)
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def save_pdf_report(summary: Dict, out_dir: Path):
    pdf_path = out_dir / "summary_report.pdf"
    with PdfPages(pdf_path) as pdf:
        lines = [
            f"Problem: {summary.get('problem')}",
            f"Run mode: {summary.get('run_mode')}",
            f"Coefficient family: {summary.get('coefficient_family')}",
            f"Target axis: {summary.get('target_axis')}",
            f"Coefficient range: {summary.get('coefficient_range')}",
            "",
            "Train stats:",
        ]
        for k, v in summary.get("train_stats", {}).items():
            lines.append(f"  {k}: {v}")
        add_text_page(pdf, "Experiment overview", lines)

        if "modes" in summary:
            mode_lines = []
            for mode_name, mode_summary in summary.get("modes", {}).items():
                mode_lines.append("=" * 80)
                mode_lines.append(f"MODE: {mode_name}")
                mode_lines.append("=" * 80)
                if "full_val_metrics" in mode_summary:
                    mode_lines.append("Full validation metrics:")
                    for method, metrics in mode_summary.get("full_val_metrics", {}).items():
                        mae = metrics.get("mae", None)
                        rel = metrics.get("rel_l2", None)
                        grad = metrics.get("grad_mae", None)
                        varr = metrics.get("var_ratio", None)
                        mode_lines.append(f"  {method}: MAE={mae}, relL2={rel}, gradMAE={grad}, varRatio={varr}")
                if "subset_avg" in mode_summary:
                    mode_lines.append("Subset averages:")
                    for method, metrics in mode_summary.get("subset_avg", {}).items():
                        mae = metrics.get("mae", None)
                        rel = metrics.get("rel_l2", None)
                        grad = metrics.get("grad_mae", None)
                        varr = metrics.get("var_ratio", None)
                        mode_lines.append(f"  {method}: MAE={mae}, relL2={rel}, gradMAE={grad}, varRatio={varr}")
                mode_lines.append("")
            add_text_page(pdf, "All-modes combined summary", mode_lines)

        full_lines = []
        for method, metrics in summary.get("full_val_metrics", {}).items():
            full_lines.append(method)
            for k, v in metrics.items():
                full_lines.append(f"  {k:<18}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            full_lines.append("")
        add_text_page(pdf, "Full validation metrics", full_lines)

        subset_lines = []
        for method, metrics in summary.get("subset_avg", {}).items():
            subset_lines.append(method)
            for k, v in metrics.items():
                subset_lines.append(f"  {k:<18}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            subset_lines.append("")
        add_text_page(pdf, "Subset metrics", subset_lines)

        runtime_lines = []
        for method, metrics in summary.get("runtime", {}).items():
            runtime_lines.append(method)
            for k, v in metrics.items():
                runtime_lines.append(f"  {k:<25}: {v:.6e}" if isinstance(v, float) else f"  {k}: {v}")
            runtime_lines.append("")
        add_text_page(pdf, "Runtime metrics", runtime_lines)

        for png in sorted(out_dir.rglob("*.png")):
            add_image_page(pdf, png, str(png.relative_to(out_dir)))
    return pdf_path


def make_results_zip(out_dir: Path):
    zip_path = out_dir.parent / f"{out_dir.name}.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file in out_dir.rglob("*"):
            if file.is_file() and file != zip_path:
                zf.write(file, arcname=file.relative_to(out_dir.parent))
    return zip_path


# ============================================================
# Experiment runners
# ============================================================
def run_main_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path, tag: str = "main") -> Dict:
    ensure_dir(out_dir)
    train_ds = problem.build_dataset(cfg, cfg.train_size, device, tag=f"train_{tag}")
    val_ds = problem.build_dataset(cfg, cfg.val_size, device, tag=f"val_{tag}")
    stats = compute_train_stats(train_ds)
    print(f"[{problem.name}] train stats: {stats}")

    full_val_metrics: Dict[str, Dict] = {}
    subset_metrics: Dict[str, List[Dict]] = {}
    subset_preds: Dict[str, List[torch.Tensor]] = {}
    histories: Dict[str, Any] = {}
    runtime: Dict[str, Dict] = {}
    best_epochs: Dict[str, int] = {}

    subset_indices = list(range(min(cfg.vc_eval_samples, len(val_ds))))

    for model_key in cfg.run_methods:
        if model_key == "vc_pinn":
            continue
        model_name = nice_name(model_key)
        model = build_model(model_key, cfg, problem, stats)
        model, hist, metrics, best_val, best_epoch, train_time = train_amortized_model(cfg, problem, model, model_name, train_ds, val_ds, stats, device, out_dir, cfg.epochs if model_key == "cabissm" else cfg.baseline_epochs)
        histories[model_name] = hist
        full_val_metrics[model_name] = metrics
        best_epochs[model_name] = best_epoch
        runtime[model_name] = {"train_time_sec": train_time, "train_time_min": train_time / 60.0}
        runtime[model_name].update(benchmark_inference(model, val_ds, cfg, device, n_batches=10))
        save_curves(hist, out_dir, problem.name, model_name)
        m, p = evaluate_subset(model, val_ds, subset_indices, device)
        subset_metrics[model_name] = m
        subset_preds[model_name] = p

    if "vc_pinn" in cfg.run_methods:
        m, p, h = run_vc_pinn_baseline(problem, cfg, val_ds, device, subset_indices)
        subset_metrics["VC-PINN-style"] = m
        subset_preds["VC-PINN-style"] = p
        histories["VC-PINN-style"] = h
        runtime["VC-PINN-style"] = {
            "subset_total_time_sec": sum(x.get("wall_time_sec", 0.0) for x in m),
            "mean_time_per_sample_sec": mean_metric(m, "wall_time_sec"),
        }

    save_reconstruction_plot(problem, val_ds, subset_indices, subset_preds, out_dir, tag=tag)
    save_barplot(problem.name, subset_metrics, out_dir, tag=tag)

    summary = {
        "problem": problem.name,
        "run_mode": cfg.run_mode,
        "tag": tag,
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "train_stats": stats,
        "best_epochs": best_epochs,
        "full_val_metrics": full_val_metrics,
        "subset_avg": {method: {k: mean_metric(metrics, k) for k in metrics[0].keys()} for method, metrics in subset_metrics.items()},
        "runtime": runtime,
    }
    save_json(summary, out_dir / f"{tag}_summary.json")
    save_summary_tables(summary, out_dir)
    return summary


def ablation_cfg(base: CFG, name: str) -> CFG:
    cfg = replace(base)
    cfg.run_methods = ("cabissm",)
    cfg.epochs = base.epochs
    if name == "full_cabissm":
        pass
    elif name == "no_ssm_attention_only":
        cfg.use_ssm = False
    elif name == "no_cross_ssm_only":
        cfg.use_cross_attention = False
    elif name == "no_fourier_features":
        cfg.use_fourier_features = False
    elif name == "no_gradient_loss":
        cfg.lambda_grad = 0.0
    else:
        raise ValueError(name)
    return cfg


def run_ablation_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path) -> Dict:
    ensure_dir(out_dir)
    # Use the same dataset across ablations for fairness.
    train_ds = problem.build_dataset(cfg, cfg.train_size, device, tag="train_ablation_shared")
    val_ds = problem.build_dataset(cfg, cfg.val_size, device, tag="val_ablation_shared")
    base_stats = compute_train_stats(train_ds)
    subset_indices = list(range(min(cfg.vc_eval_samples, len(val_ds))))
    all_metrics, all_runtime, all_histories, subset_metrics, subset_preds = {}, {}, {}, {}, {}
    for abl in cfg.ablation_names:
        print("\n" + "=" * 100)
        print(f"Ablation: {abl}")
        print("=" * 100)
        acfg = ablation_cfg(cfg, abl)
        model_name = f"CABiSSM_{abl}"
        model = CABiSSMInverse(acfg, problem, base_stats["u_mean"], base_stats["u_std"])
        model, hist, metrics, best_val, best_epoch, train_time = train_amortized_model(acfg, problem, model, model_name, train_ds, val_ds, base_stats, device, out_dir, acfg.epochs)
        all_metrics[model_name] = metrics
        all_runtime[model_name] = {"train_time_sec": train_time, "train_time_min": train_time / 60.0}
        all_runtime[model_name].update(benchmark_inference(model, val_ds, acfg, device, n_batches=10))
        all_histories[model_name] = hist
        save_curves(hist, out_dir, problem.name, model_name)
        m, p = evaluate_subset(model, val_ds, subset_indices, device)
        subset_metrics[model_name] = m
        subset_preds[model_name] = p
    save_reconstruction_plot(problem, val_ds, subset_indices, subset_preds, out_dir, tag="ablation")
    save_barplot(problem.name, subset_metrics, out_dir, tag="ablation")
    summary = {
        "problem": problem.name,
        "run_mode": "ablation",
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "train_stats": base_stats,
        "full_val_metrics": all_metrics,
        "subset_avg": {method: {k: mean_metric(metrics, k) for k in metrics[0].keys()} for method, metrics in subset_metrics.items()},
        "runtime": all_runtime,
    }
    save_json(summary, out_dir / "ablation_summary.json")
    save_summary_tables(summary, out_dir)
    return summary


def run_sweep_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path, sweep_type: str) -> Dict:
    ensure_dir(out_dir)
    values = cfg.sensor_sweep_values if sweep_type == "sensor" else cfg.noise_sweep_values
    rows = []
    all_summaries = []
    for value in values:
        if sweep_type == "sensor":
            scfg = replace(cfg, n_sensors=int(value), train_size=cfg.sweep_train_size, val_size=cfg.sweep_val_size, epochs=cfg.sweep_epochs, baseline_epochs=cfg.sweep_epochs, run_methods=("cabissm", "deeponet", "fnoenc"))
            tag = f"sensor_{int(value)}"
        else:
            scfg = replace(cfg, sensor_noise_std=float(value), train_size=cfg.sweep_train_size, val_size=cfg.sweep_val_size, epochs=cfg.sweep_epochs, baseline_epochs=cfg.sweep_epochs, run_methods=("cabissm", "deeponet", "fnoenc"))
            tag = f"noise_{float(value):.3f}".replace(".", "p")
        print("\n" + "=" * 100)
        print(f"Sweep run: {tag}")
        print("=" * 100)
        run_dir = out_dir / tag
        summary = run_main_experiment(scfg, problem, device, run_dir, tag=tag)
        all_summaries.append(summary)
        for method, metrics in summary.get("full_val_metrics", {}).items():
            row = {"sweep_type": sweep_type, "sweep_value": value, "method": method}
            row.update(metrics)
            rows.append(row)
    save_table_csv(rows, out_dir / f"{sweep_type}_sweep_metrics.csv")
    # plot MAE vs sweep value
    methods = sorted(set(r["method"] for r in rows))
    plt.figure(figsize=(8, 5))
    for method in methods:
        xs = [r["sweep_value"] for r in rows if r["method"] == method]
        ys = [r["mae"] for r in rows if r["method"] == method]
        plt.plot(xs, ys, marker="o", label=method)
    plt.xlabel("number of sensors" if sweep_type == "sensor" else "sensor noise std")
    plt.ylabel("MAE")
    plt.title(f"{problem.name}: {sweep_type} sweep")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"{sweep_type}_sweep_mae.png", dpi=200)
    plt.close()
    summary = {
        "problem": problem.name,
        "run_mode": f"{sweep_type}_sweep",
        "coefficient_family": cfg.coefficient_family,
        "config": asdict(cfg),
        "rows": rows,
        "subruns": all_summaries,
    }
    save_json(summary, out_dir / f"{sweep_type}_sweep_summary.json")
    return summary


def run_all_modes_experiment(cfg: CFG, problem: BaseProblem, device: torch.device, out_dir: Path) -> Dict:
    """
    Runs all journal-level modes sequentially for one selected equation:
      1) main: CABiSSM + DeepONet + FNOEnc + VC-PINN-style
      2) ablation: architecture/loss ablations for CABiSSM
      3) sensor_sweep: sparse-sensor robustness
      4) noise_sweep: noisy-observation robustness
    Each mode gets its own subfolder. A combined final_summary.json, PDF, and ZIP are saved at the parent folder.
    """
    ensure_dir(out_dir)
    modes_to_run = ["main", "ablation", "sensor_sweep", "noise_sweep"]
    mode_summaries: Dict[str, Dict] = {}
    for mode in modes_to_run:
        print("\n" + "#" * 120)
        print(f"RUNNING MODE: {mode.upper()} for problem {problem.name}")
        print("#" * 120)
        mcfg = replace(cfg, run_mode=mode)
        mode_dir = out_dir / mode
        if mode == "main":
            mode_summaries[mode] = run_main_experiment(mcfg, problem, device, mode_dir, tag="main")
        elif mode == "ablation":
            mode_summaries[mode] = run_ablation_experiment(mcfg, problem, device, mode_dir)
        elif mode == "sensor_sweep":
            mode_summaries[mode] = run_sweep_experiment(mcfg, problem, device, mode_dir, sweep_type="sensor")
        elif mode == "noise_sweep":
            mode_summaries[mode] = run_sweep_experiment(mcfg, problem, device, mode_dir, sweep_type="noise")
        else:
            raise ValueError(mode)
    combined = {
        "problem": problem.name,
        "run_mode": "all_modes",
        "coefficient_family": cfg.coefficient_family,
        "target_axis": problem.target_axis,
        "coefficient_range": [problem.coeff_min, problem.coeff_max],
        "config": asdict(cfg),
        "modes": mode_summaries,
    }
    save_json(combined, out_dir / "all_modes_summary.json")
    return combined


# ============================================================
# Main
# ============================================================
def main():
    cfg = CFG()
    set_seed(cfg.seed)
    torch.set_float32_matmul_precision("high")
    device = get_device()
    print("Device:", device)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    problem = get_problem(cfg.problem_name)
    root_dir = Path(cfg.out_root)
    family_tag = cfg.coefficient_family if problem.name in ["vkdv_paper", "vsg"] else "default"
    out_dir = root_dir / f"{problem.name}_{family_tag}_{cfg.run_mode}"
    ensure_dir(out_dir)
    print("=" * 100)
    print(f"Running problem       : {problem.name}")
    print(f"Run mode              : {cfg.run_mode}")
    print(f"Coefficient family    : {family_tag}")
    print(f"Target axis           : {problem.target_axis}")
    print(f"Coefficient range     : {problem.coeff_min} to {problem.coeff_max}")
    print(f"Output folder         : {out_dir.resolve()}")
    print("=" * 100)
    start_all = time.time()
    if cfg.run_mode == "main":
        summary = run_main_experiment(cfg, problem, device, out_dir, tag="main")
    elif cfg.run_mode == "ablation":
        summary = run_ablation_experiment(cfg, problem, device, out_dir)
    elif cfg.run_mode == "sensor_sweep":
        summary = run_sweep_experiment(cfg, problem, device, out_dir, sweep_type="sensor")
    elif cfg.run_mode == "noise_sweep":
        summary = run_sweep_experiment(cfg, problem, device, out_dir, sweep_type="noise")
    elif cfg.run_mode == "all_modes":
        summary = run_all_modes_experiment(cfg, problem, device, out_dir)
    else:
        raise ValueError(f"Unknown run_mode: {cfg.run_mode}")
    summary["total_wall_time_sec"] = time.time() - start_all
    save_json(summary, out_dir / "final_summary.json")
    pdf_path = None
    zip_path = None
    if cfg.make_pdf_summary:
        pdf_path = save_pdf_report(summary, out_dir)
        print(f"PDF report saved: {pdf_path}")
    if cfg.make_zip:
        zip_path = make_results_zip(out_dir)
        print(f"ZIP saved: {zip_path}")
    print("\nExperiment completed.")
    print("Artifacts saved in:", out_dir.resolve())
    try:
        from IPython.display import FileLink, display
        if pdf_path is not None:
            display(FileLink(str(pdf_path)))
        if zip_path is not None:
            display(FileLink(str(zip_path)))
    except Exception:
        pass


if __name__ == "__main__":
    main()


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Running problem       : adr
Run mode              : main
Coefficient family    : default
Target axis           : x
Coefficient range     : 0.0015 to 0.006
Output folder         : /kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/adr_default_main
[adr] Building train_main dataset: 16384 samples, 128 simulation batches
  [adr/train_main] batch 12/128 | elapsed 0.5s
  [adr/train_main] batch 24/128 | elapsed 0.9s
  [adr/train_main] batch 36/128 | elapsed 1.4s
  [adr/train_main] batch 48/128 | elapsed 1.9s
  [adr/train_main] batch 60/128 | elapsed 2.3s
  [adr/train_main] batch 72/128 | elapsed 2.8s
  [adr/train_main] batch 84/128 | elapsed 3.3s
  [adr/train_main] batch 96/128 | elapsed 3.8s
  [adr/train_main] batch 108/128 | elapsed 4.2s
  [adr/train_main] batch 120/128 | elapsed 4.7s
  [adr/train_main] batch 128/128 | elapsed 5.0s
[adr] Building val_main dataset: 2048 samples, 16 simulation batches
  [adr/val_main] batch

/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/adr_default_main/summary_report.pdf

/kaggle/working/PUBLISHABLE_inverse_pde_cabissm_pipeline/adr_default_main.zip